# NB12：PLS 定量 II — VIP、PLS-DA 與離群值診斷

**食品分析實驗 週次 14｜銜接 W13（NB11）與 W14 課堂 deck**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 用標準公式計算 **VIP**（Variable Importance in Projection），並用三個「會驗證自己」的檢查確認算法沒有寫錯：VIP 不應該全部相同、`mean(VIP²)` 應該非常接近 1、高 VIP 波段應該對得上已知的化學吸收。
2. 認識一個**真實發生過的計算 bug**（迴圈裡多寫了一個 `.sum()`），並實際跑一次錯誤版本，看它怎麼被上面三個檢查抓到。
3. 分辨 **VIP（重不重要）** 與**迴歸係數的正負號（方向）**是兩件不同的事。
4. 用 **VIP > 1** 篩選波長重建模型，並誠實比較篩選前後的 RMSECV／RMSEP（可能變好，也可能變差）。
5. 用 one-hot 編碼把分類問題交給 PLS 做（**PLS-DA**），用 argmax 判定類別，看懂混淆矩陣、準確率、敏感度、特異度。
6. 用 PCA 算出 **Hotelling T²**（模型平面內的槓桿）與 **Q 殘差／SPE**（模型解釋不到的新變異），畫出帶 95% 界限的影響圖（influence plot）。
7. 說出偵測到離群值後**該做什麼、不該做什麼**，並理解「模型遷移」（換儀器）為什麼常見高 Q。

## 如何使用本筆記本

- 直接「全部執行」即可用三組內建的**模擬光譜資料**（VIP 迴歸示範、六種食用油、PCA 校正/新樣品）跑完整份筆記本。
- 三個小節個別使用 `load_spectra()` 載入各自的示範資料；若要換成班上實際資料，把對應小節的 `load_spectra(...)` 換成你的檔案路徑或 URL 即可。
- 光譜資料格式規範與共用函式定義在下方 Setup 小節，與 NB09–NB13 完全一致。

> ⚠️ 本筆記本三組內建資料皆為**模擬教學資料**，不是真實量測值。

## 1. 環境設定 (Setup)

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import confusion_matrix

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 14
np.random.seed(RNG_SEED)

### 光譜資料格式 (Shared Spectral Data Format) — NB09–NB13 共用

從本週開始，資料改用**寬格式 (wide format)**，跟 NB01 的長格式不同：**一列＝一條完整光譜**。

| 規則 | 說明 |
|---|---|
| 中繼資料欄位在前 | 欄名**不是數字**的欄位，例如 `sample_id`（樣品代號）、`group`（組別 1–5，沿用 NB01 的定義；TA 提供的參考/示範資料用 `0`）、`instrument`（儀器代號）、`rep`（重複次數）、`role`（`library`/`unknown`/`mixture` 等角色）、以及依主題而定的標籤欄位，例如 `class`（化合物名稱）、`oil`（油品代號）、`acidity_pct`、`aspartame_pct`、`sucralose_pct` 等。 |
| 光譜欄位在後 | 欄名**能轉成浮點數**的欄位＝光譜資料本身，欄名即該點的波長 (nm，NIR) 或波數 (cm⁻¹，Raman)。程式一律依欄名數值排序，不假設檔案裡欄位順序已經由小到大。 |
| 光譜欄位不可缺值 | 每一列在所有光譜欄位都必須有數值，不可有 NaN。 |
| **一個檔案只用一種共同 x 軸** | Raman 位移與 NIR 波長不可混在同一個檔案；不同解析度/範圍的儀器資料也請分開存檔，或先內插到同一組 x 軸再合併。 |
| 標籤欄位的值用英文代碼 | 例如油品用 `EVOO`/`COC`，化合物用 `aspartame`/`sucralose`。畫圖的圖例文字直接取自欄位值，中文會因 Colab 字型問題變成方框。 |

**也相容 SpectraView 的「rows」匯出格式**（`specview/formats/__init__.py` 的 `save_combined_csv(..., layout="rows")`）：
第一列表頭的第一格是 `x:<軸標籤>`（例如 `x:Raman shift (cm-1)`），之後每一格才是 x 值本身；
從第二列開始，每一列的第一格是**光譜名稱**、其餘是強度值。這種格式沒有額外中繼資料欄位，
只有一個 `sample_id`——如果要加上組別/儀器等資訊，請自行用 `sample_id` 做 `merge`。

共用讀取/拆解/畫圖函式（簽名固定，NB09–NB13 都一樣）：

```python
load_spectra(source=None)          # -> 寬格式 DataFrame；None 時退回內建模擬光譜
split_spectra(df)                  # -> (meta, X, axis)
plot_spectra(axis, X, labels)      # -> matplotlib Axes，依 labels 分色、圖例自動去重
```

In [ ]:
def _is_number(text):
    try:
        float(text)
        return True
    except (TypeError, ValueError):
        return False


def load_spectra(source=None):
    """讀取寬格式光譜 CSV（見上方「光譜資料格式」說明）。

    Parameters
    ----------
    source : str, path, file-like, or None
        - None：退回本筆記本內嵌的模擬光譜字串 SAMPLE_CSV（模擬光譜，非真實量測）。
        - 本機/雲端硬碟路徑，或 http(s) URL（例如已發布的 Google 試算表 CSV 連結）：
          直接用 pandas.read_csv 讀取。

    Returns
    -------
    pandas.DataFrame
        寬格式：中繼資料欄位（欄名非數字）+ 光譜欄位（欄名為數字，代表 nm 或 cm-1）。
    """
    if source is None:
        return pd.read_csv(io.StringIO(SAMPLE_CSV))
    return pd.read_csv(source)


def split_spectra(df):
    """把 load_spectra() 讀進來的寬格式 DataFrame 拆成 (meta, X, axis)。

    自動判斷兩種輸入：
    - 一般格式：欄名可轉成浮點數的欄位＝光譜欄位；其餘欄位＝中繼資料 (meta)。
    - SpectraView「rows」匯出格式：第一個欄名以 "x:" 開頭，代表該欄其實是
      「樣品名稱」，其餘欄名本身就是 x 值；此時 meta 只有一欄 sample_id
      （由第一欄改名而來），其他中繼資料需自行用 sample_id 合併(merge)。

    Returns
    -------
    meta : pandas.DataFrame   中繼資料（含 sample_id 或其他非光譜欄位）
    X    : numpy.ndarray, shape (n_spectra, n_points)   光譜強度矩陣
    axis : numpy.ndarray, shape (n_points,)             x 軸，已由小到大排序
    """
    cols = list(df.columns)

    if isinstance(cols[0], str) and cols[0].startswith("x:"):
        meta = df.iloc[:, [0]].rename(columns={cols[0]: "sample_id"}).reset_index(drop=True)
        spec_cols = cols[1:]
    else:
        spec_cols = [c for c in cols if _is_number(c)]
        meta_cols = [c for c in cols if c not in spec_cols]
        meta = df[meta_cols].reset_index(drop=True)

    axis = np.array([float(c) for c in spec_cols])
    X = df[spec_cols].to_numpy(dtype=float)

    order = np.argsort(axis)
    axis = axis[order]
    X = X[:, order]
    return meta, X, axis


def plot_spectra(axis, X, labels, ax=None, title=None, xlabel="x", ylabel="Intensity (a.u.)"):
    """畫出多條光譜，依 labels 分色，圖例只顯示每個 label 一次。"""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))
    labels = list(labels)
    unique_labels = sorted(set(labels), key=labels.index)
    cmap = plt.get_cmap("tab10")
    color_of = {lab: cmap(i % 10) for i, lab in enumerate(unique_labels)}
    seen = set()
    for i in range(X.shape[0]):
        lab = labels[i]
        show_label = lab if lab not in seen else None
        seen.add(lab)
        ax.plot(axis, X[i], color=color_of[lab], linewidth=1.3, alpha=0.85, label=show_label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    return ax

In [ ]:
# ---- Embedded demo spectra (fallback), Part 1: VIP regression demo ----
# Exact content of data/nb12_demo_spectra_vip.csv.
SAMPLE_CSV_VIP = """sample_id,group,instrument,rep,acidity_pct,900,910,920,930,940,950,960,970,980,990,1000,1010,1020,1030,1040,1050,1060,1070,1080,1090,1100,1110,1120,1130,1140,1150,1160,1170,1180,1190,1200,1210,1220,1230,1240,1250,1260,1270,1280,1290,1300,1310,1320,1330,1340,1350,1360,1370,1380,1390,1400,1410,1420,1430,1440,1450,1460,1470,1480,1490,1500,1510,1520,1530,1540,1550,1560,1570,1580,1590,1600,1610,1620,1630,1640,1650,1660,1670,1680,1690,1700
VIPDEMO-000,0,sim_PLS_VIP_demo,1,7.1549,0.33378533,0.33841272,0.33381766,0.33313412,0.32779815,0.3308139,0.32727829,0.33123796,0.32564146,0.32417546,0.32135494,0.31862122,0.32109063,0.32024639,0.3196515,0.32358428,0.32732617,0.33189091,0.32967475,0.33463388,0.32925531,0.3311036,0.32837866,0.33678918,0.33324257,0.33575575,0.34420526,0.34289824,0.34463379,0.3513166,0.34833016,0.35081776,0.34488367,0.34547413,0.34331201,0.33709761,0.33933776,0.33034678,0.32905298,0.32859067,0.33044234,0.32592721,0.32502684,0.32673434,0.32936453,0.3239183,0.32067548,0.32307947,0.32373759,0.317891,0.31673794,0.30693318,0.30850445,0.30499645,0.30384897,0.30180291,0.30389608,0.30090363,0.30921909,0.30598239,0.31256658,0.32041657,0.32156321,0.31455879,0.32228488,0.31898632,0.31704959,0.31839616,0.31588306,0.31692502,0.30566776,0.30660061,0.30654775,0.30196375,0.31271386,0.31681576,0.312041,0.31698735,0.31633402,0.31866732,0.31611778
VIPDEMO-001,0,sim_PLS_VIP_demo,1,4.8047,0.26589789,0.27002479,0.27241032,0.2689312,0.26780061,0.26699523,0.27241404,0.27440315,0.27212193,0.26618494,0.26069605,0.2607973,0.25717504,0.25856505,0.26959064,0.26946024,0.27514688,0.27670863,0.28398728,0.28597756,0.28029028,0.28189936,0.28083499,0.2861679,0.28480571,0.27860897,0.28452153,0.27585195,0.27062992,0.26738773,0.27323074,0.2758604,0.2753354,0.28108829,0.28684387,0.28634544,0.29205257,0.29708943,0.29568771,0.29411871,0.29847569,0.2998645,0.30558846,0.29872298,0.29623318,0.30273775,0.30400108,0.30335445,0.30846937,0.31067887,0.31715047,0.31550697,0.32068682,0.32714534,0.32901509,0.33600354,0.33368455,0.32996385,0.33160354,0.32515196,0.32355935,0.32059073,0.31619111,0.31544188,0.31570291,0.31260841,0.31525407,0.31934383,0.31821759,0.31495873,0.32539385,0.31577687,0.32138744,0.32245165,0.32495031,0.31943016,0.32297437,0.32318637,0.32801507,0.32651731,0.32736563
VIPDEMO-002,0,sim_PLS_VIP_demo,1,6.5137,0.31561106,0.31359379,0.31087937,0.31050466,0.31415897,0.3154379,0.31424608,0.31613183,0.31772803,0.318833,0.32161544,0.32441068,0.32824282,0.32624319,0.32194548,0.32746863,0.32513962,0.3200763,0.32355021,0.31898116,0.31881583,0.31983063,0.32822676,0.32567986,0.32863836,0.33079973,0.33259745,0.33845829,0.34184189,0.34771447,0.34024161,0.34715076,0.34326025,0.33400147,0.3396197,0.3390531,0.33530102,0.33104146,0.32959532,0.32246209,0.32903943,0.32856033,0.32535033,0.32894343,0.32659277,0.33551258,0.32513176,0.32585908,0.32147087,0.32563054,0.32551342,0.32308922,0.31884761,0.32134024,0.31808426,0.32066617,0.31772719,0.31482536,0.3223996,0.32278047,0.33107196,0.32909452,0.33193627,0.33381379,0.33000142,0.33359161,0.33701246,0.33171269,0.33133007,0.33030023,0.32577371,0.32549295,0.32737936,0.32593538,0.32934244,0.33146666,0.33148468,0.333055,0.34460248,0.34399432,0.34030848
VIPDEMO-003,0,sim_PLS_VIP_demo,1,7.3006,0.30703343,0.30816349,0.30325058,0.30338796,0.30827964,0.31089218,0.3016042,0.30486913,0.30239627,0.29908177,0.29718571,0.29584626,0.29666729,0.29325541,0.3001743,0.30547387,0.30835207,0.31106709,0.31179188,0.3131062,0.31994651,0.31538306,0.31866007,0.32486961,0.32501004,0.32551471,0.33268328,0.33797983,0.34312578,0.34534041,0.35107734,0.3498155,0.34707023,0.3411142,0.34145864,0.33742029,0.33753757,0.32644619,0.32603096,0.3293325,0.32141058,0.3253055,0.32531301,0.32509678,0.32240475,0.32553154,0.32088727,0.32831659,0.32200087,0.32166636,0.31786262,0.31431277,0.30961214,0.30644428,0.30137816,0.30189495,0.30015878,0.30406849,0.30971675,0.31474982,0.31882987,0.32534762,0.33234677,0.33399898,0.33305935,0.33624062,0.33703042,0.34430048,0.33178125,0.33736385,0.33279337,0.34063912,0.33663942,0.33923541,0.33934962,0.34177257,0.33194004,0.33805922,0.34251147,0.33896837,0.34516372
VIPDEMO-004,0,sim_PLS_VIP_demo,1,6.2066,0.30771547,0.31020878,0.31032209,0.30801283,0.31102621,0.30904564,0.30917725,0.31781804,0.3124593,0.32036648,0.31545616,0.3176819,0.31454998,0.31577434,0.32049457,0.31358767,0.30535493,0.31368926,0.30545885,0.30606207,0.30685171,0.30139908,0.31200135,0.31197143,0.30823255,0.31167011,0.31133605,0.31199257,0.31440235,0.31163521,0.31488862,0.31385375,0.31998632,0.30816339,0.31432069,0.31068741,0.30718891,0.30344261,0.30945508,0.30753643,0.30242306,0.30150342,0.30231189,0.30692335,0.2985208,0.30126621,0.3042081,0.29647032,0.30018934,0.30358945,0.298794,0.29961419,0.29518091,0.30078416,0.29596086,0.29812459,0.30007848,0.29715993,0.30116998,0.30044714,0.2995027,0.30075,0.29710976,0.29098857,0.29992045,0.29973124,0.29632546,0.2958999,0.30288618,0.29930191,0.30083987,0.30155443,0.31006841,0.30032005,0.29340278,0.29907723,0.2945752,0.29096458,0.29544255,0.29270345,0.29640417
VIPDEMO-005,0,sim_PLS_VIP_demo,1,5.7418,0.28365425,0.28859511,0.28944431,0.29309952,0.28971816,0.2945087,0.29237438,0.29518239,0.29754695,0.30199365,0.30522178,0.30953213,0.30758204,0.30625422,0.30707803,0.30492622,0.30041295,0.30050325,0.29879875,0.30206735,0.292501,0.30284418,0.29026413,0.29230169,0.29870799,0.29545048,0.29775244,0.29769677,0.29550561,0.29571827,0.29904879,0.30168126,0.30285263,0.30202484,0.30101124,0.30599158,0.29641686,0.30343992,0.30757726,0.3090404,0.30826602,0.30913445,0.30864699,0.30641305,0.29981245,0.3054193,0.30659074,0.3126365,0.307251,0.30815544,0.31193152,0.30793325,0.3117911,0.31417035,0.31227585,0.3135884,0.30880957,0.31054049,0.31350823,0.31764681,0.31255481,0.31160781,0.31216719,0.31681528,0.31103875,0.31637182,0.31559421,0.31794728,0.3168618,0.32089378,0.32621353,0.32257017,0.32269242,0.31663299,0.32529888,0.32128498,0.32366199,0.32235816,0.31944938,0.31992069,0.31805294
VIPDEMO-006,0,sim_PLS_VIP_demo,1,6.8116,0.32258612,0.31046202,0.31655703,0.31453179,0.31705135,0.3099877,0.31513664,0.32194503,0.31727362,0.32440949,0.32201342,0.32536745,0.32209477,0.32475418,0.32143759,0.32659252,0.31877897,0.32426065,0.32137291,0.32268316,0.32811109,0.32422987,0.32483354,0.32920261,0.3343885,0.33444479,0.34533162,0.34493594,0.34697935,0.34868086,0.35133717,0.35304939,0.35266723,0.3468406,0.34142777,0.33960644,0.33040688,0.33761172,0.33506,0.33093623,0.33307017,0.32786547,0.33628926,0.33075592,0.33016092,0.33802852,0.32920389,0.33334365,0.32930509,0.32953115,0.32922374,0.32692161,0.33145589,0.3289718,0.32312518,0.33185821,0.32541002,0.32660107,0.33044311,0.32844374,0.3376847,0.33276664,0.33315377,0.34116291,0.33614721,0.33900228,0.33718829,0.34083786,0.34422216,0.33879307,0.34774511,0.3471553,0.35453675,0.35626437,0.350865,0.34970183,0.34972087,0.34867303,0.3405685,0.34062996,0.34309443
VIPDEMO-007,0,sim_PLS_VIP_demo,1,6.5816,0.28029309,0.27899042,0.27897093,0.27786347,0.27638794,0.28203899,0.28471305,0.28312819,0.28605633,0.28467107,0.29101115,0.29486698,0.29456154,0.29429332,0.28905473,0.28544672,0.2856607,0.28852621,0.28143734,0.27733559,0.27964772,0.28130315,0.28626297,0.28705175,0.28306387,0.28439536,0.2887841,0.29166914,0.2906094,0.29770688,0.29407091,0.29576375,0.29330598,0.29369868,0.28984444,0.28892028,0.29005045,0.2856645,0.28820064,0.27947989,0.28222522,0.27949972,0.27828786,0.28234544,0.28809092,0.28314082,0.28525339,0.27632068,0.2820864,0.27708067,0.27985013,0.28060799,0.27440264,0.27745146,0.27082753,0.27310444,0.27141487,0.27476596,0.27577352,0.27163447,0.27881828,0.27986114,0.28739157,0.27576296,0.28386681,0.28638213,0.2898389,0.28832713,0.28461711,0.28933959,0.28497458,0.28504161,0.28472612,0.28464992,0.29019737,0.28070365,0.2808397,0.28812999,0.28665648,0.28781068,0.28455918
VIPDEMO-008,0,sim_PLS_VIP_demo,1,5.3358,0.31967458,0.31596344,0.31325857,0.31004375,0.3133365,0.30834483,0.31010162,0.30885965,0.30909701,0.31167485,0.30096957,0.3143475,0.3108931,0.30798495,0.30726245,0.31108469,0.30845711,0.31374922,0.30787861,0.30516747,0.30317181,0.30641448,0.30885082,0.30098598,0.30689671,0.30389572,0.2985813,0.30143522,0.29077249,0.28821508,0.29486839,0.2952766,0.29571187,0.29546553,0.29243795,0.29964442,0.29799913,0.30246361,0.30100278,0.29715552,0.30245401,0.30292341,0.30268716,0.29795253,0.29667321,0.301873,0.29910911,0.29881672,0.30217283,0.30255131,0.30135011,0.29978969,0.30559438,0.29860225,0.30199505,0.30286555,0.29721288,0.29819301,0.3016923,0.29748233,0.29328697,0.30032091,0.29499575,0.29400968,0.29851505,0.28797435,0.2899855,0.28783004,0.29325156,0.28539936,0.29344636,0.28665982,0.28549878,0.28748051,0.28439421,0.28309648,0.28477194,0.28724422,0.28454404,0.28792979,0.28549511
VIPDEMO-009,0,sim_PLS_VIP_demo,1,5.8623,0.26795786,0.26949902,0.26903944,0.26514409,0.26849669,0.26734986,0.27173871,0.2717743,0.26987824,0.2680986,0.274708,0.27031039,0.27273983,0.27277187,0.2745616,0.27695263,0.27365664,0.27452545,0.27875682,0.26828419,0.27871557,0.27858922,0.28349641,0.2838893,0.28413982,0.28333054,0.28857213,0.29040896,0.29360988,0.2956686,0.29388731,0.29790476,0.29446658,0.29657459,0.29523882,0.29720374,0.29263527,0.29695205,0.29087623,0.28938544,0.29421064,0.28901322,0.29436085,0.295022,0.29507535,0.29643566,0.29658108,0.3006465,0.29742267,0.29307056,0.29911893,0.30414889,0.30611416,0.30661285,0.30587771,0.30387546,0.31177796,0.30550112,0.30386307,0.29971506,0.31255392,0.30685508,0.30153099,0.30994988,0.30969518,0.30296872,0.31037606,0.30782926,0.30915816,0.31032742,0.30409299,0.31136609,0.30476687,0.30439493,0.31153352,0.30820611,0.31138395,0.31683449,0.31328497,0.31308568,0.31889808
VIPDEMO-010,0,sim_PLS_VIP_demo,1,6.7316,0.27164944,0.27763079,0.27589871,0.27514106,0.27682184,0.27617623,0.27868047,0.27332805,0.27514497,0.27107511,0.26833882,0.26740127,0.26541158,0.27072858,0.27213436,0.27560095,0.27245311,0.27825352,0.28588127,0.28644626,0.28365871,0.28522093,0.28712478,0.29139655,0.29467619,0.29362651,0.29615937,0.29800604,0.30139739,0.3074164,0.30108731,0.30494965,0.29876975,0.29749328,0.29396174,0.29536643,0.29999119,0.29927903,0.29603109,0.29316809,0.29632363,0.2901438,0.28930803,0.29123152,0.29352758,0.2890176,0.29225985,0.29961297,0.29366485,0.29234553,0.2904311,0.29293995,0.29108047,0.28790194,0.28621324,0.28027502,0.2831699,0.28809746,0.28996579,0.29325052,0.29515439,0.30388097,0.30353987,0.29814029,0.3045612,0.30429914,0.30384714,0.30411942,0.30699448,0.31261396,0.30667327,0.30875634,0.30953966,0.30511852,0.31004229,0.30728806,0.31022103,0.31048168,0.3146528,0.30885164,0.30898374
VIPDEMO-011,0,sim_PLS_VIP_demo,1,3.3178,0.32183093,0.32211752,0.32304975,0.32508257,0.3216888,0.32209616,0.32461985,0.32460881,0.32584113,0.32254387,0.33245183,0.32675629,0.32683834,0.32487154,0.32821012,0.32918219,0.32537454,0.32446831,0.32337238,0.32987619,0.32392404,0.3192521,0.31832203,0.31714179,0.30942004,0.3035896,0.29622287,0.28819184,0.28438156,0.28342591,0.27862972,0.27782576,0.28036954,0.28859947,0.29420027,0.30032819,0.31097636,0.31511553,0.32120291,0.32208723,0.32124807,0.32612666,0.32516324,0.32842908,0.32180816,0.32409674,0.32611106,0.33186582,0.33104934,0.32997128,0.3356107,0.34589373,0.34635504,0.35482363,0.36060729,0.36148932,0.36184224,0.35913118,0.35164105,0.34063061,0.34207478,0.33634552,0.32733807,0.32634158,0.32195258,0.32887924,0.32266105,0.32433717,0.32340906,0.32439697,0.31978534,0.31750932,0.31991772,0.32135547,0.31796552,0.32054649,0.31766096,0.32717204,0.32196084,0.32715999,0.32708891
VIPDEMO-012,0,sim_PLS_VIP_demo,1,6.2344,0.34044238,0.33385078,0.33566784,0.33047904,0.33712414,0.33123275,0.33047891,0.32955891,0.32524182,0.32312758,0.32881485,0.32447973,0.32635638,0.33250642,0.32568615,0.3323426,0.32731936,0.32310885,0.32781057,0.32916353,0.32808712,0.32380836,0.32286244,0.32893025,0.33261242,0.33324135,0.32827988,0.32981039,0.33221925,0.33512959,0.33314584,0.33239102,0.32982811,0.32040792,0.32252611,0.32454438,0.31776723,0.32668876,0.31713528,0.32042439,0.32040744,0.32037388,0.31313621,0.31546742,0.31997506,0.3152081,0.31677257,0.30644958,0.31365043,0.31156323,0.30896207,0.30207399,0.30289963,0.30222623,0.30034181,0.29739322,0.30380418,0.30412404,0.30152333,0.30819783,0.30169494,0.30699016,0.31142232,0.30850343,0.30621124,0.30761932,0.30498966,0.30249724,0.30018594,0.30062941,0.29827286,0.2981186,0.29391034,0.2971442,0.29979583,0.29627226,0.30376143,0.29773349,0.29817025,0.29861213,0.29667017
VIPDEMO-013,0,sim_PLS_VIP_demo,1,6.6803,0.27395476,0.27684637,0.27566482,0.27621847,0.27754478,0.27941871,0.27205268,0.27635266,0.27399192,0.27405691,0.27105976,0.27412545,0.27386116,0.27314201,0.27003502,0.27462975,0.27015723,0.26878867,0.27577237,0.27328822,0.26933382,0.27237556,0.27521802,0.27911447,0.27721965,0.27746805,0.2789574,0.28028647,0.28570783,0.28502692,0.28943961,0.28498078,0.28863483,0.28281839,0.28111199,0.27825615,0.27596734,0.2792436,0.27113947,0.27071537,0.27082681,0.27296531,0.26789407,0.27469678,0.26752971,0.26847661,0.27006617,0.26981076,0.26602037,0.27031582,0.27183837,0.26500659,0.2656745,0.2641885,0.26847252,0.26121277,0.26210757,0.26086769,0.2638918,0.26344865,0.26441293,0.26573679,0.26969216,0.26652521,0.27000931,0.26701227,0.2704731,0.26303431,0.2624152,0.26645508,0.26802839,0.26160897,0.25655255,0.2659019,0.26458996,0.26182076,0.26871439,0.26916638,0.26387805,0.26469802,0.26777594
VIPDEMO-014,0,sim_PLS_VIP_demo,1,4.9938,0.30937303,0.31144891,0.30966611,0.31455255,0.31581422,0.31703309,0.31054171,0.3076312,0.30907525,0.30896421,0.30607959,0.30785203,0.30773828,0.30800563,0.30596486,0.31276173,0.30970974,0.30674002,0.31437252,0.31893014,0.31584363,0.31132804,0.31483342,0.31884427,0.31546041,0.31393141,0.30995683,0.31246537,0.31005349,0.31591856,0.31257336,0.31452569,0.30842904,0.31420395,0.31790543,0.32178844,0.32186631,0.31688228,0.32834937,0.32023479,0.32039403,0.32077724,0.31884832,0.32138762,0.32075098,0.32174097,0.32153558,0.32338649,0.32698932,0.32420955,0.32438254,0.33123442,0.33146051,0.33535028,0.33211044,0.33508081,0.33290671,0.3306402,0.33173709,0.33135893,0.331274,0.32597251,0.3308434,0.33137235,0.32906186,0.32676723,0.32534155,0.3301502,0.32432474,0.32952253,0.31972882,0.31888459,0.32561269,0.32133871,0.32892361,0.32618467,0.32735999,0.33093187,0.33408764,0.33405183,0.33162169
VIPDEMO-015,0,sim_PLS_VIP_demo,1,5.5359,0.31852529,0.31573145,0.31787738,0.31145769,0.31543882,0.31942284,0.31896214,0.31793135,0.32529531,0.33014634,0.33985844,0.34206472,0.34128368,0.34508495,0.33276007,0.33145534,0.32332183,0.31652539,0.31154589,0.31189723,0.31302404,0.30274294,0.30654688,0.30767756,0.3013378,0.2999176,0.30434209,0.29785383,0.29588181,0.29702132,0.29591937,0.29652826,0.29681307,0.29994358,0.29769938,0.3000131,0.2958122,0.29918412,0.29724135,0.30289975,0.29949578,0.30176645,0.2973321,0.29488053,0.29928159,0.29614648,0.29187151,0.29470191,0.29278527,0.298428,0.29354754,0.29511271,0.29392911,0.2952678,0.29251083,0.29213653,0.29142734,0.28606862,0.28962525,0.28584508,0.28560669,0.28885137,0.28322072,0.28721596,0.29041386,0.287361,0.29003562,0.28575779,0.29015691,0.28467186,0.29084591,0.29335589,0.30054569,0.30006206,0.28652876,0.28801997,0.28417084,0.28842727,0.28445144,0.27964917,0.28206364
VIPDEMO-016,0,sim_PLS_VIP_demo,1,4.1436,0.27994393,0.28809136,0.28529469,0.29174627,0.29216297,0.28290276,0.29148833,0.28419611,0.28595263,0.28057959,0.27619873,0.27439192,0.27319231,0.28009718,0.27914509,0.2797607,0.28838002,0.28821057,0.29333498,0.29085205,0.29454137,0.29048734,0.28973135,0.29099972,0.28978948,0.28413465,0.27937238,0.28314779,0.27178553,0.26244026,0.2711144,0.26911036,0.26944822,0.2810939,0.28542458,0.28678045,0.29087464,0.29598605,0.30189727,0.29829727,0.30643466,0.30433659,0.30562634,0.30961041,0.30493303,0.30831438,0.30754108,0.30007811,0.31034771,0.31382292,0.31706201,0.32574335,0.32562113,0.33309317,0.33595235,0.33771843,0.33075384,0.331916,0.32812637,0.33008002,0.3205833,0.32038869,0.3172002,0.31115977,0.3113394,0.31522452,0.31662391,0.31920195,0.31387938,0.31093059,0.31451285,0.31084626,0.31915354,0.31471579,0.31621717,0.32257173,0.31914428,0.31485923,0.32055787,0.31784734,0.31660237
VIPDEMO-017,0,sim_PLS_VIP_demo,1,6.251,0.26494497,0.26692231,0.26506491,0.26718588,0.26670238,0.27036568,0.26753198,0.26714497,0.27012042,0.26676716,0.2744419,0.27864623,0.28264364,0.27775811,0.27928324,0.26873323,0.27663182,0.27279964,0.27429077,0.26895679,0.27163222,0.27118281,0.26985019,0.27143567,0.27694179,0.26915068,0.27371078,0.27715912,0.28084271,0.27426155,0.27393418,0.27190812,0.27828136,0.27622745,0.27589603,0.27494312,0.26939948,0.27721938,0.2764493,0.26965885,0.27138489,0.27068828,0.27129319,0.27906511,0.2766461,0.27082506,0.27980121,0.27512569,0.27384286,0.27020451,0.26927859,0.26957093,0.27694308,0.2640645,0.26411591,0.2685886,0.26711359,0.26554344,0.27754495,0.2707717,0.27146617,0.27410564,0.27747169,0.27854484,0.27992139,0.27818733,0.2760961,0.2813399,0.27632178,0.27464794,0.27795899,0.27896629,0.27406907,0.27439016,0.27596198,0.27510494,0.27709069,0.28115891,0.28140758,0.27278719,0.28457612
VIPDEMO-018,0,sim_PLS_VIP_demo,1,7.8564,0.27563673,0.27755698,0.27261392,0.28168137,0.28004871,0.27892351,0.28245843,0.27874572,0.28485276,0.2826598,0.2842687,0.27611222,0.27942072,0.27434732,0.2818675,0.28075172,0.28234791,0.2890697,0.28767537,0.29270403,0.28791949,0.29100889,0.29696929,0.29322748,0.30133656,0.310045,0.31068073,0.3237836,0.32495816,0.33424876,0.33173311,0.33779776,0.3257853,0.33047674,0.31772295,0.32149484,0.31304473,0.30425339,0.30948911,0.30469148,0.30523865,0.30560389,0.30849604,0.30532746,0.3088322,0.30445531,0.30608737,0.30896254,0.31040625,0.30503566,0.30154945,0.30020234,0.29680498,0.28912244,0.29849618,0.28816957,0.28742535,0.29475652,0.30093735,0.30478044,0.30802238,0.31384426,0.31879591,0.31771005,0.31879513,0.32329903,0.3246263,0.32222514,0.33436497,0.33188952,0.33571521,0.33540127,0.32894322,0.33987086,0.32892961,0.33067629,0.33411525,0.33132605,0.33548084,0.33648344,0.33407697
VIPDEMO-019,0,sim_PLS_VIP_demo,1,4.4936,0.29503985,0.29251707,0.29437793,0.29581863,0.29835536,0.29409714,0.29783569,0.30092557,0.30215457,0.30435873,0.29927629,0.30681885,0.30952699,0.30486947,0.30631309,0.30237092,0.30395829,0.3015853,0.29715798,0.30073825,0.30000302,0.29409691,0.2962442,0.29447712,0.28965826,0.2939173,0.28571949,0.28515884,0.27615949,0.28032703,0.27644358,0.28246534,0.28268817,0.28939663,0.29505771,0.28838761,0.29706844,0.29899391,0.29980888,0.30057845,0.30378477,0.30652029,0.30734311,0.30917099,0.30639453,0.31015955,0.31000677,0.31239061,0.31420302,0.3160172,0.32064675,0.3204712,0.3212899,0.32808455,0.33637604,0.33640606,0.33406527,0.3368889,0.33131923,0.32970609,0.31760494,0.31630081,0.31676383,0.31675095,0.31741934,0.3134386,0.3159077,0.31358863,0.30813639,0.30908949,0.30348057,0.30175502,0.29659984,0.29845366,0.30122935,0.3075219,0.30649625,0.30726551,0.31415186,0.3236798,0.31712468
VIPDEMO-020,0,sim_PLS_VIP_demo,1,5.3142,0.27165518,0.27831811,0.27337578,0.27912858,0.27681045,0.27104306,0.27280126,0.28036454,0.28459262,0.29128212,0.29342713,0.29902221,0.298077,0.29532068,0.28967835,0.29374399,0.28946756,0.28658805,0.27946012,0.28398304,0.27459721,0.28167808,0.27994841,0.279766,0.27070881,0.28523593,0.27553076,0.27602006,0.27689651,0.27618662,0.27247574,0.28064032,0.27853734,0.28054557,0.27611656,0.28280126,0.28266583,0.28761306,0.2818399,0.27833673,0.28505631,0.28635303,0.28904093,0.28815879,0.28363317,0.28679787,0.28362463,0.28926145,0.29660028,0.28614014,0.28626597,0.28787768,0.29278767,0.29657892,0.29552459,0.29126907,0.29743815,0.2979758,0.29932773,0.2954797,0.28792948,0.29486924,0.29040286,0.2870128,0.29110121,0.29219687,0.29352802,0.29122359,0.29399593,0.29739527,0.30031765,0.30217041,0.29763089,0.3044664,0.29622417,0.30006295,0.29751956,0.28798238,0.29327033,0.29514075,0.29626027
VIPDEMO-021,0,sim_PLS_VIP_demo,1,7.4581,0.30181193,0.30419263,0.30026613,0.3053584,0.30310952,0.31010353,0.30360336,0.31126321,0.31382327,0.32498315,0.3286364,0.33211058,0.33503929,0.33354076,0.3304536,0.31990283,0.31750415,0.30882853,0.30472966,0.30199166,0.30230433,0.30778852,0.30119123,0.30632955,0.31024485,0.31516371,0.31680777,0.31620171,0.32914277,0.33171054,0.33344003,0.33121008,0.3233646,0.32639305,0.31646137,0.31378185,0.31425914,0.3081012,0.29919342,0.30005912,0.30277239,0.30044255,0.29672646,0.29256996,0.29359267,0.29478694,0.2957503,0.2979757,0.29596288,0.28490601,0.28550835,0.28034173,0.27680653,0.27671512,0.27210271,0.26790399,0.2765825,0.27225339,0.27454349,0.28014551,0.29079066,0.29045204,0.28819548,0.29229645,0.2891332,0.29488974,0.29278182,0.2838673,0.29030991,0.29193454,0.28264286,0.28599648,0.28404098,0.28707875,0.28195096,0.29020696,0.28769535,0.29368602,0.292417,0.28929879,0.28686087
VIPDEMO-022,0,sim_PLS_VIP_demo,1,5.757,0.28502209,0.28528683,0.28452934,0.28799178,0.29408647,0.28249669,0.29022988,0.28890282,0.28553023,0.29035353,0.29621953,0.28552285,0.29058744,0.28705106,0.29469179,0.29378491,0.29324309,0.2929762,0.29694951,0.29792597,0.29234236,0.29120296,0.29588856,0.29975808,0.2972684,0.30357867,0.29929683,0.29875319,0.30179669,0.29738442,0.29909801,0.30104536,0.30072262,0.30788333,0.30603354,0.30414389,0.30240943,0.30707418,0.30783421,0.30364374,0.30071394,0.30647291,0.30506772,0.30483275,0.30656528,0.31005617,0.30691203,0.30775059,0.30704058,0.30695475,0.30686266,0.31261136,0.31153595,0.3106586,0.31168623,0.31431219,0.31545357,0.31409578,0.31463222,0.31583789,0.31219992,0.31031641,0.32137512,0.31729781,0.32006554,0.31302134,0.31653695,0.32001919,0.32003877,0.31557054,0.31687817,0.3137641,0.31027334,0.3104376,0.31087739,0.3152129,0.3221669,0.31947475,0.32749527,0.32596358,0.32587682
VIPDEMO-023,0,sim_PLS_VIP_demo,1,5.1073,0.2681557,0.27376233,0.26960435,0.27821215,0.27247886,0.27345088,0.27653717,0.27637522,0.28295092,0.28398852,0.29137469,0.28329266,0.2889063,0.28960386,0.28672776,0.28158971,0.27965447,0.28296373,0.2785115,0.27451384,0.2758067,0.27993521,0.2802337,0.27750129,0.27843656,0.27495139,0.27352547,0.26691849,0.2669352,0.27061237,0.2688995,0.27072636,0.27380606,0.26578026,0.27779426,0.28078642,0.28154142,0.2841184,0.28136851,0.28801287,0.28868299,0.28857347,0.29076781,0.29350384,0.29013334,0.28881106,0.29102768,0.29563829,0.29525596,0.29426042,0.29651212,0.30015315,0.29817201,0.3014418,0.30734956,0.30793043,0.30531207,0.3052099,0.30299446,0.30590113,0.2995775,0.29935185,0.29733159,0.29434575,0.29103134,0.29582456,0.30635725,0.29781976,0.30254921,0.30288658,0.30822039,0.30327364,0.30167911,0.29939531,0.30953761,0.30695953,0.29708538,0.30413253,0.30072133,0.30300962,0.30119552
VIPDEMO-024,0,sim_PLS_VIP_demo,1,6.3415,0.31651426,0.31826546,0.31596206,0.31412278,0.31610594,0.31673286,0.31449991,0.31153862,0.31567453,0.30921051,0.31440447,0.3096782,0.30946268,0.31201433,0.31054273,0.31361354,0.31511902,0.31430042,0.31646872,0.31956656,0.3165189,0.31609165,0.31909709,0.32023825,0.31652598,0.32212701,0.32155016,0.33077103,0.32548774,0.32813153,0.33064972,0.32308116,0.32333214,0.32551117,0.32461386,0.31918518,0.32168381,0.32171021,0.32082385,0.32329915,0.32038055,0.31752181,0.32212422,0.3215744,0.31747154,0.31625685,0.3199694,0.31972537,0.32247547,0.31824914,0.31974187,0.31913555,0.31517689,0.31547206,0.30830907,0.30943282,0.31269007,0.31266662,0.31452478,0.31699322,0.31404746,0.32258152,0.32116918,0.32279118,0.31518581,0.31784041,0.3157126,0.31721286,0.32047463,0.32036907,0.3219382,0.31626147,0.32118988,0.31632436,0.32096569,0.3231489,0.31772357,0.32044266,0.32358585,0.32421081,0.31910434
VIPDEMO-025,0,sim_PLS_VIP_demo,1,3.1541,0.25991629,0.26344021,0.25374248,0.25568549,0.25662207,0.26142097,0.25514695,0.26755838,0.26828661,0.27002442,0.26148113,0.26967163,0.26900427,0.27261355,0.26713064,0.26717561,0.26109269,0.26707213,0.2682383,0.26290828,0.25492668,0.25782941,0.25351779,0.2544467,0.24738038,0.24394807,0.23641165,0.22907706,0.22519322,0.2144854,0.21999227,0.21708526,0.21990736,0.23199118,0.23853568,0.24766747,0.25511987,0.2558926,0.2636711,0.26017818,0.26633405,0.2662551,0.26523902,0.2638001,0.27026988,0.26447594,0.26905223,0.27741749,0.28000195,0.28061096,0.28341226,0.29592305,0.29752528,0.30507724,0.31100506,0.30753769,0.30425761,0.29918924,0.29465456,0.28879668,0.28505079,0.28358153,0.27696051,0.27345079,0.27648958,0.27110576,0.27806439,0.27677912,0.28102886,0.27972318,0.28049131,0.28281424,0.28327837,0.28365772,0.28174405,0.28257215,0.27639347,0.28164091,0.2758578,0.28346161,0.27455828
VIPDEMO-026,0,sim_PLS_VIP_demo,1,3.769,0.30521174,0.30160405,0.30800672,0.30178365,0.30337409,0.3069498,0.30205784,0.30479263,0.29964302,0.30921374,0.31134539,0.31116806,0.3088162,0.30536426,0.31414354,0.30799994,0.30984917,0.31542964,0.31288394,0.31308212,0.31730877,0.30879071,0.31069737,0.30144307,0.30028526,0.29532981,0.29046195,0.28462613,0.2743877,0.27377088,0.27758267,0.27344257,0.28508372,0.29202824,0.28995265,0.29724164,0.31332735,0.31586357,0.31478995,0.31800876,0.32072603,0.32431324,0.3206732,0.32973354,0.32366388,0.3253321,0.32926398,0.33322737,0.33322935,0.33779219,0.34692872,0.34840254,0.35458447,0.36374099,0.36710277,0.36704664,0.3648104,0.36511317,0.36022614,0.35549641,0.34660525,0.3397269,0.33796358,0.34714727,0.33779933,0.3367653,0.33700346,0.33360561,0.33377803,0.33779917,0.33742304,0.33077886,0.33321929,0.33391257,0.33769363,0.33799173,0.34044259,0.34195611,0.34340406,0.34305422,0.3481165
VIPDEMO-027,0,sim_PLS_VIP_demo,1,7.3566,0.27399333,0.2799841,0.28127052,0.28082769,0.27836143,0.2823527,0.27500509,0.28560258,0.28505547,0.28361358,0.28852923,0.28318188,0.28916496,0.2897316,0.28712111,0.28887203,0.28538076,0.2919024,0.27951571,0.28637065,0.28893621,0.29167018,0.29589112,0.29349977,0.2950806,0.30174699,0.30455602,0.31095945,0.31301749,0.31223006,0.32243553,0.3149765,0.31814456,0.31548813,0.30785662,0.30822673,0.29738723,0.29610127,0.29659529,0.2980489,0.29190584,0.29093768,0.29526921,0.29609702,0.29065422,0.29734891,0.29953938,0.29605945,0.29572433,0.29543525,0.28656693,0.28759372,0.28049793,0.28033702,0.27788092,0.2760062,0.27435644,0.27770326,0.27919351,0.28527286,0.28979915,0.29815382,0.29898418,0.30017677,0.29665097,0.29672868,0.29725565,0.29846084,0.28707896,0.28439117,0.28391618,0.27861038,0.28148355,0.28148727,0.28204678,0.28760725,0.29273119,0.29800994,0.30519417,0.30350763,0.30435887
VIPDEMO-028,0,sim_PLS_VIP_demo,1,3.7912,0.27997915,0.28074272,0.28492814,0.28577493,0.28298033,0.284885,0.28324009,0.28540539,0.29076156,0.29369224,0.2923195,0.2989239,0.29805671,0.3025223,0.29566525,0.30077439,0.29710647,0.29362643,0.29613864,0.29434401,0.2912216,0.29055654,0.2963893,0.2911627,0.29098603,0.28925186,0.27905146,0.27416741,0.2668591,0.27004755,0.26611193,0.26691512,0.27480498,0.28340994,0.28806369,0.28867436,0.30582719,0.30793941,0.30958644,0.31498613,0.3127506,0.31669976,0.3194939,0.3181031,0.32327198,0.31851121,0.32269419,0.32423436,0.32249165,0.32674463,0.33036628,0.3437946,0.34564207,0.34595651,0.35254746,0.35300795,0.36005758,0.34951894,0.35448524,0.34517138,0.34453919,0.33868258,0.33260797,0.34136365,0.34007131,0.33982743,0.34051251,0.34290211,0.33667082,0.3385741,0.34195774,0.33812267,0.34013232,0.33957782,0.34709367,0.3430729,0.34982927,0.35000903,0.35076539,0.34945489,0.3538345
VIPDEMO-029,0,sim_PLS_VIP_demo,1,3.1473,0.32221389,0.31985051,0.31987318,0.31278109,0.31992285,0.31482326,0.31814759,0.31434505,0.31637311,0.30991809,0.30845944,0.31070103,0.30800539,0.30531476,0.30867346,0.30408035,0.30439901,0.30690407,0.30531211,0.30414393,0.30343441,0.29725853,0.29740965,0.29194767,0.28612399,0.27794189,0.26534924,0.26567639,0.25025512,0.2491296,0.24893629,0.25410365,0.2486127,0.25887262,0.26468359,0.27171745,0.27378709,0.28334301,0.28200998,0.28448951,0.28607705,0.28848208,0.28463705,0.28660823,0.28579289,0.28467749,0.28150342,0.28630344,0.28907359,0.29120083,0.29833195,0.30021962,0.30942037,0.3124579,0.31589268,0.31969242,0.31361333,0.30652827,0.30434108,0.29550549,0.28441849,0.27918642,0.27823943,0.2763238,0.2707952,0.27279238,0.26834819,0.27298082,0.26301518,0.27208415,0.27230824,0.27075977,0.26609095,0.2682662,0.26278789,0.26787466,0.25785395,0.26167124,0.25732311,0.25248082,0.25778211
VIPDEMO-030,0,sim_PLS_VIP_demo,1,7.632,0.35230874,0.34706661,0.34815918,0.34686633,0.34639034,0.34752065,0.34703285,0.35497976,0.34489433,0.35778805,0.35753756,0.35725637,0.36041208,0.3593361,0.35112556,0.34639401,0.35566801,0.34827565,0.34537545,0.35109425,0.34236472,0.34216031,0.34880526,0.34886653,0.35131425,0.35448961,0.36242755,0.36363306,0.37572308,0.37095543,0.37303165,0.37261305,0.36624631,0.37005036,0.35652851,0.35515893,0.34389689,0.35142837,0.34125978,0.34002005,0.33522858,0.3360225,0.33082743,0.33212835,0.3337263,0.32731834,0.3284444,0.33358676,0.33016951,0.32440566,0.3235359,0.31623599,0.31501996,0.3046748,0.29853868,0.29960252,0.30741078,0.30052418,0.3078209,0.31322493,0.31362821,0.32093184,0.32054476,0.32483054,0.32298201,0.3271271,0.32204932,0.31766163,0.3302043,0.32774415,0.32253898,0.32231001,0.31982279,0.32763771,0.31835903,0.32379996,0.32257835,0.3181584,0.32479817,0.31566085,0.32077787
VIPDEMO-031,0,sim_PLS_VIP_demo,1,6.7639,0.34526457,0.34382838,0.33611571,0.34230225,0.33413486,0.33860158,0.32846254,0.32986579,0.3234774,0.33030493,0.32371067,0.31437798,0.31335213,0.31891214,0.31870081,0.31832839,0.32115988,0.32357823,0.32192085,0.31790763,0.32053232,0.32492815,0.32578523,0.32759781,0.32858657,0.3274696,0.32903187,0.33631333,0.32983749,0.33212334,0.33824327,0.33484267,0.32704734,0.3206613,0.31770193,0.31998516,0.31303484,0.31360492,0.3038115,0.30970793,0.30349463,0.30492871,0.30935967,0.3036076,0.30399138,0.2965987,0.29627302,0.29302311,0.29903313,0.29090571,0.29227953,0.29346021,0.28172804,0.28220666,0.27528049,0.28015163,0.27763287,0.28152836,0.28083276,0.28437192,0.27722396,0.27924919,0.28552571,0.28411748,0.28167724,0.27953316,0.27984583,0.27772102,0.2821378,0.27773825,0.28049561,0.27588415,0.28374297,0.27889112,0.27586289,0.27331999,0.26896074,0.27282744,0.27268557,0.27408589,0.2679442
VIPDEMO-032,0,sim_PLS_VIP_demo,1,7.8872,0.2694574,0.27704197,0.26777207,0.27286105,0.27404441,0.27204127,0.27172015,0.27316375,0.277035,0.27666898,0.28551701,0.28909395,0.2773062,0.28590777,0.281778,0.27997297,0.27838036,0.27495837,0.27510164,0.27097967,0.27682838,0.27865494,0.27938304,0.28462209,0.28352735,0.29306802,0.30020445,0.30066977,0.30811454,0.30965405,0.31056174,0.3167526,0.30798326,0.30101203,0.29700961,0.29204185,0.29518956,0.28585537,0.27749613,0.27773259,0.27600331,0.27360107,0.27388839,0.27333073,0.27515425,0.27702009,0.2694618,0.26933469,0.2675772,0.26777881,0.25710299,0.2515334,0.24936234,0.23809616,0.23109845,0.23816343,0.23550811,0.24304089,0.24511566,0.24969895,0.2558643,0.27252313,0.27613636,0.27662306,0.2793396,0.27318663,0.27466936,0.26656278,0.27127787,0.27268132,0.26845217,0.2749224,0.26444117,0.26862086,0.26698412,0.26943781,0.27011582,0.27351884,0.2692004,0.2777168,0.27491558
VIPDEMO-033,0,sim_PLS_VIP_demo,1,6.7986,0.27582769,0.26972133,0.27703312,0.27401233,0.27207761,0.27401075,0.26970505,0.27204977,0.2693098,0.27169117,0.26546166,0.26626544,0.26770833,0.26860554,0.26247898,0.27369461,0.26867693,0.26915483,0.27258853,0.26756671,0.26889714,0.27234514,0.27062672,0.28101614,0.27820572,0.28098457,0.28458681,0.28917385,0.29135846,0.2889605,0.2952681,0.29431961,0.29581659,0.28762385,0.28426057,0.28001278,0.2751366,0.27627132,0.27125258,0.27363361,0.2779779,0.27660184,0.27805951,0.26861166,0.27573549,0.27371318,0.27324161,0.27475457,0.27150855,0.26979037,0.26449143,0.27030458,0.26057411,0.2615797,0.25510727,0.2539654,0.25868438,0.25812896,0.26444185,0.26482119,0.26908828,0.27287664,0.26711818,0.27400308,0.26609411,0.27133461,0.2723783,0.27148492,0.27697921,0.27738062,0.27730102,0.2870677,0.28147927,0.28119174,0.27901331,0.28096952,0.27812911,0.27414242,0.28178718,0.27737656,0.27425769
VIPDEMO-034,0,sim_PLS_VIP_demo,1,7.614,0.28089749,0.27754322,0.27785474,0.28151468,0.27931083,0.28157204,0.28556349,0.286477,0.28529293,0.28333486,0.29172039,0.29226503,0.28839838,0.29152149,0.2917084,0.29309252,0.2861667,0.28637381,0.28214027,0.28417803,0.28336699,0.28676993,0.28206402,0.28881599,0.28580641,0.29311835,0.2995566,0.30427378,0.31126802,0.31566878,0.31862943,0.31538279,0.31224134,0.30760398,0.30481098,0.295176,0.29869419,0.29052906,0.28410935,0.27591948,0.27988427,0.27932953,0.28064235,0.28030993,0.28466647,0.27513837,0.27763032,0.27839442,0.27778776,0.26761095,0.27150675,0.26023487,0.25433305,0.25187956,0.25012197,0.2474719,0.24931713,0.25142198,0.25845672,0.25972924,0.26654604,0.27159382,0.27373012,0.27775254,0.2746513,0.27586207,0.28204594,0.28053394,0.28332446,0.28632269,0.28981533,0.28536984,0.29309626,0.28457277,0.28845798,0.27960965,0.28052183,0.27839078,0.27881461,0.27800474,0.27296325
VIPDEMO-035,0,sim_PLS_VIP_demo,1,4.7221,0.28238976,0.291495,0.28899061,0.29124923,0.28764552,0.28672446,0.29159674,0.28857887,0.28753577,0.28955333,0.29320942,0.2932413,0.29439267,0.290138,0.28556641,0.29451139,0.29071588,0.28679533,0.28835385,0.28851968,0.28587927,0.28934725,0.28877277,0.28746294,0.28575769,0.28177187,0.27284072,0.27218801,0.27307428,0.26916546,0.26871242,0.26740215,0.27297086,0.27403774,0.27905307,0.28360286,0.29009554,0.28951791,0.28847081,0.28707777,0.2972369,0.29571498,0.28770221,0.29351028,0.29617012,0.29253551,0.29445127,0.2919823,0.29781614,0.29820054,0.2951131,0.30024398,0.29839864,0.30572751,0.3012916,0.30740393,0.30295249,0.30142663,0.30540184,0.30070918,0.29907957,0.29754605,0.29258708,0.29736689,0.29374804,0.29825032,0.30074932,0.3006448,0.2981485,0.2975225,0.307142,0.30266378,0.30513139,0.29819589,0.2999283,0.30060924,0.29622408,0.29939563,0.29585089,0.29765412,0.29264276
VIPDEMO-036,0,sim_PLS_VIP_demo,1,6.7732,0.30704821,0.30854899,0.31391952,0.30806299,0.30468449,0.31306532,0.31081009,0.31288213,0.30754657,0.3047675,0.30429832,0.30248518,0.30712495,0.29759566,0.30170096,0.30406586,0.30399315,0.31724752,0.31235407,0.31398898,0.31605499,0.31111077,0.31635731,0.3193132,0.31664925,0.32651286,0.32588978,0.32981888,0.32922681,0.32719487,0.32557039,0.33185825,0.32998903,0.32870206,0.31987964,0.3290767,0.31755251,0.3156295,0.32016047,0.31423963,0.31999114,0.31762062,0.31290751,0.30928147,0.31556656,0.31707672,0.31211364,0.31620386,0.31301384,0.30999735,0.30953403,0.30479572,0.30289239,0.28748957,0.29055535,0.29631347,0.29829963,0.30350389,0.3045848,0.30935854,0.31101954,0.31006103,0.31353783,0.31803731,0.31635576,0.31520866,0.32271705,0.31554922,0.31813696,0.31274523,0.3173112,0.31479079,0.31157075,0.32180475,0.3158731,0.31719443,0.31898081,0.32201091,0.31705641,0.32176532,0.32039018
VIPDEMO-037,0,sim_PLS_VIP_demo,1,3.822,0.31562971,0.31801241,0.31561411,0.31782924,0.30962859,0.31050879,0.30874517,0.30433657,0.30632793,0.30657572,0.30330193,0.3016568,0.29870581,0.29640773,0.29417155,0.29637836,0.29563943,0.29761207,0.29782514,0.29553882,0.29716688,0.29234654,0.28631746,0.28268886,0.28270927,0.27061669,0.26620676,0.25774562,0.25591186,0.24387009,0.25172838,0.24564851,0.24665223,0.25608165,0.25651265,0.26165813,0.26163423,0.27038302,0.27128012,0.26893654,0.26829983,0.2690361,0.27636215,0.27226447,0.26658464,0.26659781,0.26474119,0.27306458,0.27095443,0.26904628,0.27445663,0.27057268,0.27951526,0.28330279,0.27781287,0.27873183,0.28212842,0.27910888,0.2673446,0.26120378,0.26390472,0.25598091,0.2542869,0.25167709,0.25217977,0.24070464,0.24476219,0.24618055,0.24416158,0.23937284,0.23788063,0.24708122,0.23452898,0.2444312,0.23693136,0.23509735,0.23133909,0.23477897,0.23329203,0.23190387,0.22982724
VIPDEMO-038,0,sim_PLS_VIP_demo,1,7.1195,0.29776324,0.29883497,0.30012029,0.29987783,0.29762831,0.29937511,0.30003773,0.29286532,0.29539409,0.29327222,0.28925895,0.28876771,0.28851257,0.28826471,0.28672667,0.29139532,0.29073662,0.29808305,0.29359216,0.29791072,0.29192323,0.29590409,0.29369624,0.29658443,0.30510782,0.30637545,0.30353488,0.31286974,0.31104526,0.31651445,0.31939886,0.31604263,0.31175573,0.31334505,0.30545846,0.30474891,0.30096228,0.29673366,0.28983602,0.29156865,0.28795872,0.29006852,0.28855283,0.28812561,0.28738607,0.28881463,0.28680277,0.28753776,0.28583686,0.28073066,0.27686868,0.27480542,0.26948679,0.26931481,0.26501849,0.26019749,0.26104512,0.27114746,0.26487807,0.27099963,0.2751106,0.27958912,0.27436525,0.27876059,0.27984393,0.28696538,0.28278137,0.28887245,0.28348186,0.28498712,0.29012886,0.29061356,0.28899629,0.28903496,0.28829173,0.2894702,0.28418522,0.28079702,0.27670029,0.27868088,0.27660489
VIPDEMO-039,0,sim_PLS_VIP_demo,1,6.6616,0.26785492,0.2728541,0.27001184,0.27355882,0.27142223,0.26818326,0.27619502,0.26888131,0.26912344,0.27722439,0.2732995,0.28032664,0.27606694,0.27956513,0.28293269,0.28163557,0.2794765,0.27716275,0.28012459,0.2770018,0.27889424,0.28843265,0.28046339,0.28878392,0.28870642,0.28731008,0.29797277,0.29750238,0.30471397,0.31133794,0.30118742,0.30980159,0.30221515,0.30187448,0.29792218,0.29645544,0.29195722,0.29207763,0.2874621,0.28423871,0.29259826,0.28974542,0.28539435,0.29160078,0.29061571,0.289782,0.28445802,0.28617618,0.29089458,0.28780307,0.28672356,0.27949558,0.27960022,0.27631161,0.2776825,0.28255043,0.27390526,0.28017645,0.28309862,0.28442222,0.27825875,0.2891878,0.29001291,0.29743309,0.2926993,0.29899955,0.29752473,0.29126848,0.29627319,0.30354594,0.30323346,0.30314667,0.31176888,0.30406192,0.30551914,0.30534477,0.30416381,0.29527799,0.30032834,0.30246435,0.30565639
VIPDEMO-040,0,sim_PLS_VIP_demo,1,3.9567,0.27488289,0.26673799,0.27335358,0.27206945,0.27099698,0.27687776,0.27491977,0.27214519,0.27479277,0.27539589,0.27021762,0.27139441,0.27675404,0.26903907,0.27766462,0.27739762,0.27833015,0.28118285,0.28543937,0.28882277,0.28736133,0.28695358,0.28018093,0.27661761,0.28062019,0.27662482,0.27008364,0.26817064,0.26625514,0.26409958,0.25322291,0.25869636,0.26735936,0.26953038,0.27809215,0.27748041,0.28726134,0.29695287,0.29230064,0.29587027,0.29139253,0.30059058,0.30146968,0.3006143,0.30130155,0.30133231,0.30226195,0.30681262,0.31428342,0.30892547,0.31444759,0.31832498,0.32441966,0.3319725,0.33434017,0.33317392,0.33441691,0.33096323,0.32722182,0.32277653,0.32509208,0.31877658,0.31790828,0.32390637,0.31453034,0.32190158,0.32356474,0.32819273,0.3313637,0.3364168,0.34476705,0.34295238,0.34403432,0.33976185,0.34474379,0.34231652,0.33756151,0.3283433,0.32800845,0.3273592,0.32849119
VIPDEMO-041,0,sim_PLS_VIP_demo,1,3.9198,0.28083761,0.27712664,0.28884379,0.28684931,0.28237773,0.28715148,0.2892492,0.29439612,0.29236277,0.30037137,0.3068734,0.30750977,0.31289763,0.31288961,0.30965559,0.31060116,0.29635433,0.30201725,0.29874221,0.30770236,0.30099629,0.2956887,0.29992605,0.29760274,0.29809183,0.29089727,0.29022477,0.27956347,0.2762424,0.27713405,0.27623723,0.27339416,0.28019168,0.29072536,0.29017943,0.30805181,0.30247978,0.31633967,0.31366744,0.32224612,0.32451476,0.32268286,0.32681117,0.32697307,0.32750727,0.32873798,0.33048527,0.33298099,0.33899189,0.34250678,0.34595925,0.34607077,0.36273396,0.36964397,0.37394477,0.37606158,0.37336671,0.37034822,0.36814591,0.36472907,0.35600205,0.35377946,0.35532349,0.34878129,0.35379478,0.3506992,0.35321424,0.35633423,0.35667332,0.36069518,0.36918602,0.36971901,0.37481761,0.37132894,0.37051746,0.36791104,0.36974922,0.36450247,0.36328649,0.36727133,0.36494684
VIPDEMO-042,0,sim_PLS_VIP_demo,1,3.8435,0.33661098,0.34397867,0.34004756,0.3361079,0.33713852,0.33959276,0.33064399,0.33034598,0.32801345,0.31923905,0.31752275,0.31909352,0.31420215,0.31627041,0.31627943,0.31616368,0.31492581,0.31508071,0.3166059,0.30786669,0.31229933,0.30657576,0.31108892,0.30036925,0.2950918,0.28823701,0.27885431,0.27603891,0.26146896,0.26535762,0.25994362,0.26479768,0.26255129,0.26444259,0.27166586,0.27566273,0.27924062,0.28503752,0.28674054,0.2881995,0.28644365,0.28675982,0.28227805,0.2781818,0.28606364,0.28432541,0.28222154,0.28549528,0.28614941,0.28596355,0.28396361,0.29254239,0.29980693,0.30376296,0.30524073,0.30247735,0.303848,0.29983583,0.29455962,0.28314692,0.28283758,0.26818248,0.26366605,0.26456647,0.26390189,0.26309996,0.25908881,0.25482027,0.26159523,0.25484331,0.25819505,0.25127833,0.25413759,0.25568075,0.24646137,0.24426196,0.24907728,0.24561508,0.24195469,0.23620497,0.2365835
VIPDEMO-043,0,sim_PLS_VIP_demo,1,7.0944,0.30055416,0.29789785,0.29510272,0.29737008,0.29530221,0.30154208,0.29654783,0.2973796,0.29811072,0.29522028,0.29655166,0.29562527,0.29420759,0.29661337,0.29467146,0.29815258,0.2979975,0.29916494,0.30279768,0.30010925,0.30195226,0.30124678,0.30479118,0.30625336,0.30627981,0.30793853,0.31551625,0.32676399,0.32537372,0.32497038,0.32262316,0.33162883,0.32184106,0.3160807,0.31551067,0.31575023,0.30876299,0.30307457,0.30224346,0.30190313,0.30505528,0.30129335,0.30288745,0.30352514,0.30738065,0.30075641,0.29794279,0.30422945,0.29571214,0.29822399,0.29138191,0.2918527,0.28061902,0.28591272,0.28389159,0.27961955,0.2801891,0.28745886,0.28801196,0.28831622,0.29532302,0.29844401,0.29941257,0.30160146,0.30164918,0.30038579,0.298853,0.30494571,0.30478923,0.29828938,0.30388578,0.30250224,0.30277148,0.30421164,0.30504866,0.30156074,0.30436006,0.30305367,0.30426245,0.30572997,0.30483802
VIPDEMO-044,0,sim_PLS_VIP_demo,1,6.3216,0.31205164,0.31395273,0.31283353,0.31656848,0.31442212,0.31260043,0.31593783,0.31219141,0.32276378,0.32463245,0.32772938,0.32990553,0.32279464,0.33023696,0.32341138,0.33123372,0.32884026,0.32325878,0.32866966,0.32269912,0.32310729,0.32115182,0.32411745,0.32690162,0.33399746,0.33336264,0.32915549,0.33938332,0.33457568,0.34067872,0.34397635,0.34273601,0.33771791,0.3476638,0.33913566,0.33615055,0.33793895,0.33429242,0.33874311,0.33508382,0.33379549,0.33243179,0.33383546,0.33881758,0.33262274,0.33355009,0.33265256,0.33675651,0.33862477,0.33894074,0.33845404,0.32890938,0.33168144,0.33066799,0.32611491,0.32366221,0.32741228,0.3321981,0.33454302,0.33193043,0.34078505,0.35001473,0.34075074,0.34932891,0.34805698,0.3480782,0.34611476,0.34293762,0.35040178,0.34558545,0.34475142,0.33689623,0.33908494,0.34242354,0.35156423,0.34506429,0.35427758,0.35544491,0.35193096,0.35083987,0.35413674
VIPDEMO-045,0,sim_PLS_VIP_demo,1,4.4041,0.31593793,0.31834574,0.30877264,0.31932444,0.31943153,0.31228616,0.31428305,0.31730311,0.31473146,0.31147521,0.31348888,0.31469509,0.31272128,0.31581447,0.31351369,0.3149585,0.31027077,0.31311278,0.31206174,0.31184504,0.3158133,0.30845115,0.31575092,0.3088371,0.31038166,0.30553714,0.29482883,0.29300893,0.29393652,0.29010031,0.28519931,0.28699607,0.29389232,0.29337414,0.30330384,0.30540242,0.30546746,0.30912956,0.31055367,0.31554896,0.31445987,0.31903876,0.32024127,0.31701593,0.31442581,0.31903635,0.31659275,0.31759872,0.31707175,0.3223201,0.31981778,0.32748766,0.32567853,0.33239643,0.33819274,0.33466108,0.33404413,0.33072998,0.32655088,0.32810863,0.32636765,0.31998009,0.32083676,0.31390742,0.32057524,0.31951643,0.32070397,0.32485939,0.32899173,0.32647928,0.33415691,0.33622353,0.33771344,0.33164929,0.33762817,0.32738805,0.32789914,0.3240887,0.32518562,0.32112085,0.326142
VIPDEMO-046,0,sim_PLS_VIP_demo,1,6.2721,0.31525396,0.32135859,0.31790199,0.32225061,0.3191065,0.31992333,0.31965673,0.32034071,0.31914406,0.32257811,0.3171419,0.31385769,0.32277046,0.31516875,0.31887955,0.31656965,0.31180928,0.31584141,0.32051818,0.31842173,0.31760055,0.31611025,0.31003845,0.31363587,0.31364773,0.31743404,0.31646134,0.31735291,0.31740324,0.31753781,0.31923317,0.31685961,0.3159653,0.31319114,0.32010721,0.30549514,0.31423304,0.31423957,0.31174689,0.30606312,0.30874858,0.30234522,0.31003381,0.30383119,0.30300573,0.29988275,0.30543644,0.30567923,0.2978594,0.30089086,0.29901162,0.3014364,0.29591792,0.29526951,0.29084627,0.29337599,0.29527493,0.29053517,0.28911414,0.29689914,0.29817433,0.29550167,0.29543978,0.29851521,0.29831998,0.30213663,0.29544059,0.28900992,0.28396252,0.27830543,0.2808209,0.26706527,0.26950789,0.27136741,0.28110206,0.28162812,0.28324027,0.28867882,0.28521994,0.28540304,0.28771915
VIPDEMO-047,0,sim_PLS_VIP_demo,1,6.4301,0.34331816,0.33787925,0.33655093,0.34399383,0.34026655,0.33873305,0.33877768,0.34021992,0.3392985,0.33425619,0.32929983,0.33336865,0.33149927,0.32876098,0.33210096,0.33186042,0.32993273,0.33290364,0.33592896,0.33137407,0.33084181,0.33086961,0.3303868,0.33011627,0.33549477,0.33715637,0.33827564,0.33443474,0.33504702,0.33627647,0.33731888,0.32993509,0.32961072,0.32851722,0.32911263,0.32844983,0.32748655,0.32555749,0.32572134,0.32272016,0.32105188,0.32411417,0.32114933,0.32312534,0.32114183,0.32239898,0.31481881,0.31897301,0.32003512,0.31434383,0.31039259,0.30825261,0.30407348,0.3063386,0.29971564,0.30630824,0.29805732,0.30290777,0.30411325,0.30498156,0.3020519,0.31038903,0.30995018,0.30660461,0.30390024,0.30554002,0.30425796,0.31135874,0.30775564,0.3058417,0.30777165,0.30864801,0.30862641,0.30679147,0.30435571,0.30952694,0.30135502,0.2987331,0.30064717,0.30193844,0.29694801
VIPDEMO-048,0,sim_PLS_VIP_demo,1,6.4444,0.28069299,0.28562781,0.28388731,0.28261798,0.28098005,0.28005633,0.28103979,0.27767246,0.28133905,0.27998885,0.28178586,0.2720702,0.27283048,0.27978933,0.27806814,0.27911609,0.28104435,0.28030227,0.28461437,0.28427889,0.28457984,0.28826806,0.28564519,0.29360188,0.29354985,0.29469088,0.29609735,0.29382462,0.30092312,0.30667369,0.30274486,0.29916906,0.30312842,0.30000702,0.30052506,0.29837681,0.28985824,0.28845047,0.2866251,0.29242528,0.29270142,0.29402881,0.29327536,0.29030341,0.29048198,0.29290983,0.29208492,0.29084007,0.29323941,0.29143738,0.28855355,0.29297456,0.28719447,0.29472151,0.28751651,0.28862217,0.29080443,0.28777844,0.28325294,0.2952934,0.29526505,0.28756898,0.29516215,0.29917958,0.29046545,0.29563983,0.29339122,0.29721864,0.29232247,0.28329591,0.28102152,0.28120668,0.2780351,0.2809931,0.29028974,0.2846315,0.29099968,0.30008768,0.30107538,0.29707886,0.29268295
VIPDEMO-049,0,sim_PLS_VIP_demo,1,7.9798,0.2642652,0.26913397,0.26567823,0.26578385,0.27268315,0.27333715,0.26877346,0.27307132,0.27120316,0.26620921,0.27028681,0.26823772,0.27014342,0.26777355,0.26968136,0.26965772,0.27992099,0.27754063,0.27600283,0.27769964,0.27554003,0.28673376,0.28535075,0.2878629,0.29776751,0.30072281,0.3063387,0.31912097,0.32230631,0.33224921,0.33613273,0.3296784,0.3266517,0.33137782,0.31931664,0.31008354,0.30114761,0.29413511,0.29554498,0.29617385,0.29407907,0.30000934,0.28986182,0.2923155,0.29458459,0.29331448,0.29140154,0.29466751,0.29365665,0.2892122,0.28514535,0.28230945,0.28135244,0.27340428,0.2684261,0.26948066,0.27488703,0.2783604,0.27836407,0.29148563,0.29748373,0.29610946,0.29717282,0.30005958,0.30605142,0.30737862,0.31231395,0.31275978,0.31750655,0.32018817,0.32349474,0.31891631,0.3274848,0.32514598,0.3223659,0.32122028,0.31939053,0.31571462,0.31405724,0.31504288,0.31350334
VIPDEMO-050,0,sim_PLS_VIP_demo,1,7.2248,0.28223194,0.28908168,0.28188844,0.28870608,0.28595239,0.28229725,0.282086,0.28879894,0.28502313,0.28846604,0.28772257,0.28591856,0.2873165,0.29052803,0.28937252,0.29286543,0.29252647,0.29326772,0.29468232,0.29087138,0.29733761,0.2972519,0.29206861,0.30669394,0.30992477,0.31636489,0.30901604,0.32568848,0.32406097,0.32919839,0.32608166,0.33297622,0.32789388,0.33064015,0.31992567,0.31788465,0.31829588,0.30954694,0.30505081,0.30953304,0.30463523,0.3054143,0.30852706,0.30886,0.31291697,0.30696994,0.3076664,0.31289353,0.30904465,0.30545196,0.3049925,0.30373073,0.29720754,0.29582089,0.28945808,0.29909458,0.29192356,0.30190446,0.30381561,0.31110596,0.31003153,0.31680047,0.31694569,0.31417355,0.32084067,0.3226,0.31882049,0.32650985,0.32962635,0.32921098,0.33559061,0.33407166,0.33557468,0.3374118,0.33217357,0.33474521,0.33401372,0.33185361,0.32742611,0.32947604,0.33350534
VIPDEMO-051,0,sim_PLS_VIP_demo,1,5.1499,0.28022041,0.27616872,0.28473547,0.27716964,0.28117504,0.28406509,0.27811102,0.27577115,0.27809597,0.2849751,0.28453365,0.2689034,0.27977574,0.28064211,0.27736132,0.28362908,0.29013478,0.28600089,0.29505344,0.29250265,0.29184072,0.29036965,0.29351625,0.29614245,0.2903042,0.28725482,0.28662305,0.28491263,0.28727838,0.28383218,0.28062358,0.28866043,0.28942715,0.29144773,0.29663114,0.30421843,0.30308992,0.3057811,0.30174827,0.30547131,0.30486484,0.31321549,0.30873481,0.31087752,0.31318095,0.31515432,0.31205067,0.31449246,0.32211945,0.31913081,0.32452274,0.32115626,0.32710665,0.33033859,0.33412791,0.32916753,0.32751602,0.33152549,0.32928697,0.32697635,0.32892855,0.3310622,0.32948809,0.33302678,0.32869801,0.32686195,0.33113541,0.33447886,0.32769227,0.33241903,0.33805569,0.33565469,0.34469214,0.33034728,0.33312585,0.33992297,0.33970519,0.33787416,0.33874321,0.33870182,0.33911777
VIPDEMO-052,0,sim_PLS_VIP_demo,1,3.2353,0.29476299,0.28860013,0.28494834,0.29116505,0.28969841,0.29373252,0.29107662,0.28958346,0.29082001,0.29602821,0.29299103,0.30168067,0.29931166,0.29587337,0.29400376,0.29937234,0.2956767,0.29331015,0.29396161,0.29662064,0.29232783,0.29612899,0.29637114,0.28587142,0.28143786,0.28182998,0.26642765,0.26612101,0.24745196,0.24462729,0.2439968,0.25588764,0.25565233,0.26560244,0.26924642,0.2818179,0.28434514,0.29472539,0.29398504,0.3031231,0.30644415,0.30467637,0.30944571,0.30930776,0.30538443,0.3073607,0.31409281,0.30863104,0.31537869,0.31795266,0.32576158,0.33462904,0.33719226,0.34236873,0.34224548,0.34835294,0.34235583,0.34400956,0.33350302,0.3341921,0.3312747,0.32113182,0.31477569,0.32147401,0.3153006,0.31592177,0.31803166,0.31789368,0.32106728,0.326891,0.32263096,0.32554925,0.32558623,0.32675574,0.32164469,0.32452161,0.32832476,0.325929,0.32340294,0.32491468,0.32584838
VIPDEMO-053,0,sim_PLS_VIP_demo,1,4.143,0.26794287,0.25593161,0.2639793,0.26593184,0.26468155,0.26280994,0.26722623,0.27074257,0.25951235,0.2576713,0.26760607,0.26461483,0.26359532,0.26613206,0.2718882,0.2618955,0.26326082,0.27506893,0.26769834,0.2767918,0.26688599,0.26817044,0.26500578,0.26958759,0.26989596,0.26742289,0.26123283,0.25586677,0.25650192,0.24789606,0.254066,0.25699316,0.25961565,0.26021028,0.26277731,0.277632,0.26922073,0.27448455,0.28463111,0.28640212,0.28074553,0.28226108,0.28556465,0.28783022,0.28755463,0.29110127,0.29565133,0.28733445,0.29196189,0.29371159,0.29983767,0.30264884,0.31135503,0.31077759,0.31574565,0.3198928,0.31884721,0.3144701,0.31293756,0.30845922,0.30775403,0.30377815,0.30579001,0.29710294,0.29738239,0.30232465,0.3024144,0.30583136,0.31123785,0.30854116,0.31538371,0.31799456,0.32489361,0.32306197,0.31991994,0.31630419,0.31033134,0.31236427,0.30933377,0.31204142,0.30754623
VIPDEMO-054,0,sim_PLS_VIP_demo,1,3.374,0.31387267,0.31344825,0.31653152,0.31643298,0.31234469,0.31510487,0.30881134,0.3183286,0.30739699,0.31283515,0.31911191,0.3113841,0.31349113,0.31181632,0.31077997,0.31263039,0.30839079,0.30207334,0.31395667,0.30556726,0.30534877,0.30080123,0.29975004,0.29035873,0.29147984,0.28422027,0.27241013,0.26886531,0.26372906,0.26581211,0.2596064,0.25673548,0.26316336,0.27105534,0.27951372,0.28011364,0.2860774,0.28770574,0.29362664,0.29517356,0.29198279,0.29569691,0.2965858,0.29623453,0.29445144,0.2971226,0.29456158,0.29979511,0.29910734,0.30417985,0.30123307,0.31561602,0.31014454,0.31459172,0.32101831,0.32047736,0.32015307,0.31442142,0.30844152,0.30764305,0.29842794,0.28978331,0.29028967,0.29202516,0.29133073,0.28421979,0.28419738,0.28543307,0.28244482,0.2809169,0.27780494,0.2827632,0.28436629,0.27930538,0.28019421,0.28181514,0.27845682,0.28144071,0.28105769,0.27626812,0.27912262
VIPDEMO-055,0,sim_PLS_VIP_demo,1,6.9223,0.28520028,0.28809754,0.28358247,0.2828886,0.28378523,0.28370226,0.28428835,0.29048141,0.28686541,0.28681693,0.29105512,0.29077015,0.28721696,0.29343101,0.28989101,0.2926694,0.29153033,0.28755796,0.29410418,0.29076515,0.29576384,0.29503485,0.29761027,0.29879376,0.30554204,0.30845997,0.30662737,0.31648722,0.32180593,0.32771289,0.32496063,0.32628782,0.32410976,0.32355647,0.31310698,0.31234507,0.3097469,0.30855186,0.30650313,0.30875087,0.31432425,0.30986876,0.3111198,0.31048549,0.31050511,0.31031175,0.30820483,0.30896166,0.30465092,0.30788177,0.31069233,0.30931892,0.30935373,0.30003737,0.30210436,0.29777929,0.29842787,0.30399953,0.31076057,0.31295685,0.3133267,0.31395045,0.31523084,0.32248779,0.32514102,0.32608534,0.32666555,0.32664506,0.33404559,0.33021594,0.32928471,0.3314836,0.33374948,0.32601898,0.33323922,0.33250127,0.33003531,0.33044394,0.33390685,0.3373325,0.33576685
VIPDEMO-056,0,sim_PLS_VIP_demo,1,6.9667,0.32158907,0.32485457,0.31775751,0.3254895,0.32281688,0.32501491,0.32532523,0.32252175,0.31955643,0.32618744,0.32250747,0.31911478,0.31662344,0.32487476,0.32116274,0.31701793,0.31736916,0.32108527,0.31791543,0.32818945,0.32628292,0.32565206,0.32734088,0.32568097,0.32923493,0.33638173,0.33549136,0.33834663,0.34009681,0.34412534,0.34217669,0.34469945,0.34232666,0.33881155,0.33155999,0.32713895,0.32785449,0.32379481,0.3224948,0.32029043,0.31181896,0.31554603,0.31587649,0.31885247,0.31574022,0.31646042,0.31755188,0.31940494,0.31465683,0.31509072,0.31015413,0.30351539,0.30903609,0.30408498,0.29708325,0.29746094,0.30142496,0.29971737,0.30273195,0.30608091,0.30897048,0.30769396,0.31159152,0.3119406,0.31029732,0.31163234,0.31359251,0.30631245,0.30700966,0.30292508,0.3015202,0.29756068,0.29594558,0.29552659,0.3040104,0.3003447,0.30260001,0.30647523,0.30562142,0.31749098,0.30886739
VIPDEMO-057,0,sim_PLS_VIP_demo,1,4.0713,0.31016798,0.3099724,0.30680345,0.31284682,0.30382474,0.30414517,0.31069234,0.31141947,0.31189877,0.31484423,0.30700818,0.30998009,0.31118745,0.31188971,0.30871535,0.30190696,0.30672667,0.29955919,0.30206886,0.29663594,0.30376867,0.29385186,0.29676922,0.28706168,0.28731157,0.28201975,0.27687673,0.26940285,0.27348592,0.26312414,0.26248505,0.26272894,0.26606707,0.27144618,0.27088083,0.2797548,0.2831922,0.28786749,0.28450692,0.28436695,0.28769837,0.2922973,0.29359161,0.29052041,0.29028177,0.28354916,0.28759239,0.28698547,0.28728661,0.29613685,0.29638107,0.30175267,0.31266369,0.30928627,0.30907856,0.31808018,0.30163359,0.30986611,0.3081706,0.29308427,0.2967354,0.28211218,0.28113703,0.284243,0.28049139,0.28123607,0.27490269,0.27552217,0.27176944,0.27269236,0.27286209,0.27253542,0.275144,0.27207765,0.27167457,0.2715012,0.27267425,0.27102983,0.26849864,0.26397451,0.26834911
VIPDEMO-058,0,sim_PLS_VIP_demo,1,7.7424,0.288213,0.28971124,0.29431399,0.28915077,0.28733622,0.29460888,0.29272199,0.29009239,0.29001871,0.29274558,0.29598627,0.29501082,0.29927517,0.29876096,0.3015349,0.2990175,0.29271102,0.29121163,0.29537285,0.29170354,0.29469302,0.29346125,0.29726801,0.30632285,0.30133561,0.30769098,0.3137755,0.32470181,0.33218892,0.32824292,0.32974722,0.33032571,0.32767463,0.32420476,0.31399301,0.31252729,0.31259558,0.30441195,0.29676478,0.30075479,0.29859596,0.30000456,0.29869711,0.29890236,0.29814671,0.30188753,0.29918618,0.29716255,0.29333578,0.29224696,0.28355649,0.28209676,0.27440476,0.27052185,0.26602113,0.26675054,0.26516719,0.26867675,0.27208504,0.28329698,0.28626915,0.29618906,0.29804692,0.30337663,0.29959382,0.29964843,0.30117905,0.29394024,0.29639988,0.29229019,0.28281831,0.27841532,0.27914821,0.28135674,0.2952991,0.29292554,0.29833401,0.29929842,0.30477132,0.30689943,0.30456588
VIPDEMO-059,0,sim_PLS_VIP_demo,1,4.6155,0.32742022,0.33029845,0.32845787,0.32780805,0.32251132,0.32677656,0.32459078,0.31914345,0.31724512,0.31141159,0.31021008,0.31100639,0.30919356,0.30786477,0.31077748,0.31434759,0.30770019,0.31203056,0.31492916,0.32141205,0.31372717,0.31469622,0.30734098,0.30389975,0.30710344,0.30334514,0.29913835,0.29414681,0.28590367,0.288812,0.27806926,0.28335591,0.28891809,0.2877915,0.29608573,0.28986108,0.29420451,0.29787837,0.30351334,0.2985317,0.30239466,0.29585478,0.2988639,0.29399458,0.30356651,0.30127358,0.30270365,0.29672977,0.30064493,0.29873339,0.30269488,0.29905014,0.30334023,0.30440513,0.30343923,0.303938,0.29996057,0.30037917,0.29824947,0.29426731,0.29114966,0.2877128,0.29187099,0.281896,0.28183798,0.28705394,0.27941057,0.27362532,0.27946029,0.27310979,0.26638024,0.26650142,0.2630549,0.26297509,0.2615183,0.2686968,0.2684631,0.2769477,0.27286668,0.2792743,0.27294434
VIPDEMO-060,0,sim_PLS_VIP_demo,1,5.3296,0.32082835,0.31961031,0.32342123,0.32304848,0.32344708,0.32251054,0.32233868,0.32182268,0.32276494,0.32697293,0.32315259,0.32747575,0.32316103,0.32377085,0.32284892,0.31720725,0.32423598,0.32353739,0.32331092,0.32034085,0.32402728,0.32599293,0.31663661,0.31513133,0.31201519,0.31515199,0.30806109,0.31088897,0.30865343,0.31293265,0.30818436,0.3124522,0.31054965,0.31512333,0.31056476,0.31329273,0.31479107,0.31068272,0.32267257,0.31370646,0.3124559,0.31233688,0.32057876,0.31928466,0.31793951,0.31777439,0.31997355,0.31812778,0.31598403,0.31976131,0.32095945,0.31939097,0.31605384,0.31949829,0.31594813,0.31587975,0.31786738,0.31830334,0.31833199,0.32017851,0.31664681,0.31319702,0.31610143,0.31919481,0.31643178,0.31691117,0.31930779,0.31823538,0.31097697,0.31289363,0.31249421,0.30866751,0.3104423,0.31304687,0.31598374,0.31449123,0.3173612,0.31453947,0.31936535,0.31709065,0.31420401
VIPDEMO-061,0,sim_PLS_VIP_demo,1,5.9048,0.32714105,0.32017601,0.32016675,0.32907245,0.32105022,0.3206817,0.32436967,0.32444672,0.32454522,0.32714182,0.32747214,0.33040927,0.33322365,0.32900056,0.3248698,0.32660814,0.32286956,0.32663653,0.32641423,0.32615635,0.31624582,0.31923023,0.32344249,0.32201298,0.32150343,0.31690588,0.31613873,0.31972648,0.31755496,0.31527228,0.31814375,0.31972446,0.32030197,0.32202592,0.32000502,0.3135459,0.31538276,0.31971173,0.32004998,0.3124504,0.31567409,0.32568063,0.31776985,0.31760199,0.31859808,0.31731075,0.31698221,0.31553796,0.32334543,0.3194384,0.3164236,0.31387872,0.31839566,0.31952868,0.31689201,0.3185733,0.31959823,0.31640405,0.31567124,0.31410994,0.31261186,0.31528166,0.31611913,0.31819627,0.31485035,0.31468338,0.31251329,0.31734799,0.31338022,0.30867603,0.31384919,0.31613443,0.31304347,0.30702189,0.30657792,0.31449875,0.31461976,0.31295521,0.3163967,0.31567013,0.31944693
VIPDEMO-062,0,sim_PLS_VIP_demo,1,7.9899,0.25580338,0.25912425,0.26104001,0.25704029,0.25702919,0.26175735,0.2613641,0.26134609,0.26075911,0.26286772,0.25650291,0.2591109,0.24927559,0.25797795,0.26372378,0.26642564,0.26488881,0.27339234,0.27894702,0.27617671,0.27801939,0.28347991,0.28661724,0.28715467,0.29901135,0.30586521,0.30773077,0.31744062,0.32337494,0.3219563,0.32630543,0.32708932,0.32348432,0.32595173,0.31577411,0.30978066,0.30512909,0.30501921,0.29696513,0.29952804,0.29203549,0.29622561,0.29774786,0.3018838,0.29897896,0.2956375,0.30395521,0.30175766,0.29636259,0.29615017,0.29868886,0.29097307,0.28377293,0.28571268,0.28118656,0.28552054,0.28620727,0.2859583,0.29345421,0.30761824,0.30221206,0.30733918,0.31632052,0.31795626,0.31865177,0.32212507,0.31546577,0.31762112,0.32587962,0.32581125,0.32998294,0.33201491,0.32787006,0.33155437,0.33294164,0.33355884,0.33347209,0.32915705,0.33431376,0.33099045,0.32676035
VIPDEMO-063,0,sim_PLS_VIP_demo,1,7.4844,0.30515626,0.30777183,0.30775319,0.30581734,0.30586135,0.31007683,0.30647743,0.30187166,0.30197985,0.30270136,0.2989265,0.29594305,0.30116961,0.29562274,0.29599419,0.29922772,0.30054428,0.30196018,0.30101772,0.30078845,0.30482784,0.30718802,0.31284105,0.31415524,0.31332393,0.32244461,0.32455522,0.3317273,0.33530313,0.33929098,0.34296275,0.33830289,0.33505545,0.33405881,0.31469932,0.31254778,0.31800655,0.31223669,0.30321483,0.30237601,0.29764971,0.29816678,0.29875379,0.29987305,0.29438106,0.29485292,0.29930159,0.29455234,0.29670232,0.29272601,0.27994413,0.28445692,0.27384075,0.27127637,0.26035793,0.26256468,0.26562442,0.27407599,0.27663588,0.27427128,0.28355328,0.28588497,0.29364969,0.28660929,0.28822976,0.2882664,0.28837732,0.28544746,0.29121119,0.2776514,0.27913428,0.2721259,0.27460836,0.26821955,0.27386811,0.28099703,0.28061798,0.282843,0.28755009,0.28596365,0.28660708
VIPDEMO-064,0,sim_PLS_VIP_demo,1,3.7216,0.29986774,0.29636163,0.29846166,0.29732612,0.29807933,0.30087554,0.29996699,0.29880352,0.2917497,0.29549583,0.29092067,0.29034084,0.29386716,0.29264518,0.28884685,0.29272619,0.29232261,0.29372288,0.29614302,0.29629962,0.29101628,0.29286464,0.28296642,0.28600457,0.28365774,0.26855438,0.26808673,0.26042844,0.2564931,0.25269403,0.24631868,0.24933564,0.25362975,0.25655404,0.26231263,0.26827557,0.27792263,0.28499538,0.28003338,0.28619952,0.28425572,0.28712944,0.28552292,0.2889632,0.27939779,0.28097798,0.28984758,0.29247267,0.28455651,0.28797919,0.29458181,0.30316167,0.30274503,0.30866796,0.31340569,0.31117551,0.31327889,0.30530755,0.30183001,0.29703857,0.29804257,0.28801415,0.28227044,0.28821544,0.27925061,0.27955619,0.27827964,0.27944706,0.27642785,0.27327119,0.26378665,0.26096478,0.2693064,0.26735151,0.26378839,0.26621811,0.27032218,0.27333695,0.27765619,0.27196178,0.27949417
VIPDEMO-065,0,sim_PLS_VIP_demo,1,7.4674,0.31872108,0.31577382,0.32072514,0.31877241,0.32145495,0.31961673,0.31869735,0.32120871,0.31734719,0.31922145,0.31743999,0.31518058,0.31638601,0.32047974,0.31590068,0.31703162,0.31748114,0.31469032,0.31063427,0.31807331,0.32210002,0.31789508,0.31809021,0.32212996,0.31925895,0.3284842,0.33235304,0.33782334,0.33932653,0.34316322,0.33679558,0.34302074,0.34016561,0.33468859,0.330023,0.3299104,0.3237186,0.3128892,0.31384856,0.31219,0.30647575,0.3127854,0.30546734,0.30953121,0.30455693,0.31019813,0.30193079,0.30317847,0.30271945,0.30060678,0.29608206,0.29696285,0.28481768,0.28499897,0.27948728,0.27731991,0.2821024,0.28552306,0.28507207,0.29344885,0.29177249,0.29725745,0.29709575,0.30381699,0.30105392,0.29965562,0.29871131,0.29832608,0.29670384,0.28956398,0.28517985,0.28370089,0.2834061,0.28248592,0.27465785,0.29108557,0.2961582,0.29973727,0.29069088,0.29734392,0.29947511
VIPDEMO-066,0,sim_PLS_VIP_demo,1,4.016,0.27455035,0.26921336,0.27019044,0.27020931,0.27098142,0.27084235,0.27295383,0.2761944,0.2731298,0.28383772,0.29040936,0.29362215,0.29395874,0.2880623,0.2836888,0.28055199,0.28256008,0.27394224,0.26801785,0.27098064,0.26477804,0.26608594,0.2648635,0.26084858,0.25821479,0.25496614,0.24642099,0.24003211,0.23902107,0.23454251,0.23212424,0.23522711,0.23616419,0.23830151,0.24945358,0.24990007,0.2531651,0.25857558,0.25647472,0.25835629,0.26418228,0.26447769,0.26046623,0.26466286,0.2669648,0.26025642,0.26192926,0.26587793,0.26823966,0.26626625,0.2687369,0.27635535,0.27871308,0.28019329,0.28371201,0.2871294,0.29421879,0.27752479,0.27797854,0.27634073,0.27050489,0.26504724,0.25847962,0.26109536,0.26195534,0.26114578,0.25636729,0.25699581,0.25566577,0.25399847,0.25452424,0.2516393,0.24962462,0.25077056,0.25189686,0.25414127,0.2516186,0.25541866,0.25799333,0.25254035,0.25669682
VIPDEMO-067,0,sim_PLS_VIP_demo,1,7.015,0.31922088,0.32368122,0.32145471,0.32424105,0.32265342,0.3182139,0.32490785,0.31836028,0.32605852,0.31886074,0.31642901,0.31684782,0.32307298,0.31976617,0.32196985,0.3156244,0.32332113,0.32276131,0.32705062,0.31958229,0.33013485,0.33033555,0.33161197,0.32995901,0.33257878,0.3404932,0.33684312,0.34514193,0.35057269,0.35051658,0.349583,0.35251211,0.35513399,0.34209333,0.34218301,0.33620792,0.33480097,0.33249401,0.32991778,0.33337007,0.33079345,0.33257843,0.32446164,0.325769,0.32984134,0.32293506,0.33001021,0.33145262,0.32642032,0.32527119,0.32636285,0.32492494,0.31831624,0.31936028,0.31160317,0.31336242,0.31383904,0.31929542,0.31951084,0.31971637,0.3183049,0.32638537,0.32788163,0.32285249,0.32989525,0.33192518,0.33019061,0.33412176,0.33451357,0.33774052,0.33883485,0.33333921,0.33946283,0.3382006,0.33401249,0.33084311,0.33306633,0.33249386,0.33148583,0.33081694,0.33568845
VIPDEMO-068,0,sim_PLS_VIP_demo,1,6.3574,0.33125934,0.33993793,0.32972608,0.32905283,0.3244485,0.32620273,0.33364231,0.33205446,0.33192417,0.3301219,0.33713695,0.33607917,0.33903787,0.33570659,0.33375295,0.33166225,0.33185435,0.32686397,0.32628779,0.32456133,0.32622393,0.32361094,0.32585193,0.31966605,0.32826385,0.33343219,0.33011696,0.33536876,0.3344559,0.33533355,0.33746158,0.33868156,0.33375643,0.32764091,0.33052421,0.32423646,0.3289123,0.32285411,0.32250336,0.32175799,0.31226803,0.31837107,0.31843043,0.31888687,0.31438391,0.31555501,0.31393786,0.31888308,0.31569922,0.31352933,0.31281933,0.30753878,0.3019305,0.30450123,0.30615332,0.30616351,0.30897574,0.30132108,0.30201689,0.30962886,0.30605725,0.3121773,0.31203273,0.31133094,0.30991345,0.31047014,0.30449662,0.30963809,0.30451894,0.30626681,0.30450914,0.30317496,0.3014085,0.30637818,0.30741638,0.30706598,0.30309567,0.30351857,0.30567572,0.30404378,0.31063966
VIPDEMO-069,0,sim_PLS_VIP_demo,1,3.758,0.29778574,0.29902021,0.30200754,0.30678232,0.30102378,0.29818264,0.29766616,0.29502028,0.29526119,0.2940312,0.297769,0.29513375,0.29555754,0.29006664,0.29630069,0.29685643,0.30060724,0.30004798,0.29872499,0.29602877,0.30234988,0.30053388,0.2970916,0.28523577,0.29153263,0.28475739,0.27698515,0.28319461,0.27262102,0.26987368,0.26830853,0.26626686,0.27451261,0.27518472,0.27543082,0.28368118,0.28980431,0.29129306,0.29405955,0.2957074,0.29689661,0.29981486,0.2998651,0.30324057,0.30023243,0.30411914,0.30323595,0.30265362,0.30659622,0.30436644,0.31250302,0.3143065,0.32063609,0.31946774,0.32508701,0.32674595,0.32645709,0.32123813,0.32407911,0.31547601,0.3091578,0.30305011,0.30034606,0.30202427,0.30445994,0.29953881,0.29700282,0.29958971,0.2910449,0.29855094,0.29142111,0.28959383,0.28288347,0.28745307,0.2902391,0.29013671,0.29054105,0.3022245,0.30326783,0.29957747,0.29952707
VIPDEMO-070,0,sim_PLS_VIP_demo,1,3.2678,0.3401329,0.34908062,0.34197668,0.34604788,0.34167656,0.34652191,0.34555179,0.34871417,0.34004387,0.33685434,0.33879438,0.33928751,0.3411514,0.34365538,0.33430089,0.33760084,0.33969867,0.32873493,0.33087124,0.32698056,0.33117518,0.32800747,0.31718621,0.320511,0.31020289,0.30270837,0.29548215,0.28178425,0.27322559,0.27047727,0.26540266,0.26434744,0.27339812,0.27415885,0.28103233,0.2900407,0.296298,0.31053821,0.31066361,0.30920869,0.31238689,0.32129275,0.31402615,0.31149258,0.31259354,0.31093828,0.31040707,0.3117018,0.31498782,0.31721267,0.32349225,0.32720407,0.33242707,0.33597637,0.3458572,0.34121767,0.33721029,0.33980927,0.32916705,0.31999535,0.31271778,0.31026821,0.30255584,0.30138216,0.30204066,0.29997366,0.29619157,0.29595077,0.29382738,0.2950048,0.29382376,0.29672896,0.29108635,0.29578019,0.29183808,0.29656642,0.29117489,0.29290441,0.28902719,0.28807988,0.29331939
VIPDEMO-071,0,sim_PLS_VIP_demo,1,7.6471,0.31283207,0.31293094,0.31168117,0.31225733,0.30939021,0.31186056,0.31250235,0.30968354,0.31033906,0.30588764,0.31238932,0.30690504,0.30295224,0.30430944,0.30749069,0.30417544,0.30726566,0.30831434,0.30020222,0.30269024,0.30241872,0.30869683,0.30713845,0.3145913,0.30784755,0.31857601,0.31959792,0.3214847,0.33606536,0.33206459,0.33345125,0.330546,0.32934135,0.32573526,0.31649487,0.30885766,0.30971916,0.30129515,0.30091488,0.29686253,0.29404758,0.2943214,0.29254038,0.29426468,0.28758853,0.29574808,0.29179795,0.29032824,0.28441413,0.27881969,0.27912367,0.27228043,0.26959119,0.26389824,0.26064321,0.26623536,0.25972068,0.26164551,0.26941024,0.2698213,0.26817076,0.27912364,0.27783594,0.27917482,0.28211865,0.28365591,0.27773921,0.28426589,0.27469135,0.2902306,0.28543646,0.29219697,0.28473893,0.28549154,0.29264776,0.28338492,0.27808938,0.27624986,0.27375487,0.27569271,0.2756285
VIPDEMO-072,0,sim_PLS_VIP_demo,1,3.0245,0.26870695,0.27433959,0.2704711,0.27538608,0.27560046,0.26320809,0.27528071,0.27053054,0.26751192,0.26273745,0.26924883,0.26897234,0.27043198,0.26915215,0.27616747,0.27265037,0.27469562,0.26979285,0.27856734,0.27065009,0.27307703,0.26947623,0.27404999,0.25871049,0.25380087,0.24744363,0.23798442,0.23265962,0.22564734,0.21652128,0.21528476,0.21906407,0.22587119,0.23580433,0.24361269,0.25038336,0.25859412,0.26952023,0.2732721,0.27910852,0.27471647,0.27577702,0.27912398,0.28283358,0.27539453,0.28374715,0.28336879,0.28388718,0.28551923,0.28911066,0.29056441,0.2952264,0.30382088,0.31036117,0.31488523,0.31458011,0.30782377,0.30885799,0.30609901,0.30041146,0.29473911,0.2902459,0.28458401,0.28679186,0.27892602,0.2782417,0.28578622,0.28230556,0.27569671,0.27940049,0.26964112,0.26822207,0.26549717,0.2677949,0.27180494,0.27606516,0.27871989,0.28195142,0.2868675,0.28626667,0.28314808
VIPDEMO-073,0,sim_PLS_VIP_demo,1,3.7383,0.25616044,0.25182408,0.2582739,0.2566322,0.25637214,0.25538349,0.2589686,0.25387802,0.25858554,0.26500635,0.26152775,0.25099768,0.2551949,0.25579307,0.25848165,0.26376235,0.26424593,0.26891446,0.2693485,0.272192,0.27630541,0.27072165,0.274585,0.2699845,0.27039092,0.26381822,0.26301129,0.25508455,0.25071838,0.2421815,0.24894868,0.2528016,0.25517476,0.26031165,0.27171965,0.27672015,0.28190479,0.2830969,0.29319308,0.29350448,0.29684979,0.30104638,0.30078417,0.30299695,0.30528172,0.30011525,0.30484062,0.31149715,0.31145604,0.31638344,0.31891315,0.32602095,0.32485048,0.33856738,0.33820654,0.34947449,0.34231361,0.33895381,0.3450917,0.33102723,0.33475356,0.32807816,0.32868222,0.32801001,0.32328393,0.32692017,0.33058162,0.32428981,0.32497184,0.32125542,0.32267218,0.33099667,0.32694318,0.33770975,0.33387509,0.32878535,0.33713459,0.33323496,0.34214143,0.34153648,0.34516211
VIPDEMO-074,0,sim_PLS_VIP_demo,1,7.8725,0.31351792,0.30822483,0.31482965,0.31478986,0.30639002,0.31072129,0.30904087,0.30880346,0.3087755,0.30610927,0.31083407,0.30499268,0.30943495,0.30943301,0.31242129,0.30469992,0.31342794,0.30223769,0.30451764,0.30860729,0.30761831,0.30676887,0.31106161,0.3168706,0.31981148,0.32283242,0.32894966,0.33322566,0.34679576,0.33714886,0.33924078,0.34104191,0.33824302,0.33571566,0.32831094,0.31565543,0.3121091,0.30883666,0.31005568,0.30836517,0.30674917,0.30799809,0.29692242,0.30756303,0.30206757,0.29799911,0.30022662,0.29765369,0.30050066,0.29653643,0.28125577,0.28673491,0.28011584,0.28001373,0.27470196,0.27291714,0.2683587,0.27389957,0.27603619,0.28654757,0.28303655,0.29148912,0.29385917,0.29388191,0.298745,0.29591522,0.29264843,0.29670148,0.29025426,0.29714065,0.29145111,0.29665654,0.29074885,0.28867811,0.2989535,0.29047084,0.29569017,0.29463981,0.28955532,0.29537107,0.29829579
VIPDEMO-075,0,sim_PLS_VIP_demo,1,6.4849,0.33228243,0.33971286,0.33020523,0.32933037,0.33422209,0.32987004,0.33033524,0.33098309,0.32154895,0.32182736,0.31913014,0.3184873,0.31432397,0.31316497,0.31316239,0.31236137,0.31984509,0.31915401,0.32312671,0.32539647,0.32545154,0.32070905,0.32049861,0.31330855,0.32317174,0.32435615,0.32471852,0.32470044,0.33527893,0.33222213,0.32905,0.32219018,0.32123094,0.32168302,0.31727572,0.31445269,0.30784578,0.31048929,0.31132551,0.303099,0.30334156,0.30440878,0.29950113,0.30225103,0.29911436,0.30200458,0.29837891,0.30761764,0.29407105,0.29572603,0.2909658,0.28650771,0.28559496,0.28239334,0.27554198,0.27837208,0.27513501,0.26725242,0.27664632,0.27954723,0.28263638,0.28026816,0.28676941,0.2867909,0.27786474,0.28240775,0.2806201,0.28147771,0.27171211,0.27185682,0.26174483,0.26614523,0.26112446,0.25658085,0.26252242,0.26449245,0.26528579,0.26723233,0.27290492,0.27056816,0.27060912
VIPDEMO-076,0,sim_PLS_VIP_demo,1,5.8209,0.27934542,0.28012479,0.28577514,0.28156509,0.28321883,0.28896348,0.27979073,0.28945187,0.29023911,0.29467777,0.29174051,0.2931292,0.29236744,0.29755863,0.29769473,0.28789191,0.29285304,0.2887149,0.28627483,0.2860957,0.29065478,0.28576221,0.28360054,0.29131465,0.29049351,0.29699001,0.29768786,0.29264222,0.29714285,0.29639012,0.29388711,0.29525977,0.29815039,0.30123889,0.29984906,0.29446026,0.29667276,0.29776918,0.29603533,0.29720672,0.29602984,0.29510707,0.29819402,0.29895304,0.29871706,0.29724611,0.29942276,0.2987767,0.29533849,0.2993778,0.29955977,0.29555847,0.29954773,0.30030036,0.30178806,0.29910822,0.3011001,0.29645287,0.29787502,0.30615897,0.29960833,0.30189336,0.30461516,0.30451556,0.3049903,0.3059415,0.31094787,0.30853137,0.31311032,0.31624418,0.31404792,0.31580331,0.31413121,0.31317743,0.3160874,0.31467493,0.31597877,0.30727276,0.31262198,0.31231422,0.31456097
VIPDEMO-077,0,sim_PLS_VIP_demo,1,5.2683,0.29649672,0.29616961,0.28915101,0.29128795,0.29541939,0.29369702,0.29673575,0.29432954,0.29829666,0.30108267,0.29655298,0.29460793,0.3034346,0.30151365,0.30118471,0.29302696,0.29100775,0.29915372,0.29352584,0.29610901,0.29392539,0.29743377,0.29192893,0.2879572,0.29410825,0.29464873,0.28480777,0.29262437,0.29032942,0.29014782,0.28821947,0.28475156,0.29044218,0.29134736,0.29697388,0.29136771,0.29238077,0.29439573,0.29700139,0.29618207,0.29468318,0.30224711,0.29522305,0.2962038,0.29444808,0.29838194,0.30038909,0.30629183,0.29636009,0.30641611,0.30237318,0.29931286,0.30471441,0.30602937,0.30622016,0.30769701,0.30726886,0.3057882,0.30202242,0.30658578,0.30232785,0.30557746,0.29908682,0.30321533,0.29776285,0.29848557,0.29880114,0.30490175,0.2951871,0.29756724,0.29186985,0.29995919,0.29840206,0.29549231,0.29496176,0.29867886,0.30065277,0.29966462,0.2961685,0.29802654,0.30574721
VIPDEMO-078,0,sim_PLS_VIP_demo,1,5.8613,0.28532549,0.28127297,0.27739173,0.28804973,0.28058628,0.28332505,0.29091505,0.28521924,0.29581255,0.29127904,0.29899178,0.3032764,0.2997595,0.30023909,0.30193825,0.29759805,0.29550291,0.29127184,0.2953304,0.2923224,0.29283391,0.29768827,0.29086844,0.29472205,0.29409409,0.29733196,0.29723894,0.30253107,0.30489459,0.30735144,0.30016558,0.30377534,0.30715106,0.30642936,0.30742834,0.30831782,0.30484435,0.30569474,0.30679312,0.30185757,0.29533666,0.30094944,0.30925683,0.30094388,0.30255459,0.3057303,0.30836614,0.30822465,0.31075307,0.31259874,0.31154091,0.31211639,0.30627589,0.31036961,0.31371553,0.31201792,0.31721435,0.31148414,0.31021388,0.31057011,0.315518,0.31227261,0.31537462,0.31956857,0.317297,0.3197911,0.32139204,0.3201638,0.32088226,0.31668909,0.32253961,0.32645016,0.32510535,0.32442872,0.32715385,0.32568966,0.32427032,0.32604146,0.3309883,0.32833735,0.3268776
VIPDEMO-079,0,sim_PLS_VIP_demo,1,5.3138,0.29551553,0.29534601,0.2953109,0.29295722,0.29106391,0.29328291,0.2945988,0.29666383,0.29981655,0.30167645,0.29405217,0.30127066,0.30032192,0.30216868,0.30057328,0.30173372,0.29784703,0.29685415,0.30050476,0.29673524,0.30138519,0.29980585,0.29516657,0.29939419,0.29213719,0.29288712,0.28943398,0.28523347,0.28679254,0.29139171,0.28725533,0.28338263,0.29270114,0.29018279,0.28638965,0.2935293,0.29362849,0.28827746,0.2949332,0.29144902,0.29736143,0.29520384,0.3004868,0.29856498,0.3028179,0.29853844,0.30286763,0.29648554,0.30235773,0.29821757,0.30308442,0.30868172,0.3057724,0.30789872,0.30688458,0.3034604,0.30997957,0.30979611,0.30855135,0.30681414,0.30373299,0.3008439,0.30528326,0.30278124,0.30299793,0.30365264,0.30107408,0.305851,0.30127239,0.30612757,0.31013493,0.31167592,0.30988355,0.3067023,0.30586955,0.30809851,0.30786365,0.30363794,0.29948341,0.29956826,0.29786631
"""

In [ ]:
# ---- Embedded demo spectra (fallback), Part 2: six edible oils --------
# Exact content of data/nb12_demo_spectra_oils.csv.
SAMPLE_CSV_OILS = """sample_id,group,instrument,rep,oil,900,915.942,931.884,947.826,963.768,979.71,995.652,1011.59,1027.54,1043.48,1059.42,1075.36,1091.3,1107.25,1123.19,1139.13,1155.07,1171.01,1186.96,1202.9,1218.84,1234.78,1250.72,1266.67,1282.61,1298.55,1314.49,1330.43,1346.38,1362.32,1378.26,1394.2,1410.14,1426.09,1442.03,1457.97,1473.91,1489.86,1505.8,1521.74,1537.68,1553.62,1569.57,1585.51,1601.45,1617.39,1633.33,1649.28,1665.22,1681.16,1697.1,1713.04,1728.99,1744.93,1760.87,1776.81,1792.75,1808.7,1824.64,1840.58,1856.52,1872.46,1888.41,1904.35,1920.29,1936.23,1952.17,1968.12,1984.06,2000
OIL-000,0,sim_PLSDA_oils_demo,1,EVOO,0.53811558,0.51862953,0.45375915,0.54334626,0.52792532,0.49477728,0.551089,0.5616631,0.58692276,0.61189127,0.6709332,0.65286662,0.69568392,0.68386536,0.70890342,0.65443071,0.60354751,0.5491895,0.54396101,0.53947677,0.52436512,0.59387662,0.59846052,0.47374473,0.53631285,0.63674114,0.65619018,0.70338003,0.71351821,0.78574386,0.63976282,0.64272806,0.70842485,0.62190218,0.59662353,0.53136591,0.4774155,0.55202701,0.56254605,0.53208831,0.58461141,0.6430969,0.63924922,0.69451971,0.7089705,0.69690235,0.65157289,0.66361532,0.64250703,0.5925724,0.53050317,0.58727709,0.53966537,0.59604046,0.51273371,0.58018811,0.52852304,0.46763697,0.49782157,0.50911809,0.51658626,0.46600695,0.46090916,0.51317018,0.48651415,0.51460402,0.53556455,0.43923227,0.51535929,0.55416964
OIL-001,0,sim_PLSDA_oils_demo,1,EVOO,0.46317161,0.52585514,0.50637664,0.53332068,0.48661505,0.44733601,0.51379718,0.51960042,0.59652374,0.60885302,0.57418405,0.6248716,0.72790522,0.72069407,0.64969318,0.64315321,0.62342205,0.58955534,0.57965172,0.53976384,0.52194944,0.52211348,0.59077459,0.48251039,0.59668051,0.63540536,0.60641718,0.69850019,0.66912743,0.72588144,0.6695733,0.67495173,0.66582106,0.60170458,0.52542548,0.48170676,0.60382721,0.53123007,0.52099622,0.57730601,0.59463736,0.67041767,0.66874975,0.68935383,0.66692132,0.70461495,0.62082322,0.65685431,0.65817119,0.50910437,0.45493654,0.57303949,0.65199222,0.51119038,0.49593071,0.55752152,0.48605005,0.48769924,0.48680619,0.51858742,0.47981585,0.50676312,0.48849609,0.46175329,0.48274442,0.45752115,0.4957977,0.45058245,0.45182132,0.55381557
OIL-002,0,sim_PLSDA_oils_demo,1,EVOO,0.49711325,0.51989198,0.48306964,0.49200908,0.52109476,0.52157052,0.47549088,0.57443238,0.54555853,0.56252969,0.60706415,0.66072145,0.76131081,0.65014717,0.68537109,0.56134733,0.60920157,0.61046697,0.53880064,0.50799536,0.53866815,0.56413862,0.57875632,0.65510865,0.61425856,0.6140541,0.66205711,0.68595285,0.7031933,0.69384767,0.68522076,0.58501674,0.65391762,0.56706297,0.51719312,0.57145469,0.59005002,0.55906802,0.55865033,0.53791206,0.72082165,0.67520746,0.62548597,0.66128347,0.70264882,0.62730813,0.67258824,0.61552101,0.64992965,0.61224313,0.44879944,0.55369758,0.48904275,0.59927803,0.55634791,0.55956412,0.48073906,0.58474261,0.58519896,0.45838163,0.51460569,0.47238085,0.49827976,0.44857979,0.57388843,0.46043509,0.48735889,0.51926155,0.47329981,0.48962974
OIL-003,0,sim_PLSDA_oils_demo,1,EVOO,0.48627406,0.44495198,0.47472756,0.48521795,0.48308294,0.50488465,0.50248183,0.5635703,0.54859126,0.59165003,0.60883507,0.64762583,0.63798225,0.71026317,0.62561049,0.60936921,0.6159772,0.58293041,0.52460575,0.44474209,0.53857141,0.53885825,0.48805794,0.59936575,0.57019103,0.58504614,0.66326867,0.67405714,0.69929194,0.62316203,0.74309449,0.62008756,0.55148925,0.60714939,0.54227047,0.55128215,0.51611581,0.52930045,0.55437804,0.53767792,0.6254517,0.61354428,0.62861344,0.68789345,0.70931447,0.67267736,0.6236854,0.65101846,0.52280884,0.52486555,0.55117493,0.48993027,0.54720438,0.54504115,0.57791643,0.49337945,0.49065095,0.51727583,0.39843665,0.61731733,0.46409022,0.46246271,0.52601082,0.48995517,0.42036863,0.51662253,0.52576052,0.47354755,0.48028139,0.51098479
OIL-004,0,sim_PLSDA_oils_demo,1,EVOO,0.50397093,0.49456852,0.46007267,0.43263248,0.48223064,0.46242305,0.54908522,0.53402816,0.58762537,0.59732379,0.64076526,0.63218936,0.71010675,0.7557226,0.65374661,0.61031538,0.59011684,0.54239916,0.50737406,0.52586336,0.50493707,0.58085355,0.53938414,0.57633224,0.63113879,0.61288633,0.7117318,0.65071108,0.65367676,0.66756417,0.66083949,0.58296407,0.60649473,0.51051913,0.60697106,0.52987082,0.49884473,0.49017943,0.5239928,0.55344375,0.55134458,0.59037137,0.65073682,0.65880369,0.72390423,0.72217824,0.65364994,0.63999391,0.53463735,0.58641971,0.54319357,0.55858227,0.60492175,0.45082426,0.54722766,0.4808292,0.53419798,0.44623892,0.4717461,0.49624789,0.51139952,0.48947077,0.45467538,0.46423004,0.52174878,0.48614835,0.41901572,0.5201148,0.50301824,0.52130492
OIL-005,0,sim_PLSDA_oils_demo,1,EVOO,0.52814394,0.45273628,0.51845226,0.47728364,0.52681123,0.546248,0.48817149,0.53479785,0.56648247,0.64810779,0.66725383,0.62332051,0.71027148,0.72272053,0.75966864,0.57544269,0.58355851,0.62356751,0.48981271,0.48093949,0.54583935,0.57261724,0.52121115,0.59356314,0.60631842,0.69424881,0.66278389,0.72421103,0.65846745,0.68340198,0.6701423,0.69316341,0.63966533,0.55573076,0.58714653,0.57251824,0.5286395,0.52478875,0.54020018,0.50314952,0.60925703,0.64512858,0.63218497,0.71791518,0.63999932,0.66317321,0.64258262,0.70913261,0.53265982,0.59201051,0.59068414,0.56288685,0.59676212,0.51053135,0.45416168,0.56404897,0.47125927,0.49429775,0.45299054,0.5388145,0.53004744,0.46625875,0.53111058,0.49980472,0.48937541,0.49724729,0.48684062,0.5195508,0.50736449,0.4809719
OIL-006,0,sim_PLSDA_oils_demo,1,EVOO,0.4907019,0.52679947,0.53268173,0.57619438,0.49976276,0.59588878,0.54480095,0.5492312,0.55816862,0.64411619,0.66093217,0.64809049,0.66699491,0.69083944,0.66781941,0.706364,0.68017738,0.62511829,0.57847353,0.53193163,0.54747864,0.58741243,0.65334051,0.62861727,0.61056697,0.6551631,0.66368608,0.67344845,0.70736573,0.71876497,0.76758979,0.66982364,0.69091838,0.6766437,0.58316338,0.6261482,0.581846,0.53853708,0.57870276,0.59200758,0.61255699,0.64793747,0.69271773,0.72572783,0.67966598,0.70501346,0.62084043,0.66020729,0.64171441,0.59620674,0.58457658,0.5915188,0.54302901,0.56122215,0.58595216,0.59359493,0.5550325,0.62961137,0.5166332,0.55685049,0.56625379,0.50985375,0.48229364,0.46779276,0.52158529,0.55950555,0.52596058,0.52200808,0.49992528,0.53753503
OIL-007,0,sim_PLSDA_oils_demo,1,EVOO,0.47732456,0.46931587,0.41385061,0.55693471,0.41735258,0.43593336,0.44258852,0.55427369,0.50241283,0.6002794,0.63534386,0.65549037,0.61267594,0.64144465,0.71465295,0.56399042,0.54475835,0.53675475,0.54938707,0.51199542,0.52784977,0.46220107,0.55876138,0.56773115,0.51086957,0.66810745,0.72755249,0.62686999,0.66983022,0.71970599,0.58777281,0.54101662,0.53761499,0.53613016,0.50974207,0.53899883,0.51685873,0.4582641,0.54365945,0.61892442,0.62260142,0.64840171,0.64062517,0.70057454,0.62928464,0.68829825,0.63111339,0.64788039,0.58827955,0.49892205,0.53113087,0.56951863,0.47733239,0.50128721,0.48763705,0.44656236,0.53055508,0.53280559,0.50515805,0.47943559,0.46637212,0.47764846,0.43872062,0.5037194,0.5097826,0.50526431,0.44672461,0.47109871,0.4613313,0.53950553
OIL-008,0,sim_PLSDA_oils_demo,1,EVOO,0.4274797,0.51886396,0.57868777,0.53205073,0.54276121,0.51520954,0.44650038,0.47218158,0.52283122,0.60319277,0.6589763,0.70741303,0.68609368,0.73893899,0.67127131,0.62217393,0.64682312,0.54139972,0.44259698,0.49426449,0.5366673,0.41641864,0.5414177,0.56641929,0.55232683,0.58194357,0.652091,0.67046199,0.75273376,0.71353652,0.61852111,0.62115789,0.65243132,0.66938482,0.58307265,0.56186635,0.61528028,0.54193324,0.54333312,0.52216412,0.58893152,0.73063883,0.61768131,0.69640986,0.64632976,0.69812961,0.70484212,0.6276311,0.63358593,0.60608391,0.57862505,0.6241726,0.58838816,0.54618733,0.52588394,0.50730885,0.54587195,0.50214973,0.61705591,0.57816359,0.49460533,0.4896409,0.57032282,0.42736409,0.48463377,0.54072576,0.46621154,0.48362224,0.4886498,0.52912778
OIL-009,0,sim_PLSDA_oils_demo,1,EVOO,0.51821349,0.52610254,0.50396211,0.54843381,0.47779113,0.45568969,0.45364854,0.47911896,0.55001597,0.64883539,0.65291049,0.65526206,0.65628524,0.67887385,0.66568206,0.62711699,0.55482314,0.55831266,0.59181329,0.44904587,0.52264403,0.45413928,0.56239259,0.54917079,0.52532502,0.64463925,0.74265316,0.69606912,0.74817614,0.69504928,0.59481916,0.67433074,0.60686195,0.53442981,0.5983736,0.56027961,0.51920126,0.57693034,0.52018174,0.60379883,0.5712012,0.66421597,0.62988851,0.75489718,0.71220262,0.74967304,0.63580586,0.61770044,0.63732609,0.46686893,0.51743989,0.59621722,0.57520649,0.59690395,0.54009391,0.56304854,0.49697094,0.45958765,0.43946489,0.48176476,0.48059609,0.47699852,0.36013102,0.4221218,0.48075797,0.51354361,0.4789112,0.46698702,0.40972397,0.58292286
OIL-010,0,sim_PLSDA_oils_demo,1,EVOO,0.55088629,0.54613905,0.49514217,0.46252722,0.49365853,0.4484333,0.51657305,0.52549786,0.57473575,0.62707142,0.69667779,0.64509172,0.72223543,0.64978292,0.62746655,0.60641914,0.55426672,0.59797598,0.56947791,0.56834312,0.51267679,0.5313889,0.51335403,0.51445234,0.60805368,0.63276652,0.60709684,0.69654718,0.61345581,0.67216994,0.63733066,0.61496078,0.56167524,0.6124334,0.57103849,0.57756255,0.52685797,0.47170584,0.52132499,0.51182125,0.60811186,0.67222645,0.66798247,0.65224924,0.66811766,0.68139646,0.59170983,0.62354905,0.59249984,0.53562833,0.52417054,0.54246436,0.50449099,0.50555509,0.5643036,0.53004382,0.57397169,0.45556737,0.41663848,0.52200473,0.52133995,0.45310671,0.42378937,0.48059874,0.41236749,0.46872478,0.39504685,0.4727655,0.39742932,0.47272975
OIL-011,0,sim_PLSDA_oils_demo,1,EVOO,0.51030695,0.5519055,0.49588628,0.48707661,0.46575111,0.57343166,0.54104899,0.62745684,0.54344862,0.61723459,0.66759606,0.75413556,0.79036161,0.69245031,0.65482213,0.59857191,0.68387861,0.59210997,0.54343353,0.55217739,0.4481168,0.48813266,0.61062754,0.61303643,0.63288047,0.61866846,0.71157551,0.78095443,0.67081851,0.71658769,0.68180021,0.68722064,0.64222088,0.60509317,0.54975228,0.56622666,0.53369077,0.55439646,0.57056884,0.62227812,0.60357169,0.64796574,0.73484187,0.70464133,0.70959651,0.74497361,0.69764951,0.65843901,0.63059208,0.54510157,0.59664443,0.62604683,0.59800354,0.51371086,0.58209952,0.52258434,0.5354759,0.53802722,0.47798024,0.57572213,0.51829205,0.55239463,0.54602718,0.57440497,0.56833877,0.4617413,0.52246011,0.43537145,0.50291659,0.54940458
OIL-012,0,sim_PLSDA_oils_demo,1,EVOO,0.56339458,0.49693163,0.52531469,0.51766966,0.5674085,0.5864581,0.52017363,0.70513629,0.58170651,0.60278953,0.67976897,0.71940317,0.73081558,0.68395376,0.7506923,0.66186547,0.68365527,0.643119,0.51028518,0.50696925,0.5024106,0.55918786,0.5674668,0.60008492,0.58655564,0.66800861,0.73647267,0.64868121,0.70426421,0.71826123,0.68635231,0.65725785,0.70479523,0.65470745,0.61142748,0.52882887,0.64035346,0.56715522,0.54245713,0.57507799,0.59975802,0.73823079,0.69267745,0.70841398,0.62647738,0.73242199,0.6369554,0.60213782,0.63883335,0.55267282,0.60413646,0.61035741,0.49561786,0.56290032,0.50012317,0.54860275,0.57328418,0.47050452,0.58993463,0.50195904,0.46576258,0.50771246,0.54426576,0.45746015,0.51260418,0.45758199,0.5606763,0.51868937,0.49546706,0.55387391
OIL-013,0,sim_PLSDA_oils_demo,1,EVOO,0.46400045,0.55372986,0.47537986,0.53585153,0.49231685,0.46268748,0.49241249,0.52457464,0.53317398,0.55218337,0.59218608,0.59736314,0.68098152,0.69325044,0.68925864,0.55495545,0.62181755,0.55411866,0.47144113,0.51740156,0.5423101,0.53734797,0.5615417,0.59434167,0.62204874,0.61665109,0.56412564,0.63880212,0.69342625,0.65661247,0.68603601,0.66412206,0.60302528,0.64561862,0.50549674,0.56358557,0.49885213,0.48852684,0.55615516,0.53097491,0.62072911,0.57374396,0.65287971,0.63915038,0.67018912,0.65503951,0.60112285,0.64053365,0.53150768,0.51499196,0.57335949,0.57141768,0.56091785,0.54897273,0.62123317,0.46973028,0.47410312,0.45235289,0.52085297,0.49301147,0.52465111,0.51874203,0.47075615,0.446988,0.4400632,0.56822782,0.51847377,0.50350211,0.46622761,0.4718649
OIL-014,0,sim_PLSDA_oils_demo,1,EVOO,0.48112932,0.39810193,0.49226732,0.48067653,0.51049845,0.51347885,0.55082285,0.53540409,0.59588458,0.49716651,0.59386745,0.74021235,0.64228942,0.6808478,0.70208357,0.6064029,0.63301061,0.51220543,0.46657831,0.47484285,0.45810439,0.53026955,0.53434383,0.58419218,0.55930746,0.73431218,0.65812282,0.68186747,0.7119216,0.68015618,0.64785267,0.65013406,0.65233494,0.61153428,0.59665406,0.51824479,0.50123629,0.61440496,0.53753975,0.58924,0.63040721,0.69174859,0.61194001,0.66819759,0.6767693,0.67238552,0.564285,0.62499141,0.48008328,0.58065206,0.57373614,0.57093775,0.50054956,0.5881058,0.51007877,0.4835662,0.47916227,0.47273525,0.50634778,0.52902429,0.41070222,0.47275325,0.5303928,0.50246445,0.51468492,0.43442682,0.53256312,0.43185585,0.50221648,0.54906391
OIL-015,0,sim_PLSDA_oils_demo,1,ROO,0.54871881,0.56365731,0.50463867,0.51088075,0.57985377,0.53932616,0.54575821,0.51366438,0.60418759,0.6595954,0.6403044,0.68504809,0.78842508,0.73328728,0.69641232,0.63036901,0.61346911,0.59347248,0.63202031,0.53773779,0.63016916,0.52797305,0.56689457,0.63893947,0.62527208,0.63352804,0.69081056,0.73578931,0.65920246,0.71409535,0.70287668,0.66468885,0.67913472,0.59629847,0.57852689,0.57643836,0.59549896,0.55948841,0.64185587,0.56788016,0.68640755,0.63467227,0.69352303,0.73601805,0.81034514,0.75428026,0.67840324,0.71404131,0.61865923,0.56386083,0.58822958,0.55616333,0.57912299,0.51241335,0.54622579,0.55471218,0.53974257,0.52718038,0.50033324,0.56435433,0.53368301,0.54804335,0.51415934,0.52871077,0.48670114,0.53161127,0.55879867,0.53190999,0.56256597,0.54070668
OIL-016,0,sim_PLSDA_oils_demo,1,ROO,0.51292296,0.45514093,0.53845906,0.51579853,0.48637269,0.55758011,0.55369153,0.58016784,0.5771365,0.63391035,0.67451233,0.76858508,0.69011602,0.71800445,0.71477805,0.59912444,0.56640113,0.68237266,0.49883803,0.57742234,0.51673431,0.52686175,0.49416463,0.56143577,0.65852262,0.6034564,0.71216896,0.72264031,0.80083151,0.69732306,0.7500543,0.70866111,0.63052133,0.54478991,0.64575269,0.5126247,0.53967595,0.59592957,0.61809779,0.56263861,0.6259345,0.65090958,0.73957563,0.81390913,0.64722366,0.74737172,0.57257833,0.65490679,0.57512364,0.66745901,0.56920453,0.51518122,0.51327141,0.50365864,0.53076571,0.51527187,0.54712376,0.58221776,0.54856205,0.47143173,0.45550936,0.49024201,0.56258667,0.50705597,0.49375495,0.52598915,0.46383681,0.47488474,0.54087087,0.50833171
OIL-017,0,sim_PLSDA_oils_demo,1,ROO,0.50597395,0.4700349,0.4664515,0.47675731,0.52219286,0.5066659,0.46807806,0.52297562,0.58910411,0.67360215,0.59138822,0.68388389,0.67986808,0.72418861,0.69094947,0.6997466,0.6240415,0.546679,0.52072267,0.54339078,0.53920466,0.62604926,0.61326255,0.52326579,0.61390007,0.57851016,0.59486842,0.71259385,0.71491293,0.6475053,0.73800881,0.60674716,0.56802254,0.61015649,0.56695623,0.53352354,0.52867357,0.54229815,0.54061998,0.56229989,0.52411991,0.58031787,0.68949487,0.65829537,0.71497047,0.73287697,0.64545092,0.63840653,0.67527769,0.58832729,0.60896835,0.57873198,0.53612195,0.49493833,0.46229711,0.44987349,0.5130851,0.51207826,0.48626428,0.52715412,0.46902375,0.53761116,0.49480195,0.49174844,0.49319053,0.48925781,0.59273203,0.42874066,0.50029036,0.5429657
OIL-018,0,sim_PLSDA_oils_demo,1,ROO,0.49874651,0.58059582,0.46872466,0.46045911,0.5383721,0.46365555,0.43830522,0.58797461,0.6111904,0.64562478,0.63514477,0.69522156,0.66916619,0.77810312,0.72216948,0.688111,0.54282532,0.61833685,0.61958651,0.5933805,0.57253338,0.54765472,0.55009716,0.61183583,0.63276309,0.65876594,0.66628774,0.76766351,0.66369054,0.70570395,0.66661535,0.70469292,0.68951218,0.65137655,0.60589621,0.50140102,0.63216675,0.5715752,0.55942589,0.58762135,0.60393865,0.66851279,0.69502484,0.68962464,0.60792833,0.6996148,0.69628363,0.6656957,0.60364127,0.5962237,0.59398739,0.51614692,0.58219994,0.43876899,0.46309615,0.47569685,0.41341452,0.47092166,0.56074529,0.56043786,0.51466081,0.57960371,0.52034252,0.49085026,0.52042794,0.50527683,0.54020003,0.45064067,0.55073929,0.49238163
OIL-019,0,sim_PLSDA_oils_demo,1,ROO,0.45125533,0.49628779,0.51665405,0.55083621,0.46901518,0.59059926,0.5547969,0.53369527,0.63149562,0.6225089,0.70304264,0.78289678,0.74058754,0.68972904,0.67556736,0.68379008,0.63440807,0.55479072,0.5351704,0.57342845,0.63690325,0.44454546,0.59479427,0.58139542,0.65069406,0.68608696,0.68283741,0.66184146,0.65658934,0.63370925,0.74078356,0.65709025,0.61274907,0.59257137,0.52416836,0.57175683,0.60373265,0.59851293,0.5031996,0.64336895,0.58901709,0.57710663,0.63550639,0.66743523,0.71251322,0.66180778,0.61886752,0.63613637,0.68880317,0.55206422,0.57349192,0.54022196,0.63193863,0.44311214,0.55942051,0.4627144,0.48956372,0.50911205,0.64032101,0.48232364,0.43727826,0.56472533,0.47926061,0.50311188,0.51186818,0.5167094,0.5060243,0.5235401,0.52832423,0.54698676
OIL-020,0,sim_PLSDA_oils_demo,1,ROO,0.46370939,0.45127778,0.51800154,0.49368784,0.52060626,0.526191,0.58368709,0.54378815,0.59366005,0.55105852,0.59426036,0.65826302,0.64193935,0.7191416,0.6200352,0.64398489,0.50532969,0.52468239,0.52909503,0.51719724,0.45213115,0.53486858,0.58559385,0.59652858,0.5739378,0.60864789,0.61403353,0.70843357,0.75089943,0.66839983,0.60473867,0.65424017,0.6434325,0.47555685,0.56404297,0.62690939,0.56457215,0.55575451,0.56267807,0.61206609,0.56270153,0.60057286,0.68174971,0.65944428,0.70390089,0.60734596,0.69035718,0.66135665,0.59435579,0.56749683,0.60050948,0.47446196,0.46985131,0.45605188,0.47937769,0.45428546,0.49550393,0.4677115,0.43567325,0.4315984,0.54206157,0.49083729,0.4844982,0.49465433,0.49029703,0.52383602,0.51608186,0.47000319,0.46065042,0.45012267
OIL-021,0,sim_PLSDA_oils_demo,1,ROO,0.53738595,0.44479975,0.40117288,0.44944191,0.45458514,0.50549874,0.46845424,0.5445196,0.5667478,0.63747682,0.71179367,0.77233022,0.73228297,0.63281139,0.64483087,0.63879694,0.67402014,0.58357992,0.57843286,0.50824228,0.53504988,0.54689103,0.51598847,0.58933361,0.63075508,0.61344666,0.68985293,0.73680177,0.6835266,0.67709247,0.64790823,0.60910915,0.6090093,0.55561026,0.53367103,0.49703578,0.45794585,0.48388793,0.54739465,0.50990613,0.56207079,0.68771431,0.6981491,0.64676234,0.70532949,0.64547855,0.71278409,0.59169157,0.58351887,0.60395379,0.4902099,0.55760864,0.46132105,0.49697348,0.47218384,0.47883278,0.48806713,0.47840872,0.44058741,0.51591023,0.51005838,0.53282743,0.53663117,0.42747436,0.53265349,0.53314242,0.4599433,0.52272921,0.53996228,0.45967961
OIL-022,0,sim_PLSDA_oils_demo,1,ROO,0.46500424,0.59125518,0.48947071,0.43269849,0.4740873,0.45062817,0.4860079,0.52502082,0.44682842,0.59930525,0.67978631,0.64856488,0.66102737,0.7210286,0.60841888,0.57714107,0.53375918,0.54830556,0.54414463,0.50966301,0.53151513,0.54947881,0.5521898,0.52397992,0.57528472,0.66779676,0.60534518,0.65883221,0.66226923,0.66796161,0.61535176,0.57799377,0.70308739,0.60800496,0.51573115,0.52370242,0.54454169,0.50571683,0.55698127,0.53070585,0.57173416,0.61672273,0.63287423,0.57447166,0.68012598,0.68004045,0.66481396,0.5734455,0.59210568,0.55527504,0.45065783,0.41543544,0.51116351,0.44736382,0.52218226,0.56607342,0.45644002,0.45189229,0.47889848,0.50632248,0.43389042,0.52913841,0.51896769,0.50576291,0.45772498,0.5159707,0.51360712,0.43127977,0.56292598,0.46729752
OIL-023,0,sim_PLSDA_oils_demo,1,ROO,0.48463866,0.52116195,0.55264277,0.4751913,0.44420733,0.49861002,0.47595646,0.51709132,0.57045216,0.60854211,0.70104787,0.64208688,0.71666268,0.68320146,0.73990588,0.75430379,0.62773739,0.6477875,0.5333945,0.47552885,0.48492732,0.4793045,0.5203556,0.5729365,0.65823155,0.67208042,0.69198238,0.6782389,0.67630287,0.70271493,0.69010435,0.66843474,0.64192993,0.59168353,0.49346877,0.50767063,0.51066796,0.61355176,0.54144861,0.60853779,0.67086405,0.65966721,0.66168156,0.6870749,0.66460337,0.7080563,0.71710808,0.66221653,0.63540765,0.60157344,0.62779065,0.5043795,0.52462482,0.53996946,0.55820384,0.40957205,0.48507461,0.49734751,0.46440869,0.53943717,0.51430115,0.48968074,0.44017318,0.45258694,0.46612441,0.52373027,0.45676735,0.42276068,0.44743709,0.4818593
OIL-024,0,sim_PLSDA_oils_demo,1,ROO,0.62524345,0.50607227,0.56355515,0.58309583,0.58136739,0.53149935,0.49567647,0.5474634,0.52059353,0.64021517,0.70007981,0.62652054,0.72004292,0.73475973,0.75483778,0.72790212,0.6672121,0.57291184,0.55943016,0.60723816,0.53106834,0.56033953,0.64053934,0.60722186,0.57680814,0.65182381,0.71856605,0.70713645,0.77119319,0.77222607,0.6781263,0.65760129,0.64245709,0.56787888,0.60512909,0.56463245,0.59466202,0.54123966,0.54291019,0.67460911,0.59668281,0.6424691,0.66765507,0.70618013,0.68841857,0.71355664,0.66335157,0.64231042,0.63862902,0.62049078,0.58586986,0.51985071,0.55377179,0.52649324,0.52362152,0.45734432,0.55238231,0.51966439,0.59339755,0.55071721,0.48044222,0.47021233,0.57113849,0.49010433,0.55247879,0.5130683,0.60925252,0.52830687,0.58573006,0.57846502
OIL-025,0,sim_PLSDA_oils_demo,1,ROO,0.48010918,0.55289293,0.49406896,0.46060525,0.54651703,0.60754966,0.51480469,0.5917171,0.55746365,0.65390083,0.62850631,0.72477036,0.71321528,0.68517631,0.68388481,0.6207136,0.55984258,0.52794067,0.54180467,0.48675911,0.54302818,0.5535045,0.56724183,0.55261383,0.60450784,0.63313475,0.71647236,0.65306027,0.67997911,0.68225331,0.66912313,0.61846582,0.59219106,0.62733915,0.53523077,0.57931576,0.61547446,0.58719877,0.47656809,0.60286964,0.62182418,0.65850404,0.60269756,0.71955886,0.65228061,0.7379234,0.70518888,0.73171448,0.57933192,0.50206149,0.57974746,0.55484347,0.49312582,0.55629282,0.5259681,0.44521406,0.4819586,0.47788564,0.49123333,0.50326523,0.46772856,0.48765233,0.51913814,0.49258871,0.48471484,0.48333362,0.49150472,0.41887931,0.50192579,0.45435633
OIL-026,0,sim_PLSDA_oils_demo,1,ROO,0.44608913,0.54203647,0.51509748,0.45280983,0.51929462,0.48560072,0.59964492,0.56647184,0.54304201,0.61952931,0.64755127,0.73006523,0.7163221,0.72356953,0.64009914,0.6844605,0.69895555,0.60591691,0.55264555,0.54569545,0.49726399,0.54956355,0.51426643,0.58397771,0.59637625,0.61053277,0.69633993,0.63529593,0.70326936,0.64808578,0.75753012,0.73692078,0.66066235,0.67201904,0.59391575,0.57370334,0.52639299,0.50956753,0.56334973,0.55980504,0.59445777,0.67516924,0.67195172,0.63602033,0.69646786,0.6648694,0.65979536,0.59509473,0.65213434,0.64854695,0.50272203,0.55522622,0.5417863,0.46830781,0.51188659,0.56264401,0.53935268,0.56994199,0.45822779,0.47511682,0.51659124,0.51907406,0.5058378,0.48934636,0.48836513,0.48043932,0.52275069,0.39428643,0.56863259,0.48349718
OIL-027,0,sim_PLSDA_oils_demo,1,ROO,0.50320474,0.42534052,0.49743203,0.46712493,0.50972141,0.5238122,0.55557388,0.50316561,0.53937191,0.58608626,0.56826439,0.6818072,0.683086,0.62523458,0.70905336,0.68314584,0.53821209,0.57898438,0.57989278,0.5483481,0.47172689,0.46177573,0.52910208,0.53785273,0.58703897,0.61507823,0.66961538,0.64782892,0.6737491,0.7951302,0.65440374,0.66262055,0.61355373,0.58003093,0.50887293,0.54823502,0.51648016,0.55321265,0.61539938,0.56371436,0.61752996,0.58015699,0.65681995,0.73560236,0.67783126,0.64503343,0.65495277,0.60674001,0.56962768,0.53265005,0.60177968,0.46957062,0.47648153,0.46360412,0.53258096,0.47018209,0.53279663,0.48699937,0.479299,0.42176661,0.52863814,0.50181541,0.49637484,0.4754741,0.46350237,0.4294456,0.51799831,0.4909259,0.46580342,0.45495398
OIL-028,0,sim_PLSDA_oils_demo,1,ROO,0.49933913,0.56885119,0.5345675,0.48101767,0.46357729,0.50455186,0.48911719,0.54786463,0.58907446,0.63397454,0.66350317,0.67252474,0.72406283,0.66545495,0.67971136,0.68326123,0.62470872,0.52758357,0.50603074,0.47255939,0.60558634,0.47818043,0.550745,0.6537558,0.65308068,0.69455929,0.65127445,0.72256727,0.82482741,0.69776793,0.69899066,0.64344317,0.62442042,0.60922983,0.62383076,0.51418192,0.48447772,0.54464702,0.56470142,0.63594609,0.6389659,0.61932909,0.69804025,0.74143015,0.71493027,0.68018516,0.69703398,0.62320528,0.5278345,0.62405809,0.49397626,0.51425616,0.5035713,0.49620389,0.49476176,0.44687963,0.52478409,0.50454096,0.50488794,0.54613284,0.54720987,0.55676042,0.5772121,0.52130951,0.44164986,0.51168769,0.50369103,0.50593313,0.47915906,0.46761954
OIL-029,0,sim_PLSDA_oils_demo,1,ROO,0.55234613,0.46899768,0.5357667,0.50866953,0.50285519,0.55192037,0.534081,0.53318552,0.58564503,0.53912148,0.63357462,0.67290484,0.65440649,0.71831742,0.70374193,0.63672827,0.61635723,0.61526664,0.6044769,0.51654805,0.55671342,0.59132585,0.56466853,0.55896845,0.58880225,0.63839812,0.63927207,0.78281813,0.6894627,0.72356022,0.69894352,0.6906533,0.61861041,0.55135598,0.64287083,0.55600653,0.55519956,0.56052396,0.61583379,0.61912624,0.66608419,0.68607308,0.6903742,0.71584159,0.68265558,0.62513559,0.66930166,0.60339791,0.71492765,0.64211017,0.60134426,0.51459187,0.60756366,0.45047219,0.4668647,0.48233193,0.47453727,0.51787724,0.52022889,0.58884076,0.55232624,0.56126366,0.52916624,0.50653859,0.48497202,0.45206265,0.47763922,0.4971938,0.54294099,0.44497025
OIL-030,0,sim_PLSDA_oils_demo,1,SOY,0.48846583,0.43795354,0.49398975,0.46584989,0.4563591,0.44176683,0.50755491,0.50909469,0.55303109,0.60189203,0.68846974,0.73349592,0.68683564,0.75026264,0.74715076,0.72782907,0.6649472,0.59330596,0.50252224,0.49628596,0.55839892,0.50314509,0.59479078,0.63829846,0.58640774,0.69112146,0.71009803,0.82614823,0.79254935,0.7332961,0.74650549,0.65652986,0.64361231,0.61491931,0.57638828,0.55727941,0.49666833,0.46160064,0.53921512,0.55023311,0.60098135,0.76213557,0.76664244,0.74312359,0.70213769,0.69396122,0.72700364,0.70974406,0.60661279,0.57919351,0.54865384,0.55607263,0.42233774,0.49501708,0.50158932,0.52474526,0.50910111,0.46063083,0.49482836,0.46680296,0.53640217,0.52791799,0.52738745,0.45571031,0.44057547,0.5429568,0.48752198,0.54953767,0.49713968,0.42106616
OIL-031,0,sim_PLSDA_oils_demo,1,SOY,0.56158039,0.49437192,0.43607726,0.54141847,0.51330836,0.53868528,0.43634549,0.58902128,0.50002379,0.62518386,0.7489506,0.6870996,0.71335233,0.77124301,0.82961979,0.78342863,0.64183677,0.65745706,0.60392881,0.57007477,0.56854444,0.53473933,0.61829391,0.66229043,0.76944316,0.74265666,0.64894412,0.72930795,0.80478364,0.78886919,0.68818871,0.72003049,0.66771492,0.63710053,0.59787475,0.6141492,0.62387805,0.59542928,0.54167179,0.58375226,0.63486286,0.70675627,0.73543027,0.76847738,0.75495258,0.81623784,0.77899501,0.73753812,0.64733086,0.62437661,0.53441337,0.50534993,0.50820494,0.54457831,0.51182366,0.47347517,0.52940656,0.55400476,0.55701559,0.5335606,0.56370934,0.53581974,0.52635451,0.52338941,0.54879151,0.58741636,0.57505641,0.53731921,0.49605049,0.49968531
OIL-032,0,sim_PLSDA_oils_demo,1,SOY,0.54213029,0.38570156,0.50728915,0.41902272,0.50268113,0.56447684,0.49128181,0.46333504,0.64584273,0.70402648,0.7173711,0.79420289,0.73971832,0.82005225,0.81569184,0.72832318,0.66132412,0.65272117,0.60996354,0.51359588,0.59268346,0.57748195,0.57655342,0.65595394,0.6823783,0.71979968,0.71073246,0.79484449,0.73115236,0.76512194,0.67894812,0.64152696,0.68385928,0.61990489,0.59026903,0.5851121,0.56341072,0.51674188,0.57313441,0.59971001,0.62465532,0.68812561,0.7219757,0.7406027,0.76257011,0.77175376,0.81155024,0.65075624,0.65549551,0.5318018,0.61852803,0.51655531,0.51957704,0.55634118,0.54134059,0.44353263,0.53192873,0.51202844,0.55928087,0.55033106,0.57458106,0.54836724,0.51778544,0.55804977,0.44088826,0.48107588,0.45847616,0.4904081,0.47882963,0.49007795
OIL-033,0,sim_PLSDA_oils_demo,1,SOY,0.47395344,0.42720096,0.492119,0.50770319,0.47563052,0.48186623,0.49858508,0.61846009,0.59812731,0.65595187,0.68206049,0.72064306,0.73332343,0.83148146,0.67629836,0.71136353,0.64576858,0.58781344,0.49867804,0.49079666,0.57419768,0.51659654,0.60201823,0.62724302,0.66518424,0.71707065,0.67928442,0.73919181,0.79928566,0.71633411,0.80991128,0.68396579,0.66021328,0.57882881,0.5981599,0.52873051,0.53635997,0.50329702,0.6145161,0.53373644,0.6601476,0.69923905,0.750753,0.72238966,0.81393886,0.79757865,0.73205318,0.6925907,0.62497314,0.61262566,0.51872776,0.54873327,0.52692377,0.44549197,0.39352018,0.4951665,0.4999073,0.56649486,0.5334685,0.52097248,0.48739625,0.52980716,0.49667479,0.46351052,0.49448215,0.53732647,0.49705932,0.45084915,0.50576695,0.44872585
OIL-034,0,sim_PLSDA_oils_demo,1,SOY,0.49619894,0.4546404,0.52186003,0.4915033,0.45091923,0.50060735,0.55988015,0.52744131,0.53185278,0.59767538,0.64524568,0.67709275,0.70045863,0.77038294,0.77238521,0.66858078,0.66188494,0.61805761,0.61238419,0.51643516,0.50464584,0.54488711,0.5544488,0.59622677,0.67756264,0.65313157,0.72004801,0.70259604,0.74945557,0.72358764,0.69671698,0.68125415,0.62861501,0.61838831,0.59937238,0.4820797,0.56988582,0.44348308,0.56043534,0.61114468,0.59976572,0.66428367,0.68566157,0.7826972,0.62939393,0.70384634,0.63188162,0.6713115,0.66692766,0.61825814,0.56199318,0.52015948,0.46371965,0.53432104,0.50710618,0.44977488,0.43887853,0.4763172,0.55812856,0.5746102,0.59011142,0.48601222,0.48932545,0.54661888,0.49179243,0.50930723,0.42420234,0.46172739,0.47573588,0.46218332
OIL-035,0,sim_PLSDA_oils_demo,1,SOY,0.42859437,0.55935781,0.50073581,0.43838937,0.54386539,0.51219171,0.55447164,0.54508258,0.5608386,0.61169889,0.70946858,0.77331914,0.78978216,0.79737923,0.79426281,0.71274046,0.68884539,0.64436799,0.57514989,0.53934811,0.51980604,0.49855518,0.50695804,0.65542841,0.64399797,0.66723449,0.79341774,0.79967889,0.77725433,0.80983776,0.73856421,0.72364682,0.7386218,0.66524634,0.57570467,0.58901483,0.59658041,0.54746707,0.55354449,0.67756939,0.63388159,0.64071755,0.7428767,0.85887336,0.80214146,0.80191447,0.64379982,0.66308778,0.68489179,0.57634575,0.60999893,0.55151469,0.53257798,0.57526775,0.48457516,0.487437,0.51701575,0.53297262,0.46973947,0.59656523,0.55745202,0.53323733,0.55143284,0.49834695,0.5541929,0.50425432,0.52881753,0.46398023,0.48816439,0.51410106
OIL-036,0,sim_PLSDA_oils_demo,1,SOY,0.4668874,0.49136714,0.53858489,0.52624091,0.53604742,0.52065359,0.56078742,0.6132423,0.59762802,0.65812676,0.71767554,0.66079569,0.76466908,0.78665874,0.77458279,0.68659918,0.68577644,0.57243815,0.55424033,0.60742148,0.48077494,0.54096403,0.5606372,0.59644206,0.62019018,0.71202446,0.68425865,0.70460064,0.79052511,0.71825756,0.75731485,0.70613727,0.63422451,0.60482393,0.57826791,0.57463987,0.51931292,0.53888095,0.5079279,0.58631311,0.5635659,0.68595749,0.71296885,0.84469923,0.82167413,0.75064157,0.76111615,0.68887882,0.5827618,0.48727237,0.59307072,0.53920417,0.42953258,0.52429851,0.53187689,0.4693584,0.53471395,0.44572777,0.51908085,0.53891926,0.56964657,0.54582017,0.52544038,0.52632786,0.46961471,0.55059596,0.47614708,0.52094218,0.48888582,0.47914672
OIL-037,0,sim_PLSDA_oils_demo,1,SOY,0.52825118,0.56432798,0.55422158,0.49032041,0.53716421,0.52987234,0.51019474,0.52493159,0.61841613,0.63666569,0.65622774,0.75444426,0.7948821,0.79133409,0.71368555,0.69370769,0.718969,0.58474141,0.56573073,0.649808,0.56627569,0.54165382,0.50645084,0.63010948,0.6037311,0.70283329,0.71023168,0.70838379,0.85535237,0.77965319,0.79712161,0.7529056,0.6505404,0.64032646,0.58616024,0.61353047,0.58013777,0.55150494,0.64999943,0.63658731,0.67735696,0.67521362,0.76629034,0.72137703,0.80780563,0.73831929,0.74972002,0.68569367,0.65709917,0.61788294,0.56592515,0.54092776,0.46775786,0.44253566,0.56550499,0.53937427,0.46376296,0.61070021,0.51914802,0.57788211,0.57535665,0.55423918,0.56092633,0.46922171,0.51441242,0.50144402,0.51341058,0.4984038,0.48355273,0.47996118
OIL-038,0,sim_PLSDA_oils_demo,1,SOY,0.479444,0.52808815,0.52824775,0.51208172,0.53704615,0.55045394,0.52370864,0.58537177,0.68745543,0.61823262,0.70260706,0.73059038,0.86849377,0.73431156,0.76522562,0.757157,0.63262897,0.61322454,0.55348229,0.5072072,0.55271614,0.58470037,0.57154437,0.62760532,0.68992751,0.7549332,0.69893935,0.80323107,0.76791821,0.71209172,0.73283471,0.75257486,0.69566281,0.59659722,0.66832235,0.59478761,0.56392099,0.54779429,0.60413207,0.60415792,0.70802437,0.72288726,0.7784132,0.7716108,0.73421657,0.80066942,0.70600931,0.69646309,0.65132746,0.57636735,0.60766756,0.49018631,0.5907598,0.49896015,0.55752112,0.57201927,0.49262217,0.50907123,0.58924283,0.56894557,0.56569627,0.48108717,0.47523337,0.54963707,0.57705491,0.48744298,0.47953637,0.52176654,0.53177804,0.53566075
OIL-039,0,sim_PLSDA_oils_demo,1,SOY,0.41612721,0.55176112,0.47466041,0.54613652,0.42111807,0.52369372,0.50527887,0.50893939,0.6299854,0.69256185,0.67156838,0.7671392,0.72655246,0.7460151,0.73535779,0.68635121,0.67288813,0.5508921,0.60058433,0.57937809,0.52707289,0.49810827,0.47322323,0.59873815,0.61436331,0.69146441,0.68659014,0.75904689,0.77048614,0.76806184,0.75544524,0.71871596,0.70204912,0.58957088,0.61287563,0.56524944,0.53031318,0.51019932,0.62545499,0.60935991,0.62249458,0.69349491,0.73429814,0.74338453,0.71886826,0.76594712,0.72011074,0.66981781,0.67113951,0.5358744,0.51473116,0.53381129,0.54456452,0.52700554,0.5281835,0.41379159,0.52693851,0.53520988,0.55913245,0.5146513,0.562916,0.57395823,0.58450179,0.51865166,0.44776153,0.49826139,0.51988714,0.51417197,0.56943148,0.48580669
OIL-040,0,sim_PLSDA_oils_demo,1,SOY,0.46029694,0.51163744,0.45466894,0.44691415,0.45916325,0.54866113,0.52651556,0.53075847,0.61693904,0.66037252,0.6483346,0.76627384,0.77089114,0.68111257,0.69024849,0.70250589,0.65878484,0.57954616,0.51474883,0.54288932,0.45550756,0.5592577,0.55382111,0.59232606,0.60922878,0.71238197,0.62860653,0.71688956,0.78806545,0.73209766,0.79421461,0.71628607,0.62577688,0.65689517,0.57322888,0.57450225,0.51518614,0.51866224,0.58851808,0.59669091,0.60594111,0.64878588,0.75273914,0.82666224,0.73339953,0.77985093,0.6048232,0.61936709,0.65209374,0.64382274,0.54822979,0.54230337,0.51515533,0.53600145,0.44994815,0.5192828,0.48353668,0.4987088,0.52531621,0.52711691,0.50580069,0.51760794,0.47702378,0.56051456,0.52133473,0.47311186,0.55059235,0.53803667,0.50638003,0.46596689
OIL-041,0,sim_PLSDA_oils_demo,1,SOY,0.52936922,0.56190704,0.50054238,0.48702996,0.60412374,0.57702595,0.55077767,0.5289899,0.623919,0.69869082,0.73797199,0.78727226,0.78423641,0.82881968,0.7345087,0.69600947,0.6409824,0.58653663,0.65242838,0.48717819,0.55824135,0.51929706,0.53155625,0.61830617,0.59386842,0.69682325,0.83228643,0.76749054,0.80397937,0.74687411,0.77609148,0.66090582,0.68392463,0.62829855,0.68294212,0.56791599,0.54483973,0.53971351,0.53226833,0.64249509,0.71252004,0.68781811,0.77423454,0.76188699,0.80128305,0.78992913,0.6814209,0.66201863,0.68107605,0.52368201,0.56439939,0.52537507,0.4862224,0.51634974,0.5231906,0.53375508,0.4140281,0.59707298,0.52454628,0.59479934,0.54930622,0.52250951,0.52979267,0.6464131,0.57405809,0.49349245,0.5146837,0.54986748,0.52810636,0.50824619
OIL-042,0,sim_PLSDA_oils_demo,1,SOY,0.45465326,0.52784532,0.54253084,0.58936397,0.48252099,0.53898766,0.52791134,0.60633971,0.59148516,0.61936667,0.6313385,0.77402345,0.77969064,0.81952668,0.72710496,0.80785608,0.62322016,0.62335639,0.58272777,0.58913518,0.53311881,0.51247076,0.53212483,0.63778217,0.65094989,0.70510115,0.69982908,0.7684447,0.78554779,0.79613782,0.68950531,0.68030603,0.73659646,0.6087584,0.58481311,0.54679002,0.49083384,0.51045974,0.58631657,0.57680078,0.72783083,0.63447153,0.66018952,0.75217965,0.72884981,0.81887229,0.71326359,0.70538103,0.57916808,0.61797428,0.58156165,0.58051664,0.52383698,0.52691724,0.50950393,0.54116307,0.49418183,0.54386377,0.51269226,0.5734293,0.53497935,0.51161393,0.4971439,0.5707123,0.44122349,0.4399776,0.52306758,0.45720583,0.47826061,0.4771385
OIL-043,0,sim_PLSDA_oils_demo,1,SOY,0.50680162,0.50506547,0.51904924,0.54564416,0.56053554,0.49033926,0.58330571,0.67054266,0.5602424,0.70462536,0.76600877,0.78798978,0.80180103,0.71675878,0.83348413,0.72756109,0.67965508,0.68830918,0.5088889,0.57864156,0.59994524,0.61432007,0.63239876,0.72322219,0.6945556,0.76856555,0.79981846,0.74016468,0.78268862,0.8914475,0.77376404,0.74502013,0.65144318,0.64745957,0.57235084,0.57508016,0.60956439,0.55696298,0.58361936,0.69392453,0.70228337,0.75176461,0.8143882,0.78253589,0.79833818,0.71802626,0.79637462,0.71316437,0.67246114,0.58466896,0.61773602,0.59296718,0.55930058,0.47969856,0.50615228,0.5398473,0.51425711,0.57458795,0.61277903,0.55204503,0.57591567,0.54730847,0.60265966,0.58487781,0.6471594,0.53672805,0.49738453,0.55059293,0.52477674,0.54169419
OIL-044,0,sim_PLSDA_oils_demo,1,SOY,0.50737166,0.54791193,0.43500306,0.56375637,0.53093219,0.52466036,0.53046719,0.63408909,0.59649516,0.71733224,0.69998797,0.76870383,0.80892461,0.87705723,0.78059583,0.73516995,0.67042111,0.61237703,0.56349931,0.63897581,0.62018936,0.54333168,0.59979191,0.57990471,0.70619923,0.7406537,0.75877621,0.75245714,0.82937853,0.81812094,0.76277593,0.76246813,0.63564852,0.6225092,0.57209368,0.58924993,0.63937827,0.57738139,0.63891672,0.62435717,0.65241435,0.7568229,0.78147129,0.84648457,0.83804365,0.75297329,0.65021587,0.71200869,0.61062279,0.55692857,0.51872138,0.52725652,0.52902587,0.52844517,0.60659386,0.47767973,0.53225472,0.49430423,0.51763881,0.56982556,0.5847938,0.50817403,0.59203297,0.49665144,0.56386034,0.51918138,0.55580625,0.5416852,0.50741621,0.53796607
OIL-045,0,sim_PLSDA_oils_demo,1,CAN,0.42817251,0.46615424,0.535955,0.45291459,0.46295567,0.56215076,0.51350574,0.53872906,0.53600332,0.63026212,0.70520201,0.72579204,0.78642992,0.7703315,0.77370609,0.71980868,0.62877248,0.6201973,0.62091138,0.53972541,0.47348061,0.53935751,0.54624674,0.69146006,0.65952971,0.62964073,0.70743152,0.80386107,0.7128791,0.73432394,0.73050913,0.68751384,0.62612209,0.61209958,0.51407393,0.52187654,0.50901942,0.52366453,0.54171011,0.58570687,0.6149145,0.65973702,0.7343312,0.78582299,0.79764941,0.69947576,0.72209431,0.69647781,0.60710136,0.59207912,0.52925229,0.52804642,0.43608064,0.50622233,0.46366588,0.45123668,0.46766643,0.51867861,0.51680291,0.48896204,0.5135581,0.46270281,0.42918143,0.40433915,0.47768901,0.47306413,0.48035629,0.48160285,0.52762067,0.44207981
OIL-046,0,sim_PLSDA_oils_demo,1,CAN,0.56369548,0.51800385,0.47583972,0.40811745,0.45971156,0.48796458,0.51520167,0.55305938,0.57566027,0.68043041,0.64072412,0.72337107,0.73535897,0.79478055,0.74540225,0.72490873,0.64381026,0.74209873,0.58414808,0.50408006,0.52869858,0.52228899,0.59117171,0.6333102,0.65139829,0.67354994,0.69516619,0.81655855,0.81112464,0.64401551,0.74042949,0.66722348,0.64223931,0.68208723,0.53276369,0.51106255,0.63888274,0.52455155,0.55220549,0.63964193,0.61664326,0.68618671,0.73811938,0.81466088,0.70309861,0.7208342,0.71575193,0.71175877,0.60085882,0.60071258,0.52635941,0.50607058,0.55686153,0.48726044,0.61034781,0.50396513,0.53363489,0.51788907,0.42582523,0.42640902,0.50749053,0.44376769,0.46609101,0.47453751,0.45132027,0.53532893,0.51916025,0.49498935,0.51044692,0.45172518
OIL-047,0,sim_PLSDA_oils_demo,1,CAN,0.47918487,0.57587453,0.48738358,0.49365148,0.56654607,0.55458687,0.50664695,0.59592324,0.61105464,0.66164975,0.73935261,0.78582691,0.8297624,0.80965044,0.77795392,0.71589392,0.72718001,0.58745196,0.65841075,0.64812239,0.64232082,0.55948237,0.67172975,0.66710079,0.70563062,0.83911261,0.75638994,0.80187308,0.7426593,0.69877504,0.77574208,0.74441813,0.7119128,0.65290658,0.52474448,0.5939848,0.61188163,0.63265916,0.54976627,0.58421236,0.635139,0.68496245,0.68913971,0.75653836,0.85665999,0.73723857,0.67142762,0.68957368,0.63153513,0.63460799,0.54230402,0.53971591,0.49734668,0.48278559,0.53173833,0.51952148,0.51145677,0.53395147,0.47997546,0.5137836,0.52117306,0.56032651,0.43714237,0.49883421,0.53090633,0.48976985,0.51656089,0.54039137,0.53211199,0.44377155
OIL-048,0,sim_PLSDA_oils_demo,1,CAN,0.36852663,0.50803765,0.49098705,0.53823377,0.4904647,0.45119085,0.55625809,0.48727342,0.57937905,0.62101936,0.71411456,0.76604168,0.74035897,0.72929148,0.73121206,0.66506391,0.67377652,0.60210674,0.60621354,0.63387387,0.48910159,0.50911293,0.57500162,0.60552703,0.61705039,0.60662998,0.81877488,0.75047622,0.714118,0.65929089,0.78858968,0.70854541,0.65186843,0.56699393,0.56828111,0.53133536,0.43448371,0.64541195,0.57343356,0.57412761,0.5920651,0.65426411,0.81971782,0.7679723,0.70767513,0.78371475,0.73455999,0.69013347,0.6147703,0.56888334,0.54901321,0.50683559,0.44243154,0.44972866,0.48563609,0.41845445,0.47655172,0.46875352,0.51101321,0.48790776,0.4693439,0.46941647,0.49877788,0.51921527,0.43805362,0.41950003,0.45125778,0.38812779,0.44963732,0.51495301
OIL-049,0,sim_PLSDA_oils_demo,1,CAN,0.57634407,0.51724535,0.52151743,0.4931023,0.49998759,0.54136129,0.58273101,0.59019094,0.53030264,0.68198153,0.71573458,0.73825951,0.79565228,0.82753487,0.77146142,0.69155194,0.80345306,0.59391355,0.60143882,0.61241162,0.52148787,0.58787036,0.55390593,0.6110373,0.63267362,0.7789631,0.69343218,0.80025797,0.79486814,0.70809156,0.83456164,0.70464456,0.67347804,0.64983407,0.56817059,0.54803368,0.56579313,0.53457821,0.59202981,0.67137964,0.62238611,0.64707225,0.74910377,0.80060422,0.82080954,0.73597827,0.65185548,0.73670304,0.68116651,0.66903396,0.53384436,0.57037819,0.57230474,0.44844186,0.56023455,0.57300624,0.5377786,0.53571907,0.54372932,0.50333257,0.45590326,0.47657273,0.50661797,0.47962944,0.55502428,0.49733278,0.50922934,0.44257139,0.5296916,0.48543694
OIL-050,0,sim_PLSDA_oils_demo,1,CAN,0.45234697,0.53483231,0.50616709,0.50963845,0.59670647,0.57549404,0.5190029,0.63615843,0.59326196,0.70124988,0.65320049,0.75653138,0.78566938,0.78542842,0.70934377,0.65211857,0.67490568,0.65124681,0.59156521,0.55539219,0.50337319,0.56242437,0.57340574,0.600504,0.67802834,0.72244905,0.77269919,0.77679059,0.76765962,0.7878228,0.75976984,0.66945882,0.67768658,0.58213482,0.53332242,0.55392054,0.5491011,0.60552162,0.5832394,0.64902508,0.72866289,0.72334209,0.77722277,0.76430977,0.83423285,0.79010608,0.75704769,0.66370096,0.62256037,0.5887086,0.58447676,0.59890658,0.47866656,0.51543017,0.51285037,0.61407772,0.5139299,0.50003441,0.55119381,0.47691731,0.54294695,0.45424598,0.60222235,0.51943504,0.50412194,0.4686459,0.51517918,0.52455468,0.49200021,0.56894423
OIL-051,0,sim_PLSDA_oils_demo,1,CAN,0.49732808,0.50520341,0.54892916,0.48097395,0.50320701,0.52083916,0.54987613,0.47886407,0.63764593,0.63660696,0.66625523,0.74051888,0.69268465,0.74022235,0.75299333,0.7549292,0.6446454,0.6151187,0.55082048,0.57595252,0.52024283,0.564085,0.55477682,0.56306491,0.60361117,0.66215775,0.77548869,0.88728404,0.76550831,0.73874292,0.76608372,0.72740129,0.64307627,0.69825109,0.62016916,0.59749135,0.55034476,0.55580954,0.60759956,0.5451004,0.61168102,0.69516511,0.6765049,0.83091416,0.74957484,0.74375937,0.74060964,0.74242433,0.6315306,0.55815505,0.51619624,0.53780977,0.48933127,0.59832282,0.54888565,0.36946949,0.45804535,0.50897266,0.58729325,0.41387709,0.47692044,0.54284213,0.46689354,0.4974896,0.47461404,0.52149596,0.45066463,0.49555259,0.43642858,0.53371754
OIL-052,0,sim_PLSDA_oils_demo,1,CAN,0.50771595,0.5108782,0.51115645,0.49703335,0.5222008,0.5392544,0.53805184,0.54395226,0.59078005,0.63853722,0.7286794,0.76325647,0.86382167,0.77930261,0.76376895,0.69063115,0.66721395,0.59668719,0.54340434,0.56309571,0.61501826,0.52616637,0.55723561,0.61530332,0.66857973,0.75504775,0.71819228,0.76895787,0.75663677,0.74805747,0.74058317,0.67782945,0.69669211,0.67787677,0.57874885,0.54578424,0.61473606,0.55593824,0.64189802,0.49884339,0.65102443,0.66456008,0.72737901,0.81093749,0.831294,0.76748298,0.77854211,0.65067469,0.65751197,0.59610112,0.49891508,0.57797741,0.49903621,0.53403596,0.48131101,0.51665937,0.53321105,0.50818567,0.48907865,0.53235212,0.51662166,0.49137116,0.50995324,0.50496381,0.49463236,0.54434663,0.57783497,0.48789734,0.50362195,0.47078382
OIL-053,0,sim_PLSDA_oils_demo,1,CAN,0.52455563,0.45259851,0.49368853,0.42767334,0.46043311,0.51356104,0.5298434,0.54715623,0.59342613,0.63561961,0.72067448,0.79505948,0.77749953,0.75181576,0.68970587,0.66640275,0.62025571,0.64209255,0.48590694,0.49479925,0.48015852,0.56540128,0.52543676,0.51340738,0.54616598,0.68517923,0.64857694,0.78825571,0.80071026,0.70747749,0.71568491,0.69307304,0.58885482,0.55619774,0.54384043,0.5421867,0.54255639,0.45922425,0.56187273,0.57983697,0.61687431,0.65667015,0.75967204,0.811006,0.73957878,0.7374666,0.6474492,0.64923066,0.64402355,0.56813467,0.55114678,0.50748588,0.49992837,0.51353119,0.51277732,0.40212776,0.4704985,0.3625304,0.47941362,0.4570854,0.4552639,0.54140793,0.49343758,0.5244334,0.5043401,0.54675659,0.46961343,0.46933357,0.47584866,0.57746951
OIL-054,0,sim_PLSDA_oils_demo,1,CAN,0.49439071,0.54855015,0.53118667,0.47620205,0.56992781,0.52806967,0.58296591,0.57625384,0.62245918,0.6114886,0.75155431,0.71334898,0.77228853,0.8140328,0.73148633,0.70488752,0.64780877,0.60505472,0.58569405,0.4810201,0.51471181,0.62771079,0.48945431,0.57572402,0.67469417,0.61371692,0.69397103,0.72675871,0.83558001,0.73282743,0.76935631,0.68190327,0.66395065,0.57526507,0.52542614,0.54009095,0.4917599,0.54253489,0.5723238,0.61613956,0.70393437,0.74723536,0.68982053,0.73396395,0.84090395,0.75753785,0.71095581,0.72364063,0.60699305,0.62495871,0.54020613,0.57901507,0.5054389,0.53484306,0.53959694,0.55958215,0.46209647,0.47821885,0.43674223,0.42445757,0.47347617,0.49974042,0.54348329,0.47671331,0.56978993,0.50063399,0.52858795,0.4830957,0.44301986,0.55668106
OIL-055,0,sim_PLSDA_oils_demo,1,CAN,0.54010641,0.48104789,0.51995963,0.49184464,0.48845563,0.50500019,0.55446357,0.50576072,0.55442484,0.68254344,0.66742567,0.75069623,0.71720287,0.76380872,0.79090128,0.70533879,0.6576408,0.58485,0.56780589,0.47136737,0.47197021,0.56312711,0.49852248,0.65504956,0.67823666,0.64071923,0.64257433,0.66341836,0.78666076,0.76517083,0.79935771,0.72731286,0.72723454,0.65697565,0.54773127,0.55269827,0.51425174,0.53819322,0.54814048,0.59165407,0.69834232,0.66951955,0.68726903,0.73977367,0.75531791,0.73753932,0.72076309,0.66474729,0.67774806,0.62539259,0.52306188,0.56797346,0.5686902,0.50368912,0.46497684,0.51394014,0.49236738,0.50867753,0.45262145,0.57898268,0.48723125,0.51636672,0.47137975,0.50148983,0.40356781,0.44328781,0.39650089,0.53502309,0.43997898,0.43323774
OIL-056,0,sim_PLSDA_oils_demo,1,CAN,0.49748478,0.51075133,0.56112131,0.53958891,0.53664162,0.55903699,0.47113927,0.62781683,0.66859304,0.73926677,0.63508803,0.73153013,0.77090396,0.86306776,0.79200468,0.79545565,0.6899451,0.59756988,0.53455907,0.54259782,0.61617978,0.53552911,0.52090445,0.63702779,0.7222298,0.80944032,0.80311588,0.82234501,0.72600077,0.77707904,0.75354864,0.73713944,0.65084209,0.67687783,0.65898851,0.58772394,0.55963582,0.58163791,0.60722841,0.65595461,0.63270753,0.77275918,0.75967607,0.78124894,0.82525428,0.7559457,0.78572747,0.79099234,0.67362361,0.62000545,0.55727659,0.6111894,0.58071806,0.46541307,0.56025066,0.49204896,0.47058436,0.5143405,0.4905013,0.56353876,0.46638532,0.4652355,0.47888924,0.51121261,0.54048916,0.47317116,0.4877596,0.45389414,0.50878917,0.47571972
OIL-057,0,sim_PLSDA_oils_demo,1,CAN,0.60946021,0.49520858,0.53868275,0.55171381,0.51713333,0.53196786,0.58946206,0.57990079,0.61800125,0.67866546,0.78295584,0.76962934,0.82341018,0.81878668,0.81236327,0.73103197,0.66648981,0.60994682,0.61030055,0.58523974,0.56411294,0.59270262,0.70096369,0.67148304,0.60808218,0.77821058,0.7729465,0.75699814,0.8654282,0.80318097,0.82355122,0.72133806,0.74460362,0.64808747,0.66135573,0.63038872,0.53951378,0.60309583,0.62519823,0.63618759,0.61037047,0.7405155,0.75996379,0.85055675,0.769523,0.83781678,0.77064441,0.82130548,0.63045612,0.66463931,0.56967665,0.52805231,0.59307588,0.56496465,0.57797022,0.46935734,0.48480647,0.5559254,0.49835883,0.58504674,0.59898166,0.52719558,0.53368356,0.53352239,0.52078483,0.53098945,0.52318132,0.54962372,0.51916309,0.51165345
OIL-058,0,sim_PLSDA_oils_demo,1,CAN,0.47110267,0.49259423,0.51805025,0.5804735,0.43719066,0.45587902,0.52448859,0.55541725,0.59166652,0.71818743,0.62390252,0.73729368,0.72041987,0.76055205,0.76776078,0.6444776,0.69053752,0.567051,0.52226499,0.47722058,0.50979733,0.51752021,0.60284924,0.54083051,0.65338766,0.72311886,0.72047261,0.79174748,0.74727555,0.78625184,0.73283039,0.79556296,0.65428567,0.61071873,0.60253699,0.53978336,0.5410749,0.56523859,0.60471099,0.57806324,0.64738738,0.68577188,0.7640044,0.79893111,0.79160377,0.77251121,0.80021117,0.64806816,0.6996288,0.61130629,0.49312706,0.53457567,0.45684813,0.49929528,0.4182541,0.52630469,0.46456883,0.56182114,0.46665704,0.49776687,0.44127157,0.4904363,0.4973366,0.51585381,0.59445027,0.56154313,0.52702091,0.4826937,0.48442115,0.48354374
OIL-059,0,sim_PLSDA_oils_demo,1,CAN,0.51479278,0.48600546,0.51009784,0.48091271,0.50992908,0.57412067,0.53416441,0.58420517,0.5526509,0.63059641,0.67509045,0.75702019,0.77674575,0.73941557,0.75505008,0.70119334,0.58110821,0.5939893,0.60612872,0.50207523,0.57616276,0.5699195,0.57151794,0.61617352,0.60785201,0.71296565,0.66901746,0.71148946,0.79201017,0.72108335,0.82210323,0.7183681,0.69761577,0.65728112,0.56653106,0.55788121,0.56388142,0.54797903,0.62882814,0.63941308,0.64994747,0.73956396,0.73527274,0.79350786,0.7782798,0.74747145,0.68724047,0.65040124,0.63225859,0.6028606,0.46498654,0.508651,0.53222124,0.49326662,0.45055899,0.45179316,0.55856489,0.42699159,0.47719983,0.52344225,0.41043678,0.51717334,0.42798837,0.53047756,0.59669736,0.44001058,0.48733833,0.50472932,0.46721904,0.51187743
OIL-060,0,sim_PLSDA_oils_demo,1,COC,0.47048655,0.46079553,0.49243617,0.51625899,0.54278996,0.51343524,0.5009763,0.56774063,0.62960423,0.71463065,0.83977516,0.88564791,0.93961324,0.93994423,0.89311809,0.80510114,0.76754176,0.65430548,0.58996927,0.5074698,0.50168574,0.49897579,0.55002471,0.46923475,0.58543188,0.57986404,0.56392678,0.57125515,0.63160922,0.61754186,0.51960307,0.55538316,0.52764432,0.54301152,0.50486776,0.53775489,0.49337691,0.50439619,0.57276583,0.5273268,0.53546524,0.57726488,0.54247372,0.54334342,0.60993312,0.58164833,0.53371889,0.60820346,0.57757113,0.54004772,0.49457194,0.49712733,0.47520816,0.52857135,0.43462884,0.50194316,0.49134132,0.44520732,0.51511435,0.47675698,0.56509723,0.53728542,0.4881138,0.47529072,0.46305921,0.46149988,0.50294429,0.43300783,0.52917314,0.51825922
OIL-061,0,sim_PLSDA_oils_demo,1,COC,0.52815127,0.59042566,0.51146035,0.57115838,0.49726319,0.49867064,0.58905164,0.51564583,0.6938221,0.73549741,0.80251449,0.89999183,0.94570771,0.99323299,0.93588188,0.83757811,0.7351297,0.62574358,0.60813377,0.45507616,0.57433655,0.52883312,0.46683562,0.56807303,0.52687365,0.56260885,0.52732275,0.58183848,0.62416246,0.61212646,0.61699079,0.5603323,0.59334822,0.49316033,0.51401632,0.4198452,0.50832715,0.4998791,0.52166087,0.56600841,0.61407032,0.5349012,0.5909812,0.53508583,0.69054212,0.6636077,0.61595072,0.62631674,0.59130698,0.51260004,0.55169223,0.55918153,0.5376801,0.49812832,0.48411421,0.5935768,0.50908554,0.5199322,0.44192608,0.52531814,0.50787764,0.46352608,0.54957793,0.49788931,0.52392487,0.51642052,0.47002908,0.50957284,0.55336489,0.45376954
OIL-062,0,sim_PLSDA_oils_demo,1,COC,0.44813099,0.51457769,0.46752957,0.50058857,0.5411151,0.46431611,0.59434469,0.57984313,0.63484443,0.7831272,0.82642242,0.83035056,0.92529021,0.89887297,0.89924038,0.79713209,0.71678103,0.68827313,0.51882629,0.55571496,0.5127416,0.52819927,0.42484556,0.49369038,0.53251068,0.63373969,0.53491404,0.55321705,0.59650972,0.61459905,0.56729858,0.57940997,0.49187316,0.60647066,0.50978574,0.51754525,0.61537362,0.48598671,0.46656377,0.49042896,0.61529584,0.5613074,0.55620202,0.5989247,0.60247288,0.63885229,0.61312741,0.57986883,0.56317989,0.49261308,0.45934305,0.51521892,0.51320044,0.49506012,0.49765779,0.51047086,0.43914374,0.47103,0.44989498,0.41012711,0.50581924,0.4754124,0.50082504,0.50978186,0.44836523,0.54481182,0.49664872,0.52487991,0.55227334,0.46545515
OIL-063,0,sim_PLSDA_oils_demo,1,COC,0.49816557,0.53575073,0.53808517,0.41641307,0.55925401,0.49789446,0.5614111,0.53209874,0.61293332,0.74296634,0.74313663,0.92165415,0.91248777,0.94718621,0.90092113,0.80016063,0.78927149,0.74996126,0.61818313,0.48223882,0.53335905,0.51263055,0.56251134,0.54071538,0.64094251,0.60176049,0.61548634,0.59085844,0.57499016,0.60489419,0.60633534,0.6015382,0.5717459,0.5340941,0.53171099,0.50032086,0.47408868,0.48016689,0.54229794,0.56540316,0.54616926,0.56979295,0.61574435,0.6012846,0.73642881,0.61007681,0.56987197,0.49867983,0.55859697,0.57217769,0.51378216,0.55494375,0.47473033,0.49046337,0.48503054,0.46141105,0.43173287,0.4527582,0.4147369,0.5211522,0.48392506,0.48663219,0.46780803,0.52884993,0.4569292,0.5581689,0.53418031,0.46281562,0.48275795,0.51594554
OIL-064,0,sim_PLSDA_oils_demo,1,COC,0.48723166,0.49932502,0.62054871,0.55993784,0.56562601,0.62102748,0.58759174,0.59478832,0.60705555,0.79077663,0.79716613,0.94415968,0.88303498,0.94512948,0.92226542,0.80025496,0.74285374,0.65102805,0.62613582,0.52849418,0.54599165,0.60461691,0.61011876,0.64186753,0.59981194,0.58574025,0.63268708,0.66098402,0.6497544,0.69428972,0.59990459,0.60207469,0.53089839,0.55644312,0.60732107,0.54891518,0.57158608,0.55996325,0.58914465,0.53095609,0.60422108,0.63043635,0.66818573,0.69050892,0.61886843,0.57566746,0.5929609,0.65204625,0.5384534,0.50411317,0.49158637,0.48127409,0.55527084,0.49180019,0.49545493,0.56517597,0.55646591,0.55411812,0.48818584,0.45064007,0.5328548,0.58898828,0.50452732,0.50789793,0.38884333,0.5880673,0.51080276,0.44912593,0.51640499,0.49494975
OIL-065,0,sim_PLSDA_oils_demo,1,COC,0.59570003,0.47772568,0.51722135,0.49841259,0.49383878,0.55273807,0.62230379,0.62128554,0.64979375,0.76768958,0.84261501,0.94058299,0.93061224,0.96026425,0.97952808,0.84172589,0.7944567,0.67453109,0.61441927,0.53980247,0.66111204,0.58822021,0.59195998,0.60320781,0.53305564,0.5623369,0.60293603,0.55596707,0.59306231,0.66012334,0.53145176,0.56499249,0.57817248,0.56003403,0.54691502,0.50677381,0.56856555,0.43879169,0.55773257,0.49431622,0.5442661,0.59080043,0.6767788,0.58875061,0.57129175,0.6624686,0.60580804,0.5901317,0.59117589,0.58987032,0.54095009,0.43929471,0.50873567,0.51388869,0.46458913,0.46898238,0.53512781,0.51922841,0.47307723,0.53189295,0.51473087,0.45755219,0.49833911,0.45668094,0.4747067,0.52139998,0.46025219,0.45937052,0.52444034,0.46892099
OIL-066,0,sim_PLSDA_oils_demo,1,COC,0.48582722,0.48981181,0.51262331,0.48281371,0.47570903,0.50217734,0.59344473,0.6274229,0.65110582,0.78268875,0.84802067,0.85139514,0.99973203,1.0534834,0.91463406,0.76245747,0.80198326,0.68583469,0.59374266,0.47442039,0.52940518,0.51470065,0.51850601,0.5264435,0.59885341,0.56223325,0.61393938,0.53322768,0.5760248,0.61309465,0.56756885,0.58901424,0.56683374,0.55801506,0.52164825,0.5495807,0.46424948,0.50402446,0.58894717,0.52485777,0.62155437,0.59447727,0.58955412,0.61557418,0.6800822,0.59756801,0.55130359,0.59644471,0.58426969,0.57075476,0.46019625,0.52653976,0.50643587,0.48982595,0.47872497,0.51543508,0.4935149,0.49418838,0.57609878,0.49823437,0.44949788,0.45897693,0.46534411,0.43121621,0.53932932,0.57030306,0.47052058,0.53213502,0.46567945,0.46826544
OIL-067,0,sim_PLSDA_oils_demo,1,COC,0.47125856,0.52337952,0.52565033,0.49901844,0.47374282,0.54666493,0.59419937,0.53709694,0.6416769,0.70902839,0.77152955,0.82503791,0.88350327,0.92336686,0.88804685,0.81076644,0.7708495,0.61155373,0.52538179,0.48647978,0.52715908,0.48293543,0.4429609,0.55812501,0.5940853,0.56412744,0.60014985,0.53156935,0.53686576,0.59764934,0.58946214,0.5575186,0.53499623,0.50300735,0.57029861,0.46741992,0.49120222,0.53929134,0.46463129,0.52500748,0.53311484,0.58282705,0.47894406,0.577736,0.56219672,0.63144777,0.57127884,0.53145631,0.5684957,0.53963236,0.53956626,0.53149914,0.51209829,0.4544776,0.48723831,0.48619001,0.48835025,0.48726284,0.47737933,0.45690751,0.47077289,0.43438475,0.43772936,0.50258975,0.46086175,0.47733153,0.45210298,0.4083433,0.45760235,0.53824397
OIL-068,0,sim_PLSDA_oils_demo,1,COC,0.47131494,0.52666932,0.48064233,0.47828744,0.44687408,0.45467543,0.56603625,0.5981867,0.65736536,0.76362295,0.80376302,0.89771579,0.90693423,0.90926365,0.87263924,0.8686168,0.77260112,0.62361351,0.58778525,0.5632543,0.50383047,0.55625203,0.50222333,0.53239591,0.5493558,0.50770932,0.52901358,0.57757423,0.66547819,0.63902569,0.62375744,0.56483237,0.55246422,0.55454402,0.54795994,0.46543368,0.49267152,0.49157129,0.58245652,0.52198789,0.52951466,0.61524107,0.58361727,0.65133298,0.5794566,0.68761302,0.59468598,0.58398526,0.54795758,0.53660378,0.52414306,0.49940656,0.55539474,0.41164853,0.4512365,0.48380692,0.49304461,0.55580113,0.52021337,0.49808244,0.46598223,0.57666865,0.53293947,0.60018673,0.43852797,0.56711347,0.49423197,0.46409427,0.50338327,0.5030203
OIL-069,0,sim_PLSDA_oils_demo,1,COC,0.43722308,0.48163114,0.44532775,0.50304459,0.45161498,0.52875018,0.52957038,0.62753537,0.62098726,0.6782962,0.76240013,0.90402126,1.09576458,0.88488005,0.8738869,0.88785972,0.76669653,0.71089093,0.60887132,0.5738223,0.5340286,0.48053,0.53790177,0.50253112,0.58347877,0.4679726,0.51122871,0.60120351,0.61647801,0.68414121,0.56715749,0.60912816,0.49280947,0.64154727,0.42641528,0.58013488,0.49953058,0.44234563,0.49982512,0.57142714,0.54854392,0.54078989,0.60010143,0.60546788,0.64629115,0.63112714,0.57045613,0.53878359,0.55046975,0.52343917,0.5250566,0.46260639,0.50186891,0.46005274,0.51340866,0.50058554,0.47952368,0.42607239,0.53430712,0.4550872,0.47008856,0.48579851,0.50563722,0.46742784,0.46836085,0.43217051,0.53118973,0.46968449,0.47065955,0.40610123
OIL-070,0,sim_PLSDA_oils_demo,1,COC,0.5186422,0.52616985,0.44338012,0.49013531,0.51967078,0.45132256,0.51934655,0.60617116,0.68502824,0.66326521,0.763176,0.92876955,0.9107656,0.94341707,0.84131776,0.86192926,0.74521955,0.70581304,0.54805176,0.52180168,0.47241338,0.46483555,0.46514464,0.53695964,0.50962152,0.56373239,0.55407779,0.60886654,0.6337383,0.63047699,0.62305232,0.56673958,0.58208512,0.52494316,0.48729072,0.45931576,0.50211277,0.5161478,0.57879224,0.50802022,0.60475605,0.50387072,0.58151753,0.69212661,0.58892331,0.5606354,0.54162845,0.65255926,0.59239646,0.50123541,0.49011562,0.39808042,0.48367437,0.54938782,0.40760419,0.52756543,0.50902017,0.54731976,0.46262888,0.49499968,0.53897633,0.50440951,0.51334583,0.47232442,0.47775625,0.46173113,0.48991757,0.48665497,0.47509697,0.39952465
OIL-071,0,sim_PLSDA_oils_demo,1,COC,0.47102345,0.52110627,0.43965197,0.45680259,0.48291262,0.53750298,0.59160585,0.57922403,0.63145941,0.67926075,0.81319251,0.86060601,0.92319803,0.88339892,0.92162842,0.78899607,0.65791988,0.63780749,0.5313424,0.55536557,0.56477963,0.52663121,0.50299828,0.48803381,0.51891842,0.64279688,0.61754043,0.5473912,0.55633287,0.60712505,0.58611463,0.5709171,0.57280793,0.48646483,0.4820663,0.54398138,0.51133453,0.4899159,0.50385217,0.54741633,0.55224684,0.53366609,0.53260248,0.62397948,0.5475226,0.67218224,0.53002865,0.53189352,0.57918148,0.5746546,0.50439485,0.38425137,0.47911457,0.39632117,0.45788705,0.51101776,0.50839983,0.48155831,0.48640643,0.50144931,0.46072041,0.4615425,0.52477644,0.49594401,0.52501019,0.49278973,0.49210952,0.50982645,0.50529751,0.54522968
OIL-072,0,sim_PLSDA_oils_demo,1,COC,0.48946817,0.49295336,0.47185786,0.52169925,0.49105207,0.48074556,0.58492843,0.66146321,0.60164906,0.78432639,0.84043381,0.86567409,0.92377482,0.94576503,1.00404209,0.84915229,0.81735484,0.64965509,0.65276686,0.60061445,0.5189044,0.51591687,0.56306816,0.49247289,0.57154549,0.53960461,0.60909315,0.63902002,0.65565633,0.64165446,0.60818232,0.57836984,0.54708443,0.53849281,0.53011314,0.51558697,0.53818843,0.50513056,0.47651322,0.58063831,0.54203099,0.52590574,0.57427735,0.61025007,0.61910875,0.63048061,0.68450943,0.53573502,0.63161592,0.55509072,0.52051759,0.43277424,0.50408504,0.53774589,0.4674172,0.44745386,0.45578862,0.51470034,0.50922917,0.57159375,0.48838111,0.53981638,0.45874433,0.49865414,0.55148916,0.49602893,0.52851771,0.45655241,0.46989119,0.4856968
OIL-073,0,sim_PLSDA_oils_demo,1,COC,0.46287028,0.54053144,0.47367203,0.49591254,0.55522951,0.5074475,0.52668875,0.55646533,0.683971,0.75584551,0.88786927,0.96344971,0.86550666,0.95823942,0.97127737,0.82687079,0.7352466,0.60225225,0.63084615,0.50250825,0.52767181,0.50048671,0.53690527,0.5131999,0.53061303,0.60780065,0.58237843,0.63347348,0.57620763,0.66010599,0.61164437,0.6238929,0.58723032,0.47461532,0.41889855,0.55367643,0.48663548,0.4908451,0.48121533,0.50618039,0.55897171,0.60281324,0.57417823,0.5110942,0.6265181,0.60871875,0.55781843,0.51099302,0.48831823,0.57311987,0.48312287,0.54531753,0.48022236,0.52142754,0.44708334,0.43253504,0.47469078,0.41296914,0.42875383,0.50122838,0.54517536,0.48970106,0.41624429,0.44131167,0.48958376,0.47678692,0.578668,0.55832545,0.47058529,0.56674939
OIL-074,0,sim_PLSDA_oils_demo,1,COC,0.44781206,0.54380639,0.44533308,0.46767704,0.5611626,0.4941367,0.59166959,0.6087687,0.62645661,0.7494861,0.85520543,0.91799255,0.89548646,0.96948348,0.79600129,0.80489828,0.73361073,0.68916392,0.64318978,0.49130607,0.50066142,0.51111695,0.50139423,0.54771669,0.57561887,0.59599979,0.57217133,0.7278523,0.55826691,0.59467886,0.65575118,0.61044423,0.59077815,0.59166356,0.51775768,0.51886127,0.55514779,0.57634808,0.46715705,0.51181638,0.55174715,0.60849341,0.54098814,0.64165202,0.58022827,0.63229307,0.59807111,0.62803129,0.50594871,0.55138987,0.5016171,0.47419366,0.53369934,0.45259872,0.51914071,0.44674379,0.44022123,0.51552787,0.57415049,0.50619029,0.44920651,0.50062266,0.4748177,0.46009858,0.4954096,0.50834662,0.51938655,0.4453232,0.55220204,0.47452808
OIL-075,0,sim_PLSDA_oils_demo,1,PEA,0.44266691,0.48172232,0.44288634,0.48675081,0.44570233,0.53063835,0.51016242,0.51565461,0.59917574,0.56448435,0.64187884,0.71168697,0.72688872,0.71524287,0.72623135,0.63150908,0.60657144,0.53034552,0.53137322,0.46908705,0.46426343,0.51876593,0.49744175,0.59788572,0.64188009,0.66443687,0.70660484,0.72312854,0.65384988,0.66555304,0.70292373,0.61460132,0.59338064,0.58235349,0.56521535,0.46771182,0.52168442,0.55734694,0.5536061,0.62627089,0.58432229,0.69508858,0.76790752,0.73571458,0.75501374,0.74405275,0.58770605,0.60776735,0.63759934,0.50490129,0.53763359,0.52610334,0.5411947,0.477299,0.5133065,0.47578967,0.53900989,0.44971057,0.45390083,0.49181027,0.54470226,0.49292777,0.49246194,0.48900858,0.58626309,0.55232322,0.60080035,0.61342442,0.66052928,0.55386509
OIL-076,0,sim_PLSDA_oils_demo,1,PEA,0.49391686,0.48760953,0.42728201,0.45253827,0.46582045,0.50052432,0.47914096,0.61482608,0.5804574,0.66073726,0.6715004,0.70797046,0.75146793,0.70948044,0.62880408,0.68049421,0.58787129,0.62862935,0.50838708,0.53262047,0.52317544,0.50735214,0.53407844,0.57114372,0.62098312,0.66238132,0.68926875,0.71297098,0.63635877,0.77620571,0.72818891,0.61854041,0.59818856,0.50949278,0.44561857,0.52340076,0.59856296,0.54032629,0.55854217,0.58041943,0.63787816,0.65361817,0.65950518,0.6946743,0.75384298,0.6988922,0.60977702,0.66923685,0.61455166,0.62104718,0.54142165,0.51626289,0.49771729,0.46244718,0.50100521,0.46737779,0.44586353,0.48278688,0.49076484,0.50202935,0.5162953,0.58203958,0.49463729,0.57961839,0.55620096,0.59677095,0.65501637,0.60237412,0.62460645,0.62744913
OIL-077,0,sim_PLSDA_oils_demo,1,PEA,0.44040767,0.46023717,0.47759853,0.51107038,0.43236226,0.54683829,0.53316486,0.55240105,0.4830873,0.6176041,0.61926498,0.71972575,0.71048952,0.69807668,0.6566218,0.64553542,0.63666113,0.50254536,0.50647086,0.50825618,0.50592298,0.5031288,0.54897491,0.57310949,0.52162627,0.68471598,0.62218575,0.75191885,0.74817392,0.64215453,0.69680617,0.71538886,0.67909431,0.59407373,0.52472966,0.58226357,0.46344827,0.46622187,0.57244717,0.52043513,0.62804685,0.6039302,0.65880572,0.75629287,0.71482181,0.73192099,0.67012367,0.62037267,0.56910195,0.52177756,0.54447576,0.52508696,0.56002814,0.49698117,0.47097434,0.49947212,0.53636912,0.4206988,0.54511973,0.44641366,0.44859682,0.58875763,0.54061366,0.49538266,0.58183848,0.63761687,0.6414986,0.67197969,0.63366241,0.61222205
OIL-078,0,sim_PLSDA_oils_demo,1,PEA,0.44178938,0.49079053,0.42995339,0.51208058,0.48539953,0.54345617,0.54581729,0.51673424,0.45909115,0.55168331,0.65554752,0.77736547,0.71998049,0.74382935,0.69772582,0.55348424,0.63913346,0.56316957,0.48160655,0.55146362,0.51714101,0.64519567,0.45763727,0.54315469,0.61288101,0.63379912,0.71038757,0.74606857,0.74165948,0.72055364,0.66998886,0.67337989,0.66453061,0.59678565,0.47685246,0.6064879,0.50889782,0.566378,0.52666177,0.61746331,0.59798253,0.71167011,0.74157328,0.68916013,0.60624744,0.72521878,0.64323628,0.68299432,0.61697946,0.54891537,0.49561064,0.47806528,0.55713619,0.52908624,0.47773349,0.49736777,0.52038839,0.4758995,0.41104285,0.55037499,0.47285658,0.46624148,0.502943,0.48386733,0.50903478,0.5971272,0.59060344,0.65015697,0.64040831,0.67569176
OIL-079,0,sim_PLSDA_oils_demo,1,PEA,0.50034175,0.47522947,0.54528145,0.51860657,0.51798187,0.58577402,0.53559098,0.55797513,0.52835396,0.5536523,0.64898939,0.71350441,0.68956598,0.69620026,0.74296413,0.65142992,0.60561499,0.57363828,0.5120416,0.57226435,0.59052904,0.58298475,0.62224344,0.51051155,0.66138447,0.65808222,0.75382647,0.69891523,0.76945108,0.80375939,0.73704262,0.6862415,0.63732598,0.62820894,0.56854113,0.50773635,0.56912624,0.61330035,0.57810803,0.63311712,0.67582383,0.6695462,0.70253931,0.69860429,0.74826995,0.74751448,0.66127784,0.64809019,0.66015065,0.55337724,0.57929742,0.54306349,0.50214537,0.54231703,0.52717398,0.49823171,0.50366038,0.51516861,0.51936277,0.51752671,0.5303491,0.48700407,0.5768535,0.53423954,0.56147767,0.6566288,0.67517205,0.57481663,0.64019254,0.64323656
OIL-080,0,sim_PLSDA_oils_demo,1,PEA,0.4457836,0.48463188,0.48240098,0.53995507,0.55310745,0.54028351,0.53834729,0.53040943,0.63937379,0.55111749,0.7068972,0.74917318,0.76120692,0.74984352,0.7338788,0.71636725,0.61799492,0.63203007,0.57137565,0.55694199,0.44212802,0.50326961,0.56680738,0.57806831,0.67641168,0.61863429,0.76166414,0.77175452,0.76019957,0.7273061,0.68007868,0.64642118,0.62883095,0.62023102,0.61633227,0.59614381,0.60061041,0.60085177,0.54749318,0.58153265,0.68266256,0.64841936,0.75872517,0.74575343,0.69663144,0.7578609,0.72050929,0.64024461,0.53492278,0.60379688,0.58005971,0.57381433,0.54407394,0.49078661,0.4484744,0.50808405,0.55104536,0.50816745,0.46334332,0.44567705,0.40933545,0.52711713,0.5485602,0.46422903,0.55665619,0.67344243,0.56237803,0.72123834,0.69926938,0.65951674
OIL-081,0,sim_PLSDA_oils_demo,1,PEA,0.53142067,0.53694104,0.57803698,0.46928192,0.44645079,0.44520076,0.54751446,0.60245293,0.5664307,0.65496141,0.69249907,0.77050671,0.77712802,0.76572857,0.70039039,0.67326445,0.67893902,0.60826645,0.62273542,0.52691324,0.58563651,0.54809268,0.5914865,0.53486352,0.66650229,0.64126165,0.75985566,0.77295227,0.7254142,0.80500979,0.78645891,0.76406921,0.70660246,0.6063024,0.64736492,0.51833498,0.54487527,0.5736284,0.5115481,0.65117503,0.66508348,0.67587442,0.6331436,0.71737008,0.78404343,0.76623778,0.67586459,0.62093997,0.63177911,0.54634251,0.49769751,0.59843259,0.52723876,0.53367895,0.50503542,0.55105006,0.55058263,0.50539474,0.49447783,0.4961631,0.54995854,0.5244493,0.48259026,0.60341208,0.58171532,0.63622456,0.70399617,0.68146348,0.68504616,0.62129034
OIL-082,0,sim_PLSDA_oils_demo,1,PEA,0.5490691,0.52903784,0.5103085,0.50854081,0.50809702,0.53619391,0.52101737,0.58533303,0.57863324,0.67908295,0.69014317,0.69973205,0.76627967,0.70731967,0.84713549,0.66246888,0.63922573,0.65861418,0.54943584,0.51701445,0.51013661,0.60147062,0.53146725,0.67157304,0.66488734,0.67369886,0.75864931,0.73982083,0.74075986,0.69735648,0.67033361,0.70483066,0.57775153,0.55892104,0.52867951,0.56480419,0.501971,0.5515071,0.67386383,0.60232529,0.70751733,0.68474746,0.67225893,0.71906272,0.72313635,0.75460558,0.73564894,0.67640094,0.61176035,0.55834098,0.54270341,0.50227389,0.49233911,0.61391314,0.54690313,0.46806033,0.50926908,0.521553,0.50659878,0.48185474,0.5246405,0.52072557,0.54570865,0.59110272,0.55525299,0.6204335,0.62450687,0.62333101,0.62730895,0.62912438
OIL-083,0,sim_PLSDA_oils_demo,1,PEA,0.45043712,0.45966093,0.49500213,0.54706563,0.54548079,0.43956402,0.50371544,0.56208644,0.55644712,0.60959284,0.62453667,0.70781002,0.72013611,0.69246353,0.72545062,0.70719703,0.6403869,0.53441293,0.55875301,0.56088987,0.56953686,0.58112811,0.59014173,0.57463468,0.6377731,0.68329929,0.69553561,0.7300719,0.68873964,0.73654623,0.68703049,0.6669177,0.70274065,0.60750186,0.57977585,0.52097019,0.47691318,0.49016587,0.54000028,0.57178185,0.54669646,0.63206081,0.69579464,0.68923253,0.71454374,0.72236496,0.73719786,0.66338018,0.66813667,0.5159235,0.59083425,0.51155866,0.54392229,0.57529892,0.53046019,0.50774998,0.55309417,0.47103239,0.53108495,0.50598722,0.50770496,0.47384076,0.52183864,0.54336812,0.55856299,0.608149,0.57119943,0.67680846,0.62119294,0.61292041
OIL-084,0,sim_PLSDA_oils_demo,1,PEA,0.4761118,0.46470436,0.54004026,0.45964573,0.46620644,0.49616296,0.4690667,0.49982564,0.58193174,0.62782149,0.69205933,0.71984952,0.69191821,0.6862355,0.7460134,0.58069424,0.62773612,0.60669437,0.59462696,0.58406977,0.56823677,0.60921577,0.48208147,0.57048596,0.63119603,0.56677748,0.6611238,0.76346941,0.70317644,0.76430718,0.72988386,0.71984927,0.6684403,0.6203965,0.59490761,0.51175236,0.60212494,0.52934997,0.5220906,0.59965826,0.56368165,0.7414409,0.79056726,0.76970281,0.69427992,0.73743474,0.70212052,0.63132109,0.59888398,0.59031041,0.52274365,0.51975821,0.50924803,0.44986737,0.4828141,0.47756784,0.55760621,0.49884199,0.51000102,0.50360208,0.49588238,0.50456037,0.50782028,0.44941133,0.56432662,0.57397306,0.61762102,0.60553825,0.59839403,0.60700188
OIL-085,0,sim_PLSDA_oils_demo,1,PEA,0.49949315,0.50674761,0.49779151,0.50406769,0.47967296,0.5019056,0.54943098,0.54060197,0.60453105,0.626498,0.71017683,0.67148085,0.7447871,0.76822154,0.68276977,0.68047431,0.64300312,0.58283503,0.59273744,0.5517035,0.52412159,0.57459767,0.57556759,0.70087696,0.56319917,0.66962913,0.6965413,0.72414514,0.65703465,0.73506055,0.68876588,0.70735315,0.59857993,0.67198848,0.57166358,0.53217962,0.55337399,0.48601611,0.51744563,0.60898237,0.61042413,0.66659453,0.61152336,0.67185808,0.66630802,0.78679592,0.68633556,0.7039765,0.60055198,0.614299,0.52119736,0.50264685,0.50148902,0.5024965,0.41545924,0.49214746,0.4541613,0.46420786,0.47986481,0.47572362,0.49077819,0.54092278,0.55351351,0.59887953,0.53421405,0.59181998,0.61522874,0.62374129,0.6568505,0.63457667
OIL-086,0,sim_PLSDA_oils_demo,1,PEA,0.50205506,0.54571843,0.49202985,0.54269739,0.50363926,0.529578,0.59820434,0.5273382,0.59681692,0.67501594,0.67730169,0.76422689,0.83671599,0.79351455,0.71363643,0.64601046,0.64088086,0.56937869,0.52945814,0.57032846,0.56301882,0.48040866,0.54409904,0.55285031,0.65485181,0.68226513,0.73974486,0.72768428,0.76134766,0.76378912,0.71420574,0.7191542,0.57604615,0.65015202,0.52612013,0.54269974,0.55611138,0.55398961,0.50964325,0.59278754,0.54141265,0.65916112,0.68696607,0.79012954,0.72691573,0.69773467,0.67026577,0.62549473,0.61654507,0.58718802,0.61228276,0.47627425,0.53613158,0.45004575,0.48958112,0.54336133,0.54622306,0.53168857,0.47585966,0.53644738,0.5603826,0.45149394,0.49530896,0.5408228,0.57751431,0.61601098,0.62633918,0.62248153,0.66032386,0.58313644
OIL-087,0,sim_PLSDA_oils_demo,1,PEA,0.51743158,0.51696501,0.53431314,0.53771985,0.53265251,0.57250822,0.4742204,0.55828957,0.59196826,0.61773326,0.67248149,0.72732815,0.72317433,0.81796445,0.79471751,0.65905917,0.67362298,0.62434774,0.56904926,0.53438248,0.55749873,0.55157959,0.5436368,0.62810583,0.63585173,0.71129076,0.65149041,0.69782422,0.61790013,0.71935236,0.67014398,0.70613886,0.685612,0.61303393,0.61347409,0.52851024,0.49704087,0.60216113,0.60253567,0.61980777,0.6146471,0.70898123,0.73013074,0.83509841,0.63491401,0.68001379,0.73302043,0.68976732,0.63137328,0.57333228,0.52427317,0.59345642,0.55986846,0.52065865,0.51946682,0.56812386,0.4609377,0.45455887,0.49543992,0.45976245,0.53691809,0.53537033,0.54709998,0.52962067,0.54569072,0.62114018,0.55305814,0.58852378,0.60016542,0.55750011
OIL-088,0,sim_PLSDA_oils_demo,1,PEA,0.4564203,0.52811944,0.49822044,0.43842299,0.40976041,0.58793194,0.5403727,0.56816475,0.56516571,0.58873572,0.72621545,0.66646458,0.72038259,0.73795392,0.68120611,0.6558579,0.61100151,0.57088199,0.62023688,0.5428858,0.57564293,0.50960579,0.54172792,0.56739265,0.56538392,0.66650047,0.64483827,0.68083169,0.73936904,0.74652555,0.68311561,0.68895735,0.69306652,0.66642122,0.61404846,0.54381291,0.59643425,0.53698306,0.55162464,0.62966919,0.64317001,0.68918019,0.67353023,0.72160521,0.75309375,0.74974318,0.7224932,0.59060767,0.70866436,0.61337123,0.51859828,0.57680187,0.5112321,0.49327175,0.56980023,0.4582655,0.46158103,0.46698755,0.47855237,0.63135049,0.486235,0.46908313,0.52702662,0.57038954,0.62105209,0.57241324,0.71565471,0.65892444,0.60843839,0.58420012
OIL-089,0,sim_PLSDA_oils_demo,1,PEA,0.50969547,0.49221213,0.51594969,0.60466598,0.57269799,0.50269948,0.50793286,0.59929369,0.5506591,0.56212219,0.64292023,0.66817624,0.68547234,0.73601992,0.69165192,0.67121905,0.65399496,0.57666966,0.50815049,0.51098802,0.57771021,0.50844275,0.52563849,0.61664368,0.59796834,0.63607838,0.7189229,0.72960874,0.72466876,0.72590441,0.76540839,0.70406677,0.62552626,0.59969304,0.64921807,0.55479462,0.56765799,0.56836059,0.55131024,0.6060392,0.59830748,0.66654809,0.76089757,0.73371886,0.80976154,0.74496564,0.72893044,0.67422182,0.59212019,0.63225081,0.61576935,0.54997885,0.52442036,0.50647076,0.53204112,0.55298375,0.54552341,0.51962116,0.51008671,0.51526434,0.50531716,0.47360836,0.60838001,0.51512971,0.53999095,0.67069018,0.66431858,0.6801592,0.62845653,0.62125328
"""

In [ ]:
# ---- Embedded demo spectra (fallback), Part 3: PCA outlier demo -------
# Exact content of data/nb12_demo_spectra_outlier.csv.
SAMPLE_CSV_OUTLIER = """sample_id,group,instrument,rep,role,1100,1126.53,1153.06,1179.59,1206.12,1232.65,1259.18,1285.71,1312.24,1338.78,1365.31,1391.84,1418.37,1444.9,1471.43,1497.96,1524.49,1551.02,1577.55,1604.08,1630.61,1657.14,1683.67,1710.2,1736.73,1763.27,1789.8,1816.33,1842.86,1869.39,1895.92,1922.45,1948.98,1975.51,2002.04,2028.57,2055.1,2081.63,2108.16,2134.69,2161.22,2187.76,2214.29,2240.82,2267.35,2293.88,2320.41,2346.94,2373.47,2400
CAL-000,0,m5_demo,1,calibration,0.0019026,-0.00460417,-0.00897976,0.00071737,0.00159434,0.00039016,-0.00117444,0.01071549,0.03042999,0.08055851,0.16658277,0.27797631,0.38640714,0.44551339,0.41854221,0.3197991,0.20550393,0.11059899,0.04948892,0.01932123,0.02115705,0.04942411,0.12327219,0.21883976,0.28592351,0.28242033,0.21702626,0.1221423,0.05501859,0.01203336,0.0024982,0.00383018,0.00453058,-0.00310915,-0.00200708,0.00739684,0.00146008,-0.00061248,0.00195583,-0.0022001,-0.00665915,0.00010457,-0.00139801,-0.00244409,3.116e-05,-0.00351938,-0.00197425,-0.00746316,0.00476937,0.00569613
CAL-001,0,m5_demo,1,calibration,-0.00456273,-0.00638449,0.00155319,0.00317235,-0.00173909,-0.00392943,0.00972925,0.00613698,0.02850546,0.06879834,0.14047955,0.24077692,0.33367032,0.36982473,0.35328106,0.27833234,0.17518282,0.0888652,0.04307317,0.01514166,0.0167524,0.04573931,0.09895107,0.17701364,0.22731894,0.22458084,0.17359231,0.09631911,0.0417398,0.0140032,0.0016402,0.00166639,0.00075118,0.000694,-0.00382191,0.00229304,0.00040567,-0.00061574,-0.00059461,-0.00267709,-0.00279745,0.0074494,0.00070698,0.00295236,0.00271996,-0.003015,-0.00059568,-0.00216668,0.0020713,0.00142037
CAL-002,0,m5_demo,1,calibration,-0.00386826,0.00372906,0.00361393,-0.00554859,0.00152927,0.00150941,-0.0048169,0.00323289,0.02609284,0.05512613,0.115626,0.20911976,0.28305857,0.33226211,0.31401925,0.24246004,0.15696375,0.08080636,0.03460312,0.01233731,0.02673359,0.05350505,0.13495592,0.23391974,0.30876073,0.3052004,0.22749174,0.13303782,0.05998948,0.01880709,-0.00010965,0.00465255,-0.01030375,-0.00050993,0.00366284,0.00413907,-0.00058519,0.00483827,0.00691345,-0.00109361,0.00441839,0.00025507,0.00433135,-0.00473798,0.0029864,0.00502549,0.00455022,0.00039888,-0.00792946,0.00114476
CAL-003,0,m5_demo,1,calibration,-0.00927302,0.00056534,0.00080748,0.002606,-0.00226789,0.00336314,0.00141228,0.00561267,0.03172434,0.0867172,0.16127384,0.27251684,0.3833248,0.43579093,0.41059506,0.30878392,0.19717084,0.1139846,0.0469473,0.01737315,0.02881659,0.06161269,0.12963142,0.23819665,0.31458027,0.31431698,0.23787722,0.13171265,0.05712162,0.02005492,0.00384704,2.242e-05,0.00390017,-0.00117723,0.00051973,-0.0035081,-0.00474495,0.002044,-0.00389119,0.0005334,0.00359329,-0.00196727,0.00847159,0.00461973,-0.00309977,0.00852922,0.0020504,0.00433714,-0.00188458,0.00143631
CAL-004,0,m5_demo,1,calibration,-0.00094282,0.00131173,-0.00268107,-0.00193139,-0.00284946,-0.00820679,-0.0006034,0.01162604,0.02709348,0.07615336,0.17076595,0.27271356,0.38868792,0.44804562,0.42253056,0.32681598,0.20834873,0.10829677,0.04302045,0.01143453,0.02677924,0.04645125,0.1099646,0.18789948,0.2544844,0.24867721,0.18870318,0.11009506,0.05405578,0.01723039,0.01179906,0.0017694,0.0009345,0.00582421,5.149e-05,-0.00592249,0.0020597,0.00159546,0.00075212,0.00182409,-0.00136264,-0.0003546,-0.00515224,0.00299239,-0.00273664,0.00489243,0.00040108,-0.00340646,0.00278982,0.00143992
CAL-005,0,m5_demo,1,calibration,0.00174363,0.00020653,-0.00223804,0.00600093,-0.00609939,-0.00356131,0.00781638,0.01686333,0.02283264,0.08350699,0.15979558,0.25840916,0.36827484,0.42027059,0.40017891,0.30851009,0.20216883,0.10418743,0.04892836,0.02523152,0.02256637,0.06957006,0.16284903,0.28347101,0.37138947,0.37596836,0.27753159,0.15823134,0.07115117,0.02505599,0.00118673,0.00187581,0.00236911,0.00375914,0.00258401,0.00065622,-0.00420366,-0.00673405,-0.00342906,-0.00278538,-0.00182215,0.00101381,0.00210009,0.00799214,-0.0067814,-0.00365384,-0.00682778,-0.00464823,-0.00578516,-0.00374091
CAL-006,0,m5_demo,1,calibration,-0.00381137,-0.00518913,-0.00299634,0.00274527,0.00249321,-0.00022224,0.004843,-0.00234478,0.01525665,0.05178556,0.0986922,0.16735229,0.23495606,0.26925449,0.25809192,0.20347281,0.12032305,0.06148064,0.02680174,0.01544393,0.02377669,0.05614677,0.13151656,0.22682225,0.30365387,0.29387451,0.23086978,0.13002509,0.05120391,0.01782128,-0.0006229,0.00525309,0.00440711,0.00035122,-0.00700311,-0.00825588,0.00029118,-0.00458719,0.00817066,-0.00602901,-0.00631206,0.00367384,0.00809268,-0.00667625,0.00321517,0.00153538,-0.0017313,-0.00557353,-0.00303391,-0.00188393
CAL-007,0,m5_demo,1,calibration,0.00227189,0.002244,-0.00150131,-0.00820613,-0.00265086,-0.00183703,-0.00278497,0.00878037,0.02825632,0.06064584,0.11769665,0.19538414,0.28312671,0.3266009,0.30409386,0.2354985,0.15079122,0.07789686,0.03010835,0.02315408,0.02394912,0.05556871,0.11489015,0.20433429,0.25880868,0.26619463,0.20075865,0.11117126,0.04559401,0.00902274,0.00517466,0.00128741,0.00190598,-0.00273231,0.00324184,0.00859596,-0.00207186,0.0024587,-0.01073495,0.00701751,0.00595464,0.00024262,0.00289034,-0.00600205,0.00452515,-0.00400581,-0.00136645,-0.00218806,0.00373103,0.00089242
CAL-008,0,m5_demo,1,calibration,0.00098087,0.00429567,-0.00485871,0.00194865,0.0038438,0.0090579,0.00768927,0.00598325,0.03463284,0.071088,0.15733191,0.26774374,0.37738902,0.42717171,0.40346926,0.31429816,0.19935511,0.10576305,0.04360145,0.02141059,0.03077752,0.07159838,0.15862206,0.2767082,0.35804534,0.36955493,0.28194873,0.16195279,0.06217722,0.01934738,0.00272086,0.00193798,-0.00192482,0.00076725,0.00277487,0.00025735,0.00655777,0.00461444,0.00106253,-0.00249788,0.00032333,-0.00213037,-0.0049753,0.00063636,0.00522908,-0.00163445,-0.00372162,0.00033958,-0.00247651,0.00361733
CAL-009,0,m5_demo,1,calibration,-0.00098626,0.00029191,-0.00074776,0.00041051,-0.00326912,0.00125863,0.00026304,0.00834011,0.03521866,0.07836569,0.1722876,0.29568978,0.41219795,0.47488542,0.44986127,0.34512628,0.21976679,0.11466634,0.05046072,0.0211141,0.03284421,0.04761782,0.11866476,0.20515701,0.26946936,0.26648552,0.20343083,0.119028,0.04599287,0.02070532,0.00501615,0.0068518,0.00087163,0.00091181,0.00096042,0.00515146,-0.00118883,0.00591081,0.00113272,-0.00641874,-0.00102125,0.00574376,-0.00013074,-0.00534655,0.00188727,-0.00070158,0.00267072,-0.00164582,0.00482172,0.00236445
CAL-010,0,m5_demo,1,calibration,0.00051472,-0.00515506,-0.00150225,0.00655895,0.00992867,0.00367732,0.00190203,0.01123409,0.02258733,0.05334299,0.11921583,0.19713914,0.28259585,0.32494296,0.30868483,0.24489502,0.14957528,0.0831475,0.03440282,0.01994887,0.02188309,0.05563084,0.12661485,0.21970947,0.29183378,0.30388332,0.22110511,0.12647933,0.0517493,0.01272748,0.00680901,-0.00242717,-0.00545968,-0.00461023,-0.00061928,0.00344292,-0.00209968,0.00999109,-0.00501666,0.00409749,-0.00695853,-0.00066559,0.00061903,-0.00430291,0.00014018,0.00045389,-0.00090694,0.00473641,-0.00074087,0.00542197
CAL-011,0,m5_demo,1,calibration,-0.00397772,0.00093796,-0.00039095,-0.00239737,0.00132921,-0.00214871,0.00486226,0.00566801,0.03216432,0.07861776,0.17174019,0.28029595,0.38555906,0.43816674,0.41859085,0.32459321,0.21438162,0.11805149,0.04746701,0.01848049,0.02356991,0.0371986,0.09264064,0.17159662,0.21946001,0.21979277,0.1661402,0.09097356,0.03800103,0.01288671,-0.00341731,-0.00096671,0.00289809,-0.00077478,-0.00953352,0.00051587,0.00317031,-0.00100948,0.0055042,0.00129999,0.00058148,-0.00154994,-0.00616162,-0.00407189,-0.00230112,-0.00731136,0.00240351,-0.00432766,-0.00646481,6.848e-05
CAL-012,0,m5_demo,1,calibration,-0.00312352,-0.00335961,-0.00200955,0.00649674,-0.00252838,0.00230258,0.00376205,0.0102097,0.03181664,0.07949214,0.17165152,0.28691304,0.39213283,0.46109073,0.43327904,0.33741127,0.21810979,0.10763204,0.04880676,0.02209561,0.0167795,0.04797391,0.1043014,0.19232454,0.24668816,0.24040326,0.18084678,0.10281095,0.04276105,0.01966877,0.00013537,-0.00531016,0.00203805,0.0025771,0.0050455,-0.00378413,-0.00120293,-0.002329,0.00147079,0.00480498,-0.00180176,-0.00537364,0.00034603,-0.00548296,-0.01016689,0.00113817,0.00379708,-0.00686295,-2.748e-05,-0.00415476
CAL-013,0,m5_demo,1,calibration,-0.00278977,0.0002342,-0.00151627,-0.00444299,0.00773617,-6.667e-05,0.00665775,0.00670845,0.03221408,0.07417601,0.14445654,0.24602534,0.35123255,0.39929645,0.37034164,0.2866536,0.17975247,0.10049958,0.04504814,0.01706426,0.0331673,0.06597439,0.15667263,0.27581845,0.35376962,0.35462869,0.26542597,0.15290614,0.05863549,0.02691685,0.00829108,0.00064649,-0.00159888,0.00253056,-0.00422157,-0.00337171,-0.00078335,0.00205306,-0.00539755,-0.00843028,-0.00496567,0.00127366,0.00252737,0.00446459,0.00435363,-0.00942643,0.00792131,-0.00478015,0.00559113,0.00041747
CAL-014,0,m5_demo,1,calibration,-0.0015275,0.00218751,-0.00295334,-0.00774436,-0.00099638,-0.0040283,0.00425876,0.00769603,0.02293297,0.05633044,0.11847383,0.21221017,0.28907558,0.33477067,0.315107,0.2389694,0.15301914,0.08563043,0.03195386,0.01564656,0.02201521,0.0523736,0.10517816,0.18457175,0.25079722,0.24736608,0.19218666,0.1104293,0.03925368,0.01405441,0.00037179,0.00054444,0.00455676,0.00174786,0.00359238,1.116e-05,-0.01076405,-0.01182784,-0.00127078,0.00297673,-0.00123435,0.00068404,-0.00603153,0.00355075,-0.00993847,-0.00539668,-0.00618511,0.0042259,-0.00560183,0.00483822
CAL-015,0,m5_demo,1,calibration,-0.00076443,-0.00608837,-0.00418993,0.00221792,-0.00357539,0.00487485,-0.00055162,0.01494629,0.02337034,0.06989331,0.13606864,0.22959865,0.32142084,0.37035324,0.34626092,0.26494139,0.17027079,0.0901919,0.03693974,0.02069709,0.01550175,0.04286271,0.11039243,0.18972383,0.260805,0.26202945,0.18634317,0.11065438,0.04626434,0.01818383,0.00299206,0.00399844,-0.00010678,0.00284969,-0.00394637,0.00579673,-0.0063786,-0.00582739,0.00130833,0.00385776,0.00108363,-0.01066086,-0.00554615,0.00098231,0.00026092,0.0004965,-0.00117548,-0.00106792,0.00055266,0.00503786
CAL-016,0,m5_demo,1,calibration,0.00271199,-0.00203001,0.00241222,0.00112496,-0.00383607,0.00412528,0.00287251,0.0075972,0.02885784,0.05674424,0.14742735,0.23746763,0.33057932,0.37659523,0.35952968,0.28032109,0.1812072,0.09006533,0.04842165,0.01627239,0.02089445,0.06449228,0.12345122,0.20306686,0.28030003,0.28201623,0.20935314,0.12131024,0.05704437,0.01415795,0.00211854,0.00361226,-0.00142282,-0.0008847,0.00178971,0.0042722,0.00426693,0.00844492,0.0035712,0.00176392,-0.00855038,-6.15e-06,-0.0026227,-0.00649923,9.257e-05,0.00183295,-0.00250779,0.00283295,-0.00449439,-0.00281062
CAL-017,0,m5_demo,1,calibration,0.00442056,0.00223832,0.00193778,-0.00135515,0.0030101,-0.0043264,0.00254255,0.01057679,0.03006656,0.083042,0.16483747,0.2723814,0.37817939,0.43390303,0.41111155,0.31708841,0.20205144,0.10713491,0.05050127,0.0220356,0.01816545,0.05095101,0.10718058,0.19937129,0.26693008,0.2583903,0.19844511,0.11582976,0.04537938,0.01885218,0.00910999,-0.00034848,0.00367295,0.00216752,-0.00378901,0.00340448,0.00753917,0.00096927,0.00195689,-0.00340761,-0.00381021,-0.00183311,0.00123594,0.0016712,0.00172741,0.00368889,0.0023287,-0.00055804,0.00332447,-0.0028159
CAL-018,0,m5_demo,1,calibration,0.00749248,-0.00291754,0.00016872,-0.00381561,0.00582068,0.00285282,-0.00028258,0.00730335,0.03572632,0.07423916,0.16531756,0.27428227,0.38874023,0.44797786,0.41758335,0.3254511,0.20744043,0.10456722,0.0454024,0.01703405,0.01452144,0.03503728,0.10964719,0.18014098,0.24315684,0.24312892,0.17937721,0.10547626,0.04695429,0.01297464,0.00103561,0.00231551,0.00062286,-0.00206976,0.0035937,-0.00526564,0.00412517,0.00042514,-0.00458928,0.00166609,-0.00303543,0.00015307,-0.00062754,-0.00279601,0.00560655,-0.00032451,-0.00080428,0.00352304,0.00197528,-0.00016115
CAL-019,0,m5_demo,1,calibration,0.00091009,-0.00496864,0.00534759,-0.00622959,0.00618041,0.00284064,0.00058626,0.00539831,0.01794649,0.05664892,0.13511057,0.2221086,0.30821901,0.35171456,0.32286971,0.25750139,0.15877439,0.07768601,0.03462862,0.01443129,0.02367295,0.04238956,0.10320508,0.18790757,0.25407325,0.24115501,0.18866513,0.1113345,0.05310655,0.01665512,0.00681275,-0.00045686,-0.00091978,3.555e-05,-0.00378506,0.00046933,0.00093872,0.00391788,0.00715319,0.00135821,0.0017631,0.00152082,-0.00721147,0.00191431,0.00013742,-0.00353879,0.0016392,-0.00495315,0.00326247,-0.00244929
CAL-020,0,m5_demo,1,calibration,-0.00435468,-0.00143007,-0.0004369,-0.00492646,-0.00801036,0.00575095,-0.00262386,0.00666247,0.02688068,0.06559753,0.14407163,0.24841063,0.35241657,0.39928902,0.37449423,0.28227663,0.18657224,0.09390417,0.04959845,0.01540125,0.01644838,0.05440331,0.12679519,0.21014262,0.27844859,0.2842588,0.20870872,0.12026507,0.04784537,0.01376599,0.00547655,0.00313178,0.00195791,-0.00470816,0.00021755,0.00097249,-0.00473703,-0.00178566,0.00243893,0.00409003,0.00483171,-0.00398287,-0.00321146,-0.00034944,-0.00233648,-0.00463879,-0.00079521,0.00015697,0.00406641,8.64e-06
CAL-021,0,m5_demo,1,calibration,-0.00473289,0.00455376,-0.00311871,0.0039269,-0.00012144,-0.00197324,0.01024035,0.00768288,0.01944011,0.06767059,0.14439503,0.25249503,0.34523102,0.39403586,0.37046849,0.28380478,0.18365623,0.09822311,0.04437024,0.01943229,0.01829367,0.06299574,0.15111146,0.26368223,0.35349864,0.35311754,0.2666673,0.15667169,0.06781182,0.02181179,0.01104323,-0.00136591,0.00371607,-0.00736936,-0.00039245,0.00393079,0.00735609,-0.0078923,-0.00487782,-0.00091799,-0.00521345,-0.00187592,0.00014437,0.0057881,-0.00356896,0.00238384,-0.00620203,0.00462433,0.00182354,0.00223841
CAL-022,0,m5_demo,1,calibration,-0.00136594,0.00484112,0.00285305,-0.00688605,0.00543405,0.00875776,0.00469891,0.00854286,0.02918314,0.07932786,0.14340165,0.23886988,0.34645053,0.3924275,0.3670799,0.28643613,0.17876884,0.10096234,0.04367509,0.01703486,0.01623263,0.05422096,0.13302819,0.2431143,0.31983357,0.31713614,0.24183258,0.14116785,0.05930537,0.02401076,0.0007071,0.00020499,0.00351316,-0.00144319,-0.00058317,-0.0091996,0.00199281,0.0020519,-0.00249237,-0.00198793,0.00051876,0.00396922,-0.00895686,-0.01046575,0.00262494,0.00617154,0.00024864,-0.00160734,-0.00072772,-5.321e-05
CAL-023,0,m5_demo,1,calibration,0.00044139,0.0064338,-0.00618741,0.0029481,0.00110952,0.00266436,-0.00588253,0.01398062,0.02276516,0.07544817,0.16194893,0.27771312,0.3896232,0.44203854,0.42244383,0.32660923,0.21002288,0.1098133,0.04623566,0.02711694,0.02639317,0.06945638,0.16407434,0.28884719,0.3803563,0.37580245,0.28794691,0.16332484,0.06240112,0.02255609,0.00467547,-0.00101263,-0.00180123,-0.00258545,-0.00218403,3.816e-05,0.0044067,-0.00301566,0.00826658,-0.0040253,0.00649452,-0.00266351,-0.00033508,-0.00194512,0.00723882,-0.00149772,0.00144433,-0.00529277,-0.00120704,-0.00435992
CAL-024,0,m5_demo,1,calibration,-0.00022021,-0.00807917,-0.00725366,0.00568115,-0.00082833,-0.0033133,0.00648759,0.00612368,0.03053746,0.06848972,0.13822175,0.24167897,0.33236936,0.37965033,0.35189001,0.28954975,0.17173743,0.08951867,0.03269109,0.010361,0.02300831,0.05465293,0.10558275,0.19275705,0.25803387,0.25713619,0.19504,0.11469918,0.040503,0.01755534,0.00863268,0.00457135,0.00033393,0.0019512,0.001923,0.00544185,-0.00079631,-0.00057622,-0.00727538,-4.67e-06,-0.00172264,0.00491226,-0.00281187,0.00177572,0.00101087,-0.00142791,0.00385981,0.00019215,0.00228343,-0.00069728
CAL-025,0,m5_demo,1,calibration,-0.00123594,-0.00712508,0.00139312,-0.00076041,-0.00750272,-0.00086157,0.00034206,0.0127432,0.02675506,0.07204554,0.1451628,0.24297832,0.34515279,0.39341461,0.36683091,0.28430098,0.17864344,0.09461168,0.04414223,0.02937588,0.02275946,0.05685807,0.12977869,0.21891438,0.29127708,0.29978399,0.22563965,0.13108075,0.05529921,0.01125499,0.00462911,-0.00740805,0.00220506,0.00111973,-0.00326296,-0.00346666,-0.00363769,-0.00359326,0.00140374,0.00201045,-0.00297089,0.00055107,0.00567992,-0.00162065,0.0010321,-0.00375828,-0.00802675,-0.00159198,-0.00281568,0.00503006
CAL-026,0,m5_demo,1,calibration,-0.00322549,-0.00044548,-0.00048757,-0.00108554,0.00831052,0.0004637,-0.00223417,0.00340825,0.02039356,0.05429241,0.12063862,0.20095143,0.28071087,0.31984052,0.30825809,0.23630446,0.14816267,0.0819161,0.03725252,0.02126675,0.01772394,0.06186875,0.13178919,0.22678501,0.30280299,0.3023779,0.22227737,0.12339396,0.04815245,0.01425193,0.00067589,0.00322883,-0.00186373,-0.0033379,0.00230718,0.00149735,0.00150436,0.00562998,-0.00142758,-0.00112289,0.0076583,0.00530213,-0.00717179,-0.00039293,0.00128463,0.00844982,-4.245e-05,0.00212166,-0.00940769,0.00017913
CAL-027,0,m5_demo,1,calibration,-0.00344203,0.0076802,-0.00550076,5.094e-05,-0.0085819,0.00353713,-0.00132052,0.01188968,0.0327868,0.06934692,0.15574154,0.26066453,0.356837,0.40126483,0.38253819,0.29391757,0.18755358,0.09895492,0.04228054,0.02062557,0.02079095,0.06387735,0.12297886,0.23080861,0.30988189,0.30409095,0.23169883,0.13190406,0.05111273,0.01599812,0.00436973,-0.00651707,-0.00013632,0.00552215,-8.94e-06,0.00603186,0.00455884,-0.00085728,0.0020604,0.00412764,-0.00243196,0.00828808,0.00279163,0.00102368,-8.722e-05,-0.00076397,-0.00180257,-0.00052555,0.00165755,-0.00253741
CAL-028,0,m5_demo,1,calibration,-0.00452171,0.0061238,0.00636365,-0.00039409,-0.00078077,-0.00389231,0.0021548,-0.00069682,0.02855818,0.06700328,0.12534839,0.22128744,0.30071491,0.35069556,0.32883916,0.25723348,0.16874795,0.08994037,0.03817071,0.02009505,0.01576954,0.0577681,0.13603362,0.23884976,0.31836038,0.31964757,0.23950346,0.13812357,0.05626208,0.01171937,0.00241762,0.00296849,0.00333781,-0.00753089,-0.00570385,-0.00247068,-0.00093923,0.00792704,-0.00183184,0.0061904,0.00527057,-0.00357957,0.00350991,0.00261316,-0.00344341,-0.00097371,-0.00218122,-0.00124104,-0.00241944,-0.00477081
CAL-029,0,m5_demo,1,calibration,-0.00083819,-0.00249308,0.00119577,-0.00249772,0.00328154,-0.00169343,0.00397926,0.0101702,0.0420805,0.08859693,0.18379515,0.30650941,0.43609662,0.48326915,0.45554539,0.35456647,0.22225099,0.11657121,0.04967933,0.02142949,0.01274309,0.04513136,0.09883802,0.17100381,0.22731694,0.22016984,0.1530944,0.09798063,0.04694698,0.01226223,0.00622411,0.00073194,0.00551964,-0.00048246,0.00152961,0.00980397,-0.00885249,-0.00099548,0.00599341,-0.00659184,0.00039714,0.00359354,-0.00688708,-0.00120016,0.00364847,0.00251801,0.00594474,-0.0068521,0.0027655,-0.00182614
CAL-030,0,m5_demo,1,calibration,-0.00242098,0.00579022,-0.0069489,0.00136911,0.00470169,-0.00500989,0.00558113,0.0078617,0.02181775,0.07701289,0.1429968,0.26021729,0.35171907,0.4048716,0.3838375,0.29918103,0.19232294,0.09557299,0.03751035,0.01875103,0.02248794,0.0739511,0.15000397,0.2625901,0.34905218,0.34081472,0.25950903,0.14329147,0.07198514,0.01821543,0.00606234,0.00379982,0.00137612,0.00146442,0.00289012,0.00304026,-0.00185953,0.00216758,0.00608399,-0.00161728,0.00885243,-0.00421413,-0.00110379,0.00338903,0.00050247,0.00632285,0.00422934,-0.00011577,0.00187822,-0.00047383
CAL-031,0,m5_demo,1,calibration,-0.00484505,0.00764675,0.00023248,0.00214681,-0.01091594,-0.00328799,0.00056698,0.00794581,0.02990176,0.07161994,0.14524643,0.2576515,0.35877547,0.40649375,0.38292222,0.29809979,0.1928612,0.09543809,0.0494963,0.01623362,0.02137764,0.0556497,0.12768499,0.21464997,0.28977104,0.28579706,0.22416812,0.12398467,0.05941813,0.02146802,0.00576733,-0.00013931,0.00154702,-0.00135539,0.00141095,0.00437784,-0.00424124,0.00446488,0.00057961,0.00089578,0.00266136,-0.00056617,-0.00059608,0.00078528,-0.00754074,-0.00334207,-0.00081119,0.00375094,-0.00334972,0.00255429
CAL-032,0,m5_demo,1,calibration,0.0004997,0.00184647,0.00017312,0.00124412,0.00113509,-0.00273569,0.01557824,0.0087705,0.02990316,0.07337744,0.15017839,0.24695293,0.34282592,0.39879073,0.37846457,0.29840926,0.19474119,0.10245299,0.05128138,0.02312192,0.01783598,0.04795351,0.11777772,0.20224294,0.2695732,0.2652388,0.20636542,0.12174147,0.03986371,0.01822546,0.00284863,-0.00422878,0.00074349,-0.00355463,-0.00035086,0.00010834,0.00282966,0.00369396,0.00519368,-0.00088087,-0.00183917,-0.00602709,-0.00202032,-0.0023515,0.00423221,0.00034135,0.0056501,0.0011718,-0.00231648,0.00439954
CAL-033,0,m5_demo,1,calibration,-0.00014002,-0.00242728,0.00098191,-0.00110055,0.00058571,0.00408802,0.00668415,0.00774577,0.02007796,0.05542056,0.13050735,0.21167241,0.2950268,0.33641489,0.31779403,0.2450136,0.16292606,0.08850175,0.03934764,0.01413435,0.01936199,0.05473174,0.12493463,0.2281059,0.29540959,0.29876342,0.22594682,0.12878728,0.04827069,0.02034169,-0.00415646,0.0002995,0.00055697,0.00064467,0.00207523,0.00558582,-0.00724335,-0.00396217,-0.00958278,-0.00357859,0.00259119,-0.00167761,0.00155285,0.0014172,0.00943986,-0.00092548,-0.00197469,-0.00145852,0.00409055,-0.00426146
CAL-034,0,m5_demo,1,calibration,-0.00108136,0.00385743,-0.00103143,-0.00209097,0.00049451,-0.00087595,0.00398152,0.00628005,0.03311565,0.06180334,0.13108264,0.23216604,0.33223661,0.37534981,0.36141294,0.27049099,0.17676068,0.09153488,0.04598464,0.01828969,0.01980327,0.04891411,0.11861405,0.19997172,0.27156553,0.2673016,0.19829182,0.11607395,0.04835451,0.01437808,0.00839737,0.00205123,-0.00311394,0.00619179,-0.0016915,-0.00519258,0.00196666,8.057e-05,-0.00484954,-0.00409669,-0.00451299,0.00551912,0.00393267,-0.00294442,1.27e-05,0.00277439,0.00274228,0.00474791,0.00983171,-0.00358916
CAL-035,0,m5_demo,1,calibration,-0.00081501,-0.00297381,0.00530822,0.00048885,0.00422262,-9.908e-05,0.00272913,0.01521339,0.02897063,0.07326136,0.14472406,0.2604342,0.35029036,0.40542878,0.37961947,0.29396826,0.18332328,0.09557315,0.03567183,0.02024882,0.02422018,0.03906165,0.10181571,0.18355258,0.24306263,0.23973776,0.17713676,0.09634536,0.0379253,0.02288841,0.00112181,-0.00787285,-0.00250692,0.00271464,0.00159722,-0.00593537,0.00041072,-0.00511719,-0.00485263,0.00278137,0.00436893,-0.00915373,0.00271976,0.00073662,0.00163894,-0.00049307,0.00144082,-0.00072852,0.00138282,-0.00148336
CAL-036,0,m5_demo,1,calibration,0.00352718,-0.00606722,0.00287436,-0.00343636,-0.00776406,-0.00391568,8.238e-05,0.01439028,0.03780482,0.08063597,0.16035636,0.28138904,0.39852999,0.43366246,0.4235607,0.32522158,0.20100899,0.10725604,0.04381975,0.02003776,0.02027854,0.05309327,0.11533692,0.20241916,0.27555477,0.27529888,0.20980999,0.12584491,0.04863067,0.02039813,0.00244149,-0.003371,0.0049628,-0.00750785,0.00354056,-0.00297236,0.00010036,8.726e-05,0.00050498,-0.00708537,-0.0054834,-0.00551462,-0.00838333,-0.00027662,-0.0016737,-0.0013985,0.00032387,-0.00058132,0.00575067,0.00207724
CAL-037,0,m5_demo,1,calibration,-0.00588321,0.00195586,-0.00485887,0.0036281,-0.00480393,-0.00264331,0.00107476,0.0116187,0.02778619,0.07554336,0.16241771,0.25418034,0.36202506,0.4030549,0.38460875,0.29585222,0.18532706,0.10449744,0.04244675,0.01719038,0.01085132,0.03497453,0.08683099,0.15991164,0.21602967,0.21315552,0.16082824,0.09399168,0.03639127,0.01287349,0.0018247,6.415e-05,0.00298244,0.0031661,-0.00094748,0.00469312,0.00648991,-0.0037984,0.0020537,0.00201627,-0.00461072,0.00197971,0.00196521,-0.00828176,0.00336905,-0.0059036,-0.00356063,-0.00764527,-0.00133751,-0.00189931
CAL-038,0,m5_demo,1,calibration,0.00348897,-0.00182492,-0.00550414,0.00568467,-0.00061547,-0.00073086,0.00578746,0.00390808,0.01668041,0.06092348,0.12222519,0.20380603,0.28896318,0.3354717,0.31846771,0.23375884,0.15492064,0.08181374,0.03039248,0.01303299,0.01993529,0.0467494,0.11197902,0.18549387,0.2507266,0.23974291,0.18621808,0.10384146,0.03876689,0.01759669,0.00961259,0.00171982,-0.00706257,0.00035364,-0.0012656,-0.0056499,0.00341721,0.00519363,0.00055351,0.00359436,0.00425448,0.00300318,0.00567347,0.00409263,0.00265051,0.00661854,-0.00329352,0.00802327,-0.0024818,0.00025261
CAL-039,0,m5_demo,1,calibration,-0.00092477,0.00264984,0.00486486,0.00297765,-0.00074309,-0.00377476,0.01169697,0.00378044,0.02755111,0.07267305,0.14146131,0.24881388,0.33329334,0.38998234,0.35632144,0.29064603,0.19069161,0.09054262,0.0404104,0.02190525,0.0229583,0.0657694,0.13990867,0.23488108,0.31420995,0.31464371,0.23780283,0.13654269,0.05822601,0.02232379,0.00852419,0.00481221,-0.00690473,-0.00010371,0.00101498,-0.0020533,0.00733762,0.00669441,0.00197109,-0.00233938,-0.00219756,0.00188289,-0.00176783,0.00036312,0.00107712,0.00195274,0.00901952,-0.00544106,-0.00600233,-0.0003836
NEW-000,0,m5_demo,1,new_normal,0.00097801,-0.00092814,0.00068095,-0.00272123,-0.00255701,-0.00782105,-0.00040637,0.00610587,0.02695603,0.06386259,0.14197115,0.23395541,0.3246822,0.37999656,0.35583896,0.27363937,0.17002235,0.09326746,0.04971671,0.02023217,0.01699308,0.04191756,0.10000553,0.17816536,0.24153717,0.2344623,0.17791265,0.10507996,0.04804097,0.01047476,0.00789827,0.00523262,0.00202667,-0.00228763,-0.00201485,0.00338641,-0.00377308,-0.00269979,0.00138539,0.00415785,0.00210452,0.00744053,0.00215281,0.00283824,-0.00174995,0.00712321,-0.00479761,0.00103143,1.484e-05,0.00634421
NEW-001,0,m5_demo,1,new_normal,0.00029201,0.0029634,-0.00026536,-0.00913333,-0.00260238,-0.00118956,0.00124629,0.01272362,0.02164046,0.05493225,0.11215882,0.19255932,0.26595493,0.3162592,0.29596241,0.22453625,0.13561951,0.07769731,0.03074153,0.01447736,0.02349625,0.05685218,0.13698858,0.23550102,0.31720342,0.3202215,0.23968056,0.13332198,0.05607708,0.01221406,0.0028638,-0.00332931,-1.695e-05,-0.00511212,0.00257759,0.00467195,0.00918399,0.00199786,0.00461982,0.00263915,-0.00182659,-0.00111709,0.00253607,-0.00226648,-0.00176745,-0.00321652,-0.00345565,0.00362053,-0.00486932,-0.00707677
NEW-002,0,m5_demo,1,new_normal,-0.00316865,0.00893584,-0.00260511,-0.00291725,0.01041827,0.003281,0.00224781,0.01033339,0.01842991,0.05651572,0.12211726,0.19917459,0.29458648,0.325721,0.31051422,0.23878892,0.16453435,0.08342381,0.03873683,0.01399256,0.01701553,0.07326621,0.14919873,0.26824684,0.35402103,0.36136823,0.26425155,0.14903396,0.06511553,0.01626256,0.00803772,-0.0016553,-0.00862596,-0.00494673,0.00368752,-0.00088602,-0.00095213,-0.00847724,-0.00023092,-0.00011786,-0.00572915,0.00499161,-0.00303553,0.00317776,0.00485106,0.00400414,0.00085707,0.00488759,-0.00177011,-0.00348527
NEW-003,0,m5_demo,1,new_normal,0.00091537,-0.00234172,-0.00744743,0.00053604,-0.0029591,0.00564362,0.00028216,0.01144322,0.03134288,0.05636077,0.13233563,0.23207323,0.31565954,0.36253686,0.33801352,0.26527475,0.16705595,0.08800619,0.04371931,0.01009476,0.0261641,0.0743414,0.13416026,0.23519169,0.31004165,0.31239732,0.24224558,0.13392013,0.05878025,0.0100244,0.00484899,-0.00021376,0.00050326,0.0008284,0.00082019,-0.00403606,-5.42e-05,-0.00385491,-0.00033874,0.00452531,-0.00278854,-0.00434272,0.00567179,0.00674121,-0.00137965,0.00297479,0.00580563,0.00037088,0.00634601,-0.00068341
NEW-004,0,m5_demo,1,new_normal,-0.0037857,-0.00056747,-0.00446605,-0.00272062,-0.003443,0.00567656,-0.0017992,0.01229281,0.02710551,0.07329285,0.15061048,0.25383989,0.36778857,0.4136062,0.38448072,0.29457899,0.18718974,0.10053643,0.04242321,0.0190845,0.01450599,0.04523305,0.10719297,0.18569545,0.25051832,0.24894669,0.19183542,0.1076578,0.05142315,0.02232174,0.00316138,-0.00330778,-0.0007314,0.0044451,0.0024125,-0.00219125,-0.00208505,-0.00095028,-0.00376788,-0.00416788,-0.00466922,-0.00119144,-0.00440887,0.00172565,0.00398661,0.00309168,-0.00268695,0.00734924,-0.00151064,0.00447783
LEV-000,0,m5_demo,1,leverage_outlier,-0.0016157,-0.00017669,0.00274805,5.79e-05,5.942e-05,0.00178322,0.00146826,0.01619394,0.04281597,0.10897074,0.22777874,0.38225478,0.52979382,0.6068914,0.57180492,0.4426309,0.2827351,0.14953189,0.06268814,0.02459623,0.01591188,0.02933503,0.06037875,0.10534344,0.14339443,0.14520691,0.10858442,0.06094385,0.02460202,0.01073132,3.203e-05,0.00090283,0.00201889,0.0003086,0.00023357,0.00161192,-0.00227204,-0.00090738,-0.00110269,0.00162628,0.00133367,-0.000531,0.00292644,0.00148162,-0.0001999,-0.00031134,0.00148922,0.00124673,0.00013242,0.00024934
XINST-mp5_demo,0,mp5_demo,1,cross_instrument,0.00756518,0.00354706,-0.0025885,-0.00868567,-0.00600236,-0.00382381,0.00089567,0.01610576,0.02705143,0.07656731,0.15807304,0.2761576,0.3743413,0.42669274,0.4026103,0.3114931,0.2027389,0.10753531,0.0481056,0.02342781,0.02499524,0.0545766,0.13168135,0.22778358,0.29497384,0.29048523,0.22188822,0.12600147,0.05079068,0.02142693,0.05146401,0.0994815,0.12001966,0.0985758,0.04498234,0.01748315,0.00801102,0.00015569,-0.00436534,0.00454469,-0.00380079,-0.00093344,-0.00672392,0.00094751,0.00216914,7.555e-05,-0.00484822,-0.00276948,0.00129838,-0.00056893
XINST-mp6_demo,0,mp6_demo,1,cross_instrument,0.01486966,0.00081044,0.00563426,-0.00064409,0.00536711,0.00987363,0.00451892,0.0070395,0.02913347,0.07204745,0.14180266,0.24436079,0.33370905,0.39177956,0.36926438,0.28087871,0.18160968,0.10285448,0.03827348,0.02434392,0.02555121,0.07028517,0.15186433,0.28503389,0.35723658,0.36737669,0.26782735,0.14860727,0.07007579,0.03611189,0.04954342,0.09120807,0.11670366,0.10383808,0.0524558,0.01906816,0.00697085,0.00367825,0.00665399,0.00319377,-0.0095125,-0.00211309,0.00420286,0.00790977,0.00032522,0.00112327,-0.00137805,0.00426724,0.00703564,0.00028081
"""

In [ ]:
# So load_spectra() with NO argument still works in this notebook too
# (falls back to the Part 1 VIP demo dataset, matching the shared
# load_spectra(source=None) contract used by every other notebook):
SAMPLE_CSV = SAMPLE_CSV_VIP

> 三組資料使用三個不同的波長/波數軸（VIP 示範是 NIR 900–1700 nm、六油品是 900–2000 nm、離群值示範是 1100–2400 nm），依照共用光譜格式規範「一個檔案只用一種共同 x 軸」的規則，本筆記本分成三個 CSV 檔存放，而不是像 NB11 一樣只有一個。三個小節各自明確傳入對應的 `SAMPLE_CSV_*` 給 `load_spectra()`；若不小心呼叫 `load_spectra()`（不給參數），則會退回 Part 1 的 VIP 示範資料。

## 2. Part 1 — VIP（變數重要性投影）

### 2.1 建立一個 NB11 風格的 PLS 迴歸模型

先用交叉驗證（這裡每一列都是獨立樣品，沒有重複測量，所以用一般的 `KFold` 就足夠，不需要像 NB11 那樣以 `sample_id` 分組）挑選 LV 數，再用選定的 LV 數配適最終模型，接下來要用這個模型算 VIP。

In [ ]:
df_vip = load_spectra(io.StringIO(SAMPLE_CSV_VIP))
meta_vip, X_vip, wl_vip = split_spectra(df_vip)
y_vip = meta_vip["acidity_pct"].to_numpy()

X_tr, X_te, y_tr, y_te = train_test_split(X_vip, y_vip, test_size=0.25, random_state=14)


def cv_rmse(X, y, ncomp, k=5, seed=14):
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    errs = []
    for tr, va in kf.split(X):
        m = PLSRegression(n_components=ncomp, scale=False)
        m.fit(X[tr], y[tr])
        errs.append((m.predict(X[va]).ravel() - y[va]) ** 2)
    return float(np.sqrt(np.mean(np.concatenate(errs))))


cv_scores = {nc: cv_rmse(X_tr, y_tr, nc) for nc in range(1, 9)}
ncomp_full = min(cv_scores, key=cv_scores.get)
print("== 全波段 CV RMSE by ncomp ==")
for nc, v in cv_scores.items():
    print(f"  ncomp={nc}  RMSECV={v:.4f}")
print("選定 ncomp_full =", ncomp_full)

pls_full = PLSRegression(n_components=ncomp_full, scale=False).fit(X_tr, y_tr)
RMSECV_full = cv_scores[ncomp_full]
RMSEP_full = float(np.sqrt(np.mean((pls_full.predict(X_te).ravel() - y_te) ** 2)))
print(f"RMSECV={RMSECV_full:.4f}  RMSEP={RMSEP_full:.4f}")

### 2.2 VIP 公式與三個檢查

$$\mathrm{VIP}_j = \sqrt{\; p \cdot \dfrac{\sum_a \mathrm{SSY}_a \left(w_{ja}/\lVert w_a \rVert\right)^2}{\sum_a \mathrm{SSY}_a} \;}$$

其中 $j$ 是波長、$a$ 是第 $a$ 個潛在變數（LV）、$p$ 是波長總數、$\mathrm{SSY}_a$ 是第 $a$ 個 LV 解釋掉的 Y 變異（用 `y_loadings_` 與 `x_scores_` 算出）、$w_{ja}$ 來自 `x_weights_`。

**三個檢查**（任何時候算完 VIP，都應該先看這三個，再往下用）：

1. **VIP 不應該全部相同** —— 228 或 81 個波長理應有高有低。
2. **mean(VIP²) 應該非常接近 1**（在浮點數精度內，容許誤差 1e-9）—— 這是 VIP 定義本身保證的數學恆等式，不成立就代表公式或係數用錯了。
3. **高 VIP 的波段應該對得上已知的化學吸收** —— 對不上時，應該先懷疑計算，而不是照單全收。

In [ ]:
def compute_vip(pls):
    """Correct VIP: for each latent variable a, weight that LV's contribution
    to wavelength j (w_ja / ||w_a||)^2 by how much Y-variance that LV
    explains (SSY_a), sum over LVs, then take sqrt(p * ... / total SSY)."""
    T = pls.x_scores_      # (n, ncomp)
    W = pls.x_weights_     # (p, ncomp)
    Q = pls.y_loadings_    # (1, ncomp)
    p, ncomp = W.shape
    SSY = np.array([(Q[0, a] ** 2) * np.sum(T[:, a] ** 2) for a in range(ncomp)])
    Wn = W / np.linalg.norm(W, axis=0, keepdims=True)
    raw = (Wn ** 2) @ SSY          # shape (p,): EACH wavelength's own contribution
    vip = np.zeros(p)
    vip[:] = np.sqrt(p * raw / SSY.sum())
    return vip


VIP = compute_vip(pls_full)
mean_vip2 = float(np.mean(VIP ** 2))
n_distinct = len(set(np.round(VIP, 6)))

print("VIP 前 10 個:", np.round(VIP[:10], 3))
print(f"mean(VIP^2) = {mean_vip2:.12f}   (應該非常接近 1)")
print(f"相異 VIP 值的個數 = {n_distinct} / {len(VIP)}  (應該遠大於 1)")

assert abs(mean_vip2 - 1.0) < 1e-9, "檢查 1 失敗：mean(VIP^2) 應該 = 1，公式或係數有誤"
assert n_distinct > len(VIP) // 2, "檢查 2 失敗：VIP 不該幾乎全部相同"
peak_wl = wl_vip[int(np.argmax(VIP))]
print(f"VIP 全域最大值落在 {peak_wl:.0f} nm")
assert abs(peak_wl - 1200) < 60 or abs(peak_wl - 1450) < 60, \
    "檢查 3 失敗：VIP 最高的波段應該落在已知的醋酸(1200nm)或水(1450nm)吸收帶附近"
print("三個檢查全部通過。")

### 2.3 一個真實發生過的 bug

準備這門課的教材時，有一支計算 VIP 的腳本裡，在 `compute_vip` 這樣的程式碼中，**多寫了一個 `.sum()`**：本來 `raw = (Wn ** 2) @ SSY` 算出來的是「每個波長各自的貢獻」（長度 p 的向量），結果有人在後面手滑多接了一個 `.sum()`，把整個向量提早壓成一個總和的純量——這個數字跟波長 $j$ 完全無關，於是每個波長都拿到一模一樣的數字。腳本印出來的「VIP 前 20 名波長」其實完全沒有意義。跟第 2.2 節的正確版本相比，**下面這支函式只多了一行 `raw = raw.sum()`**，其餘完全相同。

下面的 cell 刻意重現這個 bug，並讓它去跑一次第 2.2 節的三個檢查，看它是怎麼被抓到的（用 `try/except` 包起來，讓筆記本可以繼續往下執行，不會因為示範用的錯誤而中斷）。

In [ ]:
def compute_vip_buggy(pls):
    """The bug that actually shipped once: one extra line, `raw = raw.sum()`,
    silently collapses the per-wavelength contribution vector (shape (p,))
    into a single grand total BEFORE it gets broadcast back out -- so every
    wavelength ends up with the exact same number. Everything else here is
    byte-for-byte identical to compute_vip() above."""
    T = pls.x_scores_
    W = pls.x_weights_
    Q = pls.y_loadings_
    p, ncomp = W.shape
    SSY = np.array([(Q[0, a] ** 2) * np.sum(T[:, a] ** 2) for a in range(ncomp)])
    Wn = W / np.linalg.norm(W, axis=0, keepdims=True)
    raw = (Wn ** 2) @ SSY
    raw = raw.sum()               # <-- the ONE extra line that shipped
    vip = np.zeros(p)
    vip[:] = np.sqrt(p * raw / SSY.sum())
    return vip


VIP_buggy = compute_vip_buggy(pls_full)
print("Buggy VIP 前 10 個:", np.round(VIP_buggy[:10], 3), " (應該全部一樣)")

mean_vip2_buggy = float(np.mean(VIP_buggy ** 2))
n_distinct_buggy = len(set(np.round(VIP_buggy, 6)))
print(f"buggy mean(VIP^2) = {mean_vip2_buggy:.6f}   (不再等於 1！)")
print(f"buggy 相異 VIP 值個數 = {n_distinct_buggy} / {len(VIP_buggy)}")

print("\n用第 2.2 節同樣的三個檢查，去檢驗這個錯誤版本：")
try:
    assert n_distinct_buggy > len(VIP_buggy) // 2, "檢查 2 (VIP 不應全部相同) 失敗！"
    print("檢查 2 通過（不應該發生）")
except AssertionError as e:
    print(f"檢查 2 失敗，如預期： {e}")

try:
    assert abs(mean_vip2_buggy - 1.0) < 1e-9, "檢查 1 (mean(VIP^2)=1) 失敗！"
    print("檢查 1 通過（不應該發生）")
except AssertionError as e:
    print(f"檢查 1 失敗，如預期： {e}")

print("\n結論：只要養成習慣，每次算完 VIP 都跑一次這三個檢查，"
      "這種『迴圈變數沒被真正用到』的 bug 幾乎不可能被忽略地上線。")

### 2.4 VIP 與迴歸係數的方向

VIP 只講「重不重要」，不講「正相關還是負相關」。下面把 VIP 與迴歸係數畫在一起：可以看到 1200 nm（醋酸 C–H，示意）與 1450 nm（水 O–H，示意）附近 VIP 都偏高，但兩者迴歸係數方向相反。

In [ ]:
coef_full = np.ravel(pls_full.coef_)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(wl_vip, VIP, color="tab:blue")
axes[0].axhline(1.0, color="tab:red", linestyle="--", linewidth=1, label="VIP = 1")
axes[0].set_ylabel("VIP")
axes[0].legend()
axes[0].set_title("VIP vs wavelength")

axes[1].plot(wl_vip, coef_full, color="tab:green")
axes[1].axhline(0, color="grey", linewidth=1)
axes[1].set_xlabel("Wavelength (nm)")
axes[1].set_ylabel("PLS regression coefficient")
axes[1].set_title("Regression coefficient (sign = direction) vs wavelength")
plt.tight_layout()
plt.show()

idxA = int(np.argmin(np.abs(wl_vip - 1200)))
idxB = int(np.argmin(np.abs(wl_vip - 1450)))
print(f"1200nm 附近：VIP={VIP[idxA]:.2f}  係數={coef_full[idxA]:+.4f}")
print(f"1450nm 附近：VIP={VIP[idxB]:.2f}  係數={coef_full[idxB]:+.4f}")
print("兩個波段的 VIP 可能都偏高（都重要），但係數正負號通常相反（方向不同）。")

### 2.5 VIP > 1 篩選重建模型（誠實比較）

用 VIP > 1 篩掉「不重要」的波長，重新選 LV、重新配適，然後用**篩選時完全沒用過**的測試集比較 RMSECV／RMSEP。篩選不是穩賺不賠——有時誤差會下降（雜訊被濾掉），有時反而持平或變差（丟掉了有用的共線資訊）。

In [ ]:
sel_mask = VIP > 1.0
n_sel = int(sel_mask.sum())
print(f"VIP>1 的波長數 = {n_sel} / {len(VIP)}")

cv_scores_sel = {nc: cv_rmse(X_tr[:, sel_mask], y_tr, nc) for nc in range(1, min(8, n_sel) + 1)}
ncomp_sel = min(cv_scores_sel, key=cv_scores_sel.get)
pls_sel = PLSRegression(n_components=ncomp_sel, scale=False).fit(X_tr[:, sel_mask], y_tr)
RMSECV_sel = cv_scores_sel[ncomp_sel]
RMSEP_sel = float(np.sqrt(np.mean((pls_sel.predict(X_te[:, sel_mask]).ravel() - y_te) ** 2)))

compare = pd.DataFrame([
    {"model": "全波段", "n_wavelengths": X_vip.shape[1], "n_LV": ncomp_full, "RMSECV": RMSECV_full, "RMSEP": RMSEP_full},
    {"model": "VIP>1 篩選後", "n_wavelengths": n_sel, "n_LV": ncomp_sel, "RMSECV": RMSECV_sel, "RMSEP": RMSEP_sel},
])
compare

> **選擇偏誤（selection bias）提醒**：篩選波長這個動作本身是用校正/訓練資料做的決定；如果連 RMSEP 都拿同一批測試集反覆調整篩選門檻，測試集就不再獨立了。這裡的 `X_te` 只在最後被用一次，篩選門檻（VIP>1）是憑經驗法則先定好的，不是靠看 RMSEP 調出來的。

## 3. Part 2 — PLS-DA：六種食用油分類

把類別編碼成 one-hot Y（每一類是一欄 0/1），對每一欄同時做 PLS 迴歸，**預測類別 = ŷ 最大的那一欄（argmax）**。

In [ ]:
df_oil = load_spectra(io.StringIO(SAMPLE_CSV_OILS))
meta_oil, X_oil, wl_oil = split_spectra(df_oil)

oil_codes_sorted = sorted(meta_oil["oil"].unique())
code_to_idx = {c: i for i, c in enumerate(oil_codes_sorted)}
y_oil_idx = meta_oil["oil"].map(code_to_idx).to_numpy()
Y_onehot = np.eye(len(oil_codes_sorted))[y_oil_idx]

X_oil_tr, X_oil_te, y_oil_tr, y_oil_te, Yoh_tr, Yoh_te = train_test_split(
    X_oil, y_oil_idx, Y_onehot, test_size=0.3, random_state=1, stratify=y_oil_idx)


def cv_rmse_da(X, Y, ncomp, k=5, seed=1):
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    errs = []
    for tr, va in kf.split(X):
        m = PLSRegression(n_components=ncomp, scale=False).fit(X[tr], Y[tr])
        errs.append(((m.predict(X[va]) - Y[va]) ** 2).ravel())
    return float(np.sqrt(np.mean(np.concatenate(errs))))


cv_da = {nc: cv_rmse_da(X_oil_tr, Yoh_tr, nc) for nc in range(2, 9)}
ncomp_da = min(cv_da, key=cv_da.get)
plsda = PLSRegression(n_components=ncomp_da, scale=False).fit(X_oil_tr, Yoh_tr)

pred_te = plsda.predict(X_oil_te)
pred_class = np.argmax(pred_te, axis=1)
cm = confusion_matrix(y_oil_te, pred_class, labels=list(range(len(oil_codes_sorted))))
acc_da = float(np.trace(cm) / cm.sum())

print("類別順序：", oil_codes_sorted)
print("混淆矩陣（列=實際，欄=預測）：")
print(pd.DataFrame(cm, index=oil_codes_sorted, columns=oil_codes_sorted))
print(f"\n整體 accuracy = {acc_da:.3f}  (ncomp={ncomp_da})")

In [ ]:
def sens_spec(cm, i):
    TP = int(cm[i, i])
    FN = int(cm[i].sum() - TP)
    FP = int(cm[:, i].sum() - TP)
    TN = int(cm.sum() - TP - FN - FP)
    sens = TP / (TP + FN) if (TP + FN) else float("nan")
    spec = TN / (TN + FP) if (TN + FP) else float("nan")
    return TP, FN, FP, TN, sens, spec


rows = []
for i, code_ in enumerate(oil_codes_sorted):
    TP, FN, FP, TN, sens, spec = sens_spec(cm, i)
    rows.append({"oil": code_, "TP": TP, "FN": FN, "FP": FP, "TN": TN,
                 "sensitivity": round(sens, 3), "specificity": round(spec, 3)})
per_class = pd.DataFrame(rows)
per_class

`EVOO`（特級初榨橄欖油）與 `ROO`（精製橄欖油）共用同一個「橄欖家族」基線光譜，`SOY`（大豆油）與 `CAN`（芥花油）共用同一個「大豆/芥花家族」基線——這兩對是最容易互相混淆的組合；`COC`（椰子油）與 `PEA`（花生油）各自有獨有訊號，通常幾乎不會被錯認。可以對照混淆矩陣確認是不是真的如此。

## 4. Part 3 — Hotelling T² 與 Q 殘差（離群值診斷）

用一組校正光譜（儀器 `m5_demo`）配適 PCA，然後對新樣品計算：

- **Hotelling T²** $= \sum_a \dfrac{t_a^2}{\lambda_a}$ —— 樣品在模型平面內、相對校正集的槓桿程度。95% 界限用 F 分布：$T^2_{lim} = \dfrac{k(n-1)}{n-k} F_{0.95}(k, n-k)$。
- **Q 殘差／SPE** $= \lVert x - \hat{x} \rVert^2$（重建誤差）—— 模型解釋不到的新變異。95% 界限用 Jackson–Mudholkar (1979) 近似公式，由捨棄主成分的特徵值算出。

In [ ]:
df_out = load_spectra(io.StringIO(SAMPLE_CSV_OUTLIER))
meta_out, X_out, wl_out = split_spectra(df_out)

cal_mask = (meta_out["role"] == "calibration").to_numpy()
X_cal_out = X_out[cal_mask]
n_cal_out = X_cal_out.shape[0]

pca_full = PCA(n_components=min(n_cal_out, X_out.shape[1]) - 1).fit(X_cal_out)
eigvals_all = pca_full.explained_variance_

NCOMP_PCA = 2
pca = PCA(n_components=NCOMP_PCA).fit(X_cal_out)
eigvals = pca.explained_variance_[:NCOMP_PCA]

F_crit = stats.f.ppf(0.95, NCOMP_PCA, n_cal_out - NCOMP_PCA)
T2_limit = NCOMP_PCA * (n_cal_out - 1) / (n_cal_out - NCOMP_PCA) * F_crit

theta1 = float(np.sum(eigvals_all[NCOMP_PCA:]))
theta2 = float(np.sum(eigvals_all[NCOMP_PCA:] ** 2))
theta3 = float(np.sum(eigvals_all[NCOMP_PCA:] ** 3))
h0 = 1 - (2 * theta1 * theta3) / (3 * theta2 ** 2)
z95 = stats.norm.ppf(0.95)
Q_limit = theta1 * (z95 * np.sqrt(2 * theta2 * h0 ** 2) / theta1 + 1 + theta2 * h0 * (h0 - 1) / theta1 ** 2) ** (1 / h0)

print(f"校正集 n={n_cal_out}，PCA 保留 {NCOMP_PCA} 個主成分")
print(f"T2_limit (95%) = {T2_limit:.3f}   Q_limit (95%) = {Q_limit:.6f}")


def score_new(x, pca_model, eigvals):
    sc_ = pca_model.transform(x.reshape(1, -1))[0]
    rec = pca_model.inverse_transform(sc_.reshape(1, -1))[0]
    q_ = float(np.sum((x - rec) ** 2))
    t2_ = float(np.sum((sc_ ** 2) / eigvals))
    return t2_, q_


records = []
for i in range(len(meta_out)):
    t2_, q_ = score_new(X_out[i], pca, eigvals)
    records.append({"sample_id": meta_out.loc[i, "sample_id"], "instrument": meta_out.loc[i, "instrument"],
                     "role": meta_out.loc[i, "role"], "T2": t2_, "Q": q_})
scores_df = pd.DataFrame(records)
scores_df["over_T2"] = scores_df["T2"] > T2_limit
scores_df["over_Q"] = scores_df["Q"] > Q_limit
scores_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
role_colors = {"calibration": "tab:blue", "new_normal": "tab:green",
               "leverage_outlier": "tab:orange", "cross_instrument": "tab:red"}
for role, sub in scores_df.groupby("role"):
    ax.scatter(sub["T2"], sub["Q"], s=70, label=role, color=role_colors.get(role, "grey"),
               marker="^" if role != "calibration" else "o")
ax.axvline(T2_limit, color="grey", linestyle="--", linewidth=1.2, label="T2 limit (95%)")
ax.axhline(Q_limit, color="black", linestyle="--", linewidth=1.2, label="Q limit (95%)")
ax.set_xlabel("Hotelling T^2")
ax.set_ylabel("Q residual (SPE)")
ax.set_title("Influence plot: T^2 vs Q")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

n_flag_q = int(scores_df["over_Q"].sum())
n_flag_t2 = int(scores_df["over_T2"].sum())
print(f"超過 Q 界限的樣品數：{n_flag_q}；超過 T2 界限的樣品數：{n_flag_t2}")

cross_instr = scores_df[scores_df["role"] == "cross_instrument"]
assert (cross_instr["Q"] > Q_limit).all(), "跨儀器樣品應該全部超過 Q 界限，才能示範模型遷移造成的高 Q"
print("檢查通過：兩個跨儀器樣品 (mp5/mp6) 的 Q 殘差都超過界限——"
      "模型從沒看過這種系統性的儀器響應差異。")

lev_row = scores_df[scores_df["role"] == "leverage_outlier"].iloc[0]
assert lev_row["T2"] > T2_limit, "槓桿示範點的 T2 應該超過界限，才能示範『極端但化學上合理』"
assert lev_row["Q"] < Q_limit, "槓桿示範點的 Q 應該低於界限，才能跟跨儀器樣品的高 Q 對比"
print(f"檢查通過：槓桿示範點 T2={lev_row['T2']:.2f}(超過界限{T2_limit:.2f}) 但 "
      f"Q={lev_row['Q']:.5f}(低於界限{Q_limit:.5f})——跟跨儀器樣品剛好相反。")

可以看到：**槓桿離群點**（`leverage_outlier`）T² 偏高、但 Q 仍在界限內（模型認得它的化學組成，只是比較極端）；**跨儀器樣品**（`cross_instrument`，`mp5_demo`／`mp6_demo`）則是 Q 明顯超標——模型平面解釋不到這個新的系統性差異，即使 T² 可能還好。

**發現離群值後**：先回頭檢查光譜/樣品本身（氣泡、雜質、稀釋錯誤、儀器差異）；不能默默刪除，刪除前必須記錄原因；換儀器/換批次後出現高 Q，通常代表需要重新校正或做校正轉移（calibration transfer），而不是照樣使用舊模型。

## 5. 練習題 (Exercises)

### 問題 1

把第 2.1 節的 PLS 模型 LV 數改成 `ncomp_full + 3`（刻意多加幾個 LV），重新算一次 VIP，並跑一次第 2.2 節的三個檢查。`mean(VIP²)` 還是應該非常接近 1 嗎？為什麼（提示：這個恆等式是不是跟 LV 數無關）？

In [ ]:
# TODO: 請在這裡作答

### 問題 2

對照第 2.4 節的圖，除了 1200nm 與 1450nm，還有沒有其他 VIP 明顯 > 1 但係數方向跟你預期不同的波段？你會怎麼跟同學解釋「VIP 高」不代表「正相關」？

In [ ]:
# TODO: 請在這裡作答

### 問題 3

把第 2.5 節的篩選門檻從 `VIP > 1.0` 改成 `VIP > 1.3`（更嚴格），重新選 LV、重新配適、重新算 RMSEP。篩選後的波長數、RMSECV、RMSEP 各自怎麼變化？門檻越嚴格是不是一定越好？

In [ ]:
# TODO: 請在這裡作答

### 問題 4

從第 3 節的混淆矩陣中，找出 `per_class` 表裡 sensitivity 最低的那個類別（提示：用 `per_class.sort_values("sensitivity")`），計算/確認它的 TP/FN/FP/TN，並解釋敏感度低 vs 特異度低分別代表什麼、這個類別最常被誤判成哪一類。

In [ ]:
# TODO: 請在這裡作答

### 問題 5

第 4 節的 `scores_df` 中，`leverage_outlier` 樣品的 T²／Q 分別是多少？如果今天多了一個新樣品，T² 正常但 Q 略高於界限、且預測值本身看起來合理，你會怎麼處理（直接採用？記錄後續追蹤？要求重新測量？）？

In [ ]:
# TODO: 請在這裡作答

## 6. 匯出結果 (Export)

In [ ]:
output_path = "nb12_results_summary.csv"
summary = pd.concat([
    compare.assign(section="VIP_selection"),
    per_class.assign(section="PLSDA_per_class"),
], ignore_index=True)
summary.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")